<div align="center">

# Hafta 6: NLP ve klinik metin madenciliği

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Hafta sayfası](README.md) · [Ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html) · [Etkileşimli lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/lab.html) · [PDF ders notu](Hafta06_Ders_Notlari.pdf) · [Colab'da aç](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-06/NB06_klinik_metin_madenciligi.ipynb)

</div>

**Bu Jupyter Notebook nasıl kullanılır?** Jupyter Notebook, ders sayfasındaki uygulama duraklarını izler. Her durak üç bölümden oluşur ve her bölüm tek başına da çalışır: Python temelleri, hazır prompt (istem) metinleriyle uygulama ve hazır uygulama. **Python temelleri** bölümü, durağın gerektirdiği Python bilgisini haftanın konusundan alınan küçük örneklerle öğretir; bu bölümdeki alıştırmalar sonucu kendileri denetler. **Prompt ile uygulama** bölümünde, bir yapay zekâ (YZ) asistanı için hazır promptlar bulunur: Kodu asistan yazar, siz çalıştırırsınız ve bir kontrol hücresi sonucun doğru olup olmadığını gösterir. **Hazır uygulama** bölümü ise durağın baştan sona çalışan Python programıdır ve kodu açıklayan notlar içerir.

Üç bölümü bu sırayla izleyin. Python biliyorsanız doğrudan promptlarla başlayabilir, yalnızca sonucu görmek istiyorsanız hazır uygulamayı açabilirsiniz. Sondaki haftalık proje de kendi başına çalışır.

**Başlangıç hücreleri.** Her bölüm bir başlangıç hücresiyle açılır. Bu hücre ders paketini ve haftanın paketlerini hazırlar, ayrıca bölümdeki hücrelerin kullandığı veri çağrılarını ve önceki bölümlerin kodlarını yineler. Bu sayede bölüme doğrudan başladığınızda da kodlar hata vermez. Önceki bölümleri bu oturumda çalıştırdıysanız başlangıç hücresini çalıştırmanız gerekmez; yeniden çalıştırmanın da bir zararı yoktur. Ders sayfasının doğrudan açtığı birkaç hazır uygulama kısmının da kendi başlangıç hücresi vardır.

**Veri.** Haftanın raporları sentetiktir: İki hastanenin epikrizlerini ve anjiyografi raporlarını ders paketindeki sanal klinik, Jupyter Notebook'un içinde birkaç saniyede yazar. Bir hücre dersin küçük Türkçe dil modelini (ytu-ce-cosmos/turkish-tiny-bert-uncased) ilk kez yüklediğinde Transformers paketi modeli Hugging Face Hub'dan indirir; model yaklaşık 18,5 MB'tır. Sonraki çalıştırmalar bilgisayardaki kopyayı kullanır. Model MIT lisansıyla yayımlanmıştır. 2. durağın hazır uygulaması BioBERT'in birkaç yüz kilobaytlık tokenizer'ını da indirir.

**Yerinizi bulmak.** İçindekiler paneli her durağı ve her bölümü listeler. Bu panel, Colab'da soldaki liste simgesiyle, Jupyter'de ise View → Table of Contents menüsüyle açılır. Her bölüm, ders sayfasına dönen bir bağlantıyla biter. Colab'da yaptığınız değişiklikler yalnızca Drive'a kaydedilen bir kopyada kalıcı olur: File → Save a copy in Drive komutunu kullanın.

**Nerede çalışır?** Jupyter Notebook, kendi bilgisayarınızdaki Jupyter'de (Anaconda) ve Google Colab'da çalışır. Buradaki bütün veri setleri ya sentetiktir ya da açık bir araştırma veri tabanından gelir. Gerçek hasta verileri yalnızca kurumunuzun bir bilgisayarında ve kurumunuzun kurallarına göre işlenir; bulut tabanlı bir Jupyter Notebook'a ya da bir YZ sohbetine asla konmaz.

## 0. Kurulum

**Jupyter Notebook'un ilk kullanımı.** Bir Jupyter Notebook hücrelerden oluşur. Bu hücre metin içerir, gri hücrelerde ise Python kodu vardır. Aşağıdaki kod hücresinin içine tıklayın ve Shift+Enter tuşlarına basın: Hücre çalışır, sonucu hücrenin altında görünür ve solundaki köşeli parantez içindeki sayı, örneğin `[1]`, çalıştırma sırasını gösterir. Jupyter Notebook'un arkasındaki Python süreci olan kernel, çalıştırılan hücrelerin ürettiği her şeyi hatırlar; bu yüzden hücreler yukarıdan aşağıya sırayla çalıştırılır. **Kernel → Restart Kernel and Run All Cells** komutu Jupyter Notebook'un tamamını yeniden çalıştırır, Ctrl+S (Mac'te Cmd+S) ise Jupyter Notebook'u `.ipynb` dosyası olarak kaydeder.

**Başlangıç hücresi.** Aşağıdaki kod hücresi oturumu hazırlar. Hücre, `vclinic` ders paketini ders klasöründe bulur; Jupyter Notebook tek başına açılmışsa (örneğin Google Colab'da), paketi GitHub'dan bir kez indirir. Ardından paketleri yükler, mesajların dilini seçer, haftanın paketi olan `transformers` eksikse onu pip ile kurar ve alıştırmaların beklenen cevaplarını hazırlar. Hücrenin son satırları haftanın kütüphanelerini içe aktarır. Jupyter Notebook'un her bölümü, aynı kurulumu ve bölümün veri çağrılarını içeren bir başlangıç hücresiyle açılır.

In [ ]:
# Jupyter Notebook'un başlangıç hücresi
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# haftanın kütüphaneleri

import difflib
import re
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

## Durak 1: Epikrizlerden altı tanı: Kurallar ve bir model

*Ders sayfasında: [1. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#stop-1)*

Altı kronik tanı sanal kliniğin epikrizlerinden okunur: Önce tanıların kelimeleri aranır, sonra her geçişin durumu belirlenir, ardından ilaçlar ve iki laboratuvar değeri eklenir; son olarak eğitim epikrizlerinden öğrenen bir model kurulur. Her yöntem A hastanesinin test kümesinde, model ayrıca B hastanesinin epikrizlerinde değerlendirilir.

**Durağın sonunda:** Anahtar kelime kuralının atriyal fibrilasyon için pozitif öngörü değerini (0,19), her geçişin durumu hesaba katıldığında (0,856) ve ilaçlarla laboratuvar değerleri eklendiğinde (0,98) makro F1 değerini ve TF-IDF modelinin B hastanesindeki makro F1 değerini (0,708) bulacaksınız.

Durak üç bölümden oluşur: **Python temelleri**, **Prompt ile uygulama** ve **Hazır uygulama**. Bölümleri bu sırayla izleyin. Python biliyorsanız promptlarla, yalnızca sonucu görmek istiyorsanız hazır uygulamayla başlayabilirsiniz. Her bölüm kendi başlangıç hücresiyle açılır.

### Durak 1 · Python temelleri

Bu haftanın verileri metindir. Python'da bir metin bir *karakter dizisidir* (string): Tırnak işaretleri arasında yazılmış bir karakter dizisi. Bu bölümde bir karakter dizisinin içine nasıl bakılacağı, metotlarıyla nasıl değiştirileceği, Türkçe bir metnin nasıl doğru biçimde küçük harfe çevrileceği, düzenli ifadelerle kalıpların nasıl bulunacağı, bir metnin nasıl token'lara bölüneceği ve son olarak Hugging Face Hub'dan önceden eğitilmiş bir dil modelinin nasıl çağrılacağı gösterilir.

Hücreleri sırayla çalıştırın. Bir örneğin altındaki *Python notu* yeni yapıyı açıklar. Alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerdedir.

**Başlangıç hücresi.** Aşağıdaki hücre, «Durak 1 · Python temelleri» bölümü için kurulumu, sonraki hücrelerin kullandığı veri çağrılarını ve önceki hücrelerin kodlarını yineler; böylece kodlar hata vermeden çalışır. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Durak 1 · Python temelleri» bölümü
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# alıştırma promptlarının tabloları (etkileşimli labın F bölümü)
from vclinic.week06 import alistirma_tablosu

# önceki hücrelerden yinelenen kod: Aşağıdaki hücrelerin kullandığı veriler ve kodlar
import re
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
text = "Öykü: 83 yaşında kadın hasta, eforla gelen nefes darlığı yakınmasıyla kardiyoloji polikliniğine başvurdu. Özgeçmiş: Bilinen kronik hastalığı yok. Atriyal fibrilasyon tanımlamıyor."
print(len(text), "karakter")
print(text[:5], "|", text[-31:])
print(text.lower()[:40])
print("tanımlamıyor" in text, "diyabet" in text)
print(text.split(". ")[2])
print(f"Metinde {text.count('a')} tane a harfi var.")

**Python notu.** Karakter dizisi tırnak işaretleri arasında yazılır; `len` karakterlerini sayar. Köşeli parantezler dizinin bir parçasını alır: `text[:5]` ilk beş karakteri, `text[-31:]` ise son 31 karakteri verir; negatif konumlar sondan sayılır. *Metot*, bir değere ait olan ve noktayla çağrılan bir fonksiyondur: `text.lower()` küçük harfli bir kopya verir, `text.split(". ")` metni ardından boşluk gelen her noktada böler ve bir liste döndürür, `text.count("a")` ise bir harfi sayar. `"tanımlamıyor" in text` bir karakter dizisinin başka birinde geçip geçmediğini sorar ve `True` ya da `False` verir. `f"... {değer} ..."` biçimindeki f-string değerleri bir metnin içine yerleştirir.

**Ders sayfasında:** [Çözümlü örnek: "İNFERİOR" kelimesinin iki küçük harf biçimi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#worked-example-two-lower-cases-of-i-nferi-or). Aşağıdaki hücre aynı kelimenin iki küçük harf biçimini karşılaştırır.

In [ ]:
# Python alıştırması: iki i harfi
word = "İNFERİOR"
python_lower = word.lower()


def tr_lower(text):
    return text.replace("I", "ı").replace("İ", "i").lower()


print(python_lower, len(python_lower), "|", tr_lower(word), len(tr_lower(word)))
print([hex(ord(c)) for c in python_lower[:3]])
print(tr_lower("PROKSİMAL KISIMDA"), "|", "PROKSİMAL KISIMDA".lower())

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c102)*

**Python notu.** `def tr_lower(text):` bütün haftanın kullandığı tek satırlık bir fonksiyon tanımlar: `lower()` çağrılmadan önce I'yı ı ile, İ'yi i ile değiştirir; böylece Türkçe kelimeler harflerini ve uzunluklarını korur. Python'un kendi `lower()` metodu İ harfini iki karaktere dönüştürür: `ord` bir karakterin Unicode'daki numarasını verir, `hex` ise bu numarayı 16 tabanında yazar. İkinci karakter olan 0x307 bir üst noktadır; kelimeyi bir karakter uzatır ve "inferior" kelimesinden farklı kılar. `lower()` ayrıca KISIMDA kelimesini, Türkçede bulunmayan iki i harfiyle "kisimda" yapar.

In [ ]:
# Python alıştırması: düzenli ifadeler
lab = "Laboratuvar: kreatinin 0,95 mg/dL, eGFR 59 mL/dk/1,73 m², HbA1c %7,8, LDL 83 mg/dL."
m = re.search(r"eGFR (\d+)", lab)
print(m.group(0), "→", int(m.group(1)))
print(re.findall(r"\d+(?:,\d+)?", lab))
hba1c = re.search(r"HbA1c %(\d+,\d+)", lab).group(1)
print(float(hba1c.replace(",", ".")))
for sentence in ["Diyabet yok.", "Prediyabet nedeniyle diyet önerildi."]:
    print(sentence, "| in:", "diyabet" in tr_lower(sentence), "| whole word:", bool(re.search(r"(?<!\w)diyabet(?!\w)", tr_lower(sentence))))

**Python notu.** *Düzenli ifade* (regular expression) metin için küçük bir kalıptır; ters bölü işaretleri olduğu gibi kalsın diye `r"..."` biçiminde ham bir karakter dizisi olarak yazılır. `\d` herhangi bir rakamdır, `+` ise bir ya da daha fazla anlamına gelir; böylece `\d+` bir tam sayıyı bulur. `( )` parantezleri saklanacak bir parçayı işaretler; `m.group(1)` bu parçayı döndürür. `re.search` ilk eşleşmeyi, `re.findall` ise bütün eşleşmeleri bulur. `(?:,\d+)?` Türkçedeki gibi virgüllü, isteğe bağlı bir ondalık kısım ekler; `.replace(",", ".")` bu değeri `float` fonksiyonunun okuyabileceği bir sayıya dönüştürür. `(?<!\w)` ve `(?!\w)` "önünde harf yok" ve "arkasında harf yok" demektir; yani tam kelime aranır. Bunlarla "diyabet" artık "prediyabet" kelimesinin içinde bulunmaz.

**Ders sayfasında:** [Çözümlü örnek: WordPiece "yetmezliğinin" kelimesini böler](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#worked-example-wordpiece-cuts-yetmezli-inin). Aşağıdaki hücre aynı kelimeyi içeren bir cümleyi dersin modelinin token'larına böler.

In [ ]:
# Python alıştırması: token'lar
sentence = "Dekompanse kalp yetmezliğinin nedeni araştırıldı."
print(sentence.split())
print(re.findall(r"\w+", tr_lower(sentence)))
MODEL = "ytu-ce-cosmos/turkish-tiny-bert-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
encoded = tokenizer(tr_lower(sentence))
print(encoded["input_ids"])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print(len(tokenizer), "sözlükteki parça sayısı")

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c106)*

**Python notu.** `split()` metni boşluklardan böler ve noktalama işaretlerini kelimelerin üzerinde bırakır; `re.findall(r"\w+", ...)` ise yalnızca harf ve rakam dizilerini tutar. Bir dil modeli kendi tokenizer'ını kullanır. `AutoTokenizer.from_pretrained(MODEL)` modelin tokenizer dosyalarını ilk seferde Hugging Face Hub'dan indirir ve yükler; modelin adı, Hub'daki sayfasının adıdır. Bir metin üzerinde çağrılan tokenizer, başında [CLS] ve sonunda [SEP] özel token'ları bulunan token numaralarını (`input_ids`) verir; `convert_ids_to_tokens` numaraları yeniden parçalara dönüştürür. "yetmezliğinin" kelimesi "yetmezliği" ve "##nin" parçalarına ayrılır.

In [ ]:
# Python alıştırması: önceden eğitilmiş bir modeli çağırmak
fill = pipeline("fill-mask", model=MODEL)
for guess in fill("hastanın kan [MASK] yüksekti."):
    print(f"{guess['token_str']:12} {guess['score']:.3f}")

**Python notu.** `pipeline("fill-mask", model=MODEL)` modelin yaklaşık 18,5 MB'lık ağırlıklarını indirir ve tokenizer'la modeli tek bir fonksiyonda birleştirir. [MASK] özel token'ını içeren bir cümle verildiğinde boşluk için en olası beş token'ı, olasılıklarıyla birlikte, her biri küçük bir sözlük olarak döndürür; `guess['token_str']` ve `guess['score']` bu sözlüğün anahtarlarını okur. İlk tahmin olan "basıncı", "kan basıncı" ifadesini tamamlar. Model bunu kliniğin raporlarından değil, genel Türkçe metinlerden öğrenmiştir.

**Hızlı kontrol.** `re.findall(r"\d+", "EF %55, eGFR 59")` ne döndürür?

<details><summary>Cevap</summary>

`['55', '59']` listesini, yani her rakam dizisini karakter dizisi olarak döndürür. Bunlar ancak `int` ya da `float` ile sayıya dönüşür.

</details>

**Alıştırma 1.1.** Vakanın, yani VC-2444 numaralı hastanın epikrizinde Türkçe küçük harfe çevirmeden sonra kaç kelime, yani harf ve rakam dizisi vardır? Epikriz, `reports.archive()` tablosunda hastanın satırındaki `text_tr` sütunudur.

In [ ]:
veri = alistirma_tablosu("1.1")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 1.1", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
case_text = reports.archive().set_index("patient_id").loc["VC-2444", "text_tr"]
cevap = len(re.findall(r"\w+", tr_lower(case_text)))
teach.check("Alıştırma 1.1", cevap)

**Alıştırma 1.2.** Vakanın eGFR değerini epikrizinden düzenli bir ifadeyle okuyun ve bir tam sayı olarak saklayın.

In [ ]:
veri = alistirma_tablosu("1.2")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 1.2", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
case_text = reports.archive().set_index("patient_id").loc["VC-2444", "text_tr"]
cevap = int(re.search(r"eGFR (\d+)", case_text).group(1))
teach.check("Alıştırma 1.2", cevap)

**Alıştırma 1.3.** Dersin tokenizer'ı, Türkçe küçük harfe çevrilmiş vaka epikrizi için iki özel token hariç kaç token verir?

In [ ]:
veri = alistirma_tablosu("1.3")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 1.3", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
case_text = reports.archive().set_index("patient_id").loc["VC-2444", "text_tr"]
cevap = len(tokenizer(tr_lower(case_text))["input_ids"]) - 2
teach.check("Alıştırma 1.3", cevap)

**«Durak 1 · Python temelleri» tamamlandı.** Sırada hemen aşağıdaki «Durak 1 · Prompt ile uygulama» bölümü var. Durağı burada bırakacaksanız [ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#last-two-hours-advanced-clinical-nlp-applications) ve okumaya «Son 2 saat: Gelişmiş klinik NLP uygulamaları» bölümünden devam edin.

### Durak 1 · Prompt ile uygulama

Bu bölümde dört prompt vardır: «Bir anahtar kelime kuralı», «Her geçişin durumu», «İlaçlar ve laboratuvar değerleri» ve «Epikrizlerden öğrenen bir model: TF-IDF ve lojistik regresyon».

Her promptta dört adım vardır. Promptu kopyalayıp YZ asistanınıza gönderin. Altındaki veri hücresini çalıştırın: Bu hücre, promptun tarif ettiği "veri" tablosunu hazırlar ve gösterir. Asistanın kodunu boş hücreye yapıştırıp çalıştırın. Ardından kontrol hücresini çalıştırın; bu hücre *doğru* ya da *henüz değil* yazar. Asistanın kodu kontrolden geçmezse *Hazır kod* hücresini çalıştırıp devam edin.

**Promptlarla çalışmak.** Python kodu yazabilen bir YZ asistanı yeterlidir. Google Colab'ın içinde Gemini bulunur ve pencerenin üstündeki Gemini düğmesiyle açılır; başka bir tarayıcı sekmesinde açılan ChatGPT, Claude, Copilot ya da Gemini de aynı işi görür. Haftanın bütün promptları [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-06/PROMPTLAR.md) de yer alır; GitHub her promptun yanında bir kopyalama düğmesi gösterir.

**Her promptta geçen iki ad: "veri" ve "cevap".** YZ asistanı çoğu zaman Jupyter Notebook'u göremez; bu yüzden promptlar verileri sözcüklerle tarif eder. Her promptta veri hücresi, o promptun verilerini "veri" adlı bir tabloya koyar; prompt da bu tablonun satırlarını ve sütunlarını anlatır. Prompt ayrıca asistandan sonucu "cevap" adıyla saklamasını ister; kontrol hücresi sonucu bu addan okur. Eğitimin promptlarında koddan yalnızca bu iki ad geçer.

**Onarım promptu** (çalışan ama kontrolden geçmeyen kod için):

> Aşağıdaki kod şu görev için yazıldı: [promptun Görev satırını yapıştırın]. Kod çalışıyor, ancak dersin kontrolü "henüz değil" yazıyor. Hatayı bul, iki cümleyle açıkla ve düzeltilmiş kodu ver. Sonuç yine "cevap" adıyla saklanmalı.  
>
> [kodu buraya yapıştırın]  

**Hata mesajı için prompt** (hata vererek duran kod için):

> Aşağıdaki kod bir Jupyter Notebook'ta çalıştırıldığında altta verilen hata mesajını veriyor. Hatanın nedenini sade bir dille açıkla ve düzeltilmiş kodu ver.  
>
> [kodu buraya yapıştırın]  
>
> [hata mesajını buraya yapıştırın]  

**Hiç değişmeyen kural.** Promptlarda yalnızca eğitimin sentetik ya da açık verileri bulunur. Bir YZ asistanına gerçek hasta verisi asla yapıştırılmaz.

**Başlangıç hücresi.** Aşağıdaki hücre, «Durak 1 · Prompt ile uygulama» bölümü için kurulumu ve bu bölümün veri çağrılarını yineler; böylece kodlar hata vermeden çalışır. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Durak 1 · Prompt ile uygulama» bölümü
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

from vclinic import reports
import re

if "archive" not in globals():
    archive = reports.archive()
    LABELS = list(reports.LABELS)
    texts = archive["text_tr"].to_numpy()
    y = archive[LABELS].to_numpy()
    split = archive["split"].to_numpy()
    tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")


    def tr_lower(text):
        """Türkçe kurallarına göre küçük harf: I, ı olur; İ, i olur ve metnin uzunluğu değişmez."""
        return text.replace("I", "ı").replace("İ", "i").lower()


    def words(text):
        """Bir metnin küçük harfli kelimeleri: Harf ve rakam dizileri."""
        return re.findall(r"\w+", tr_lower(text))
_COLS = ["KY", "AF", "KAH", "HT", "DM", "KBH"]
_SETS = {"training": "eğitim", "validation": "doğrulama", "test": "test", "external": "dış"}


def _summaries(mask, with_set=False, with_entities=False):
    """Bir kümenin epikrizleri ve kayıt sistemindeki altı tanı; promptların tarif ettiği biçimde."""
    t = pd.DataFrame({"epikriz": texts[mask]})
    if with_set:
        t.insert(0, "küme", [_SETS[s] for s in split[mask]])
    if with_entities:
        t["varlıklar"] = [[(e["start"], e["end"], e["label"]) for e in es] for es in archive["entities_tr"][mask]]
    for k, c in enumerate(_COLS):
        t[c] = y[mask, k]
    return t.reset_index(drop=True)

print(len(archive), "epikriz;", {_SETS[s]: int((split == s).sum()) for s in _SETS})

#### Prompt 1.1: Bir anahtar kelime kuralı

**Amaç.** Bu promptla epikrizlerde altı tanının kelimeleri aranır ve bir kelime aramasının nerede yanıldığı görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Bir tanının kelimelerini arayan basit bir aramanın epikrizlerde altı kronik tanıyı ne kadar iyi bulduğunu öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; sentetik bir kayıt sistemi olan sanal kliniğin A hastanesine ait test kümesindeki 600 epikrizi tutar. Her satır bir hastadır: "epikriz" sütunu Türkçe epikrizi tutar; "KY", "AF", "KAH", "HT", "DM" ve "KBH" adlı altı sütun 0 ya da 1 değerini alır; kayıt sisteminde sırasıyla kalp yetersizliği, atriyal fibrilasyon, koroner arter hastalığı, hipertansiyon, diyabetes mellitus ve kronik böbrek hastalığı kayıtlıysa değer 1, değilse 0 olur.  
> Görev: Her metni Türkçe kurallarına göre küçük harfe çevir: Önce I'yı ı, İ'yi i yap, ardından öteki harfleri küçült. Her tanının şu kelimelerini tam kelime olarak ara; böylece hiçbir kelime daha uzun bir kelimenin içinde bulunmaz: Kalp yetersizliği için "kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky" ve "kky"; atriyal fibrilasyon için "atriyal fibrilasyon", "af" ve "afib"; koroner arter hastalığı için "koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü" ve "akut koroner sendrom"; hipertansiyon için "hipertansiyon" ve "ht"; diyabetes mellitus için "diyabet", "diyabetes mellitus" ve "dm"; kronik böbrek hastalığı için "kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh" ve "kby". Kelimelerinden en az biri epikrizin herhangi bir yerinde bulunan tanı var kabul edilir. Altı kararı kayıt sistemindeki altı sütunla karşılaştır; her tanı için duyarlılığı, pozitif öngörü değerini ve F1 değerini (doğru pozitiflerin iki katı, doğru pozitiflerin iki katı ile yanlış pozitiflerin ve yanlış negatiflerin toplamına bölünür) ve altı F1 değerinin ortalamasını, yani makro F1 değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her tanı için bir satır ve üç değer içeren bir tablo ile makro F1 değerini yazdır. Atriyal fibrilasyonun pozitif öngörü değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her değer 0 ile 1 arasında olmalı ve VC-2444 numaralı hastanın epikrizi kalp yetersizliği, atriyal fibrilasyon ve koroner arter hastalığı için pozitif bulunmalı.  

In [ ]:
# Prompt 1.1 için veri: Promptun tarif ettiği "veri" tablosu
veri = _summaries(te)
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 1.1 kontrolü
teach.check_step("Prompt 1.1", "cevap", globals())

In [ ]:
#@title Prompt 1.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _summaries(te)
terimler = {"KY": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
           "AF": ["atriyal fibrilasyon", "af", "afib"],
           "KAH": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
           "HT": ["hipertansiyon", "ht"], "DM": ["diyabet", "diyabetes mellitus", "dm"],
           "KBH": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"]}


def turkce_kucuk(metin):
    return metin.replace("I", "ı").replace("İ", "i").lower()


def skor(kararlar):
    satirlar = []
    for s in terimler:
        gercek = veri[s].to_numpy() == 1
        tahmin = np.array(kararlar[s]) == 1
        dp, yp, yn = int((tahmin & gercek).sum()), int((tahmin & ~gercek).sum()), int((~tahmin & gercek).sum())
        satirlar.append({"tanı": s, "duyarlılık": round(dp / max(1, dp + yn), 3),
                         "PPV": round(dp / max(1, dp + yp), 3), "F1": round(2 * dp / max(1, 2 * dp + yp + yn), 3)})
    tablo = pd.DataFrame(satirlar)
    print(tablo)
    print("Makro F1:", round(float(tablo["F1"].mean()), 3))
    return tablo

kararlar = {s: [] for s in terimler}
for metin in veri["epikriz"]:
    kucuk = turkce_kucuk(metin)
    for s, kelimeler in terimler.items():
        bulundu = any(re.search(r"(?<!\w)" + re.escape(k) + r"(?!\w)", kucuk) for k in kelimeler)
        kararlar[s].append(int(bulundu))
tablo = skor(kararlar)
cevap = float(tablo.loc[tablo["tanı"] == "AF", "PPV"].iloc[0])
teach.check_step("Prompt 1.1", "cevap", globals())

**Sonuç ne gösteriyor?** Atriyal fibrilasyon için pozitif öngörü değeri 0,19 olur: Her beş alarmdan dördü yanlıştır; çünkü kural "Atriyal fibrilasyon izlenmedi" ve "AF tanımlamıyor" cümlelerini de atriyal fibrilasyon sayar. En düşük duyarlılık kronik böbrek hastalığındadır; çünkü epikrizlerin çoğu bu tanının adını hiç anmaz.

#### Prompt 1.2: Her geçişin durumu

**Amaç.** Bu promptla yok sayılan, olası ya da bir yakına ait geçişler dışarıda bırakılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Epikrizlerde altı tanının kelimelerini arayan bir aramanın, yok sayılan, yalnızca şüphelenilen ya da bir yakına ait geçişler dışarıda bırakıldığında ne kadar iyileştiğini öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; sentetik bir kayıt sistemi olan sanal kliniğin A hastanesine ait test kümesindeki 600 epikrizi tutar. Her satır bir hastadır: "epikriz" sütunu Türkçe epikrizi tutar; "KY", "AF", "KAH", "HT", "DM" ve "KBH" adlı altı sütun 0 ya da 1 değerini alır; kayıt sisteminde sırasıyla kalp yetersizliği, atriyal fibrilasyon, koroner arter hastalığı, hipertansiyon, diyabetes mellitus ve kronik böbrek hastalığı kayıtlıysa değer 1, değilse 0 olur.  
> Görev: Her metni Türkçe kurallarına göre küçük harfe çevir: Önce I'yı ı, İ'yi i yap, ardından öteki harfleri küçült. Her tanının şu kelimelerini tam kelime olarak ara; böylece hiçbir kelime daha uzun bir kelimenin içinde bulunmaz: Kalp yetersizliği için "kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky" ve "kky"; atriyal fibrilasyon için "atriyal fibrilasyon", "af" ve "afib"; koroner arter hastalığı için "koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü" ve "akut koroner sendrom"; hipertansiyon için "hipertansiyon" ve "ht"; diyabetes mellitus için "diyabet", "diyabetes mellitus" ve "dm"; kronik böbrek hastalığı için "kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh" ve "kby". Bir geçişten sonra, cümlesinin sonuna ya da başka bir tanının bir sonraki geçişine kadar şu kelimelerden biri geliyorsa geçiş sayılmaz: "yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı" ya da "(-)" (olumsuz); "şüphe", "olası" ya da "ön tanı" (olası). Aynı cümlede geçişten önce şu kelimelerden biri duruyorsa da geçiş sayılmaz: "babası", "annesi", "kardeşi", "babada", "annede" ya da "soygeçmiş" (bir yakın). Geçişlerinden en az biri sayılan tanı var kabul edilir. Altı kararı kayıt sistemindeki altı sütunla karşılaştır; her tanı için duyarlılığı, pozitif öngörü değerini ve F1 değerini (doğru pozitiflerin iki katı, doğru pozitiflerin iki katı ile yanlış pozitiflerin ve yanlış negatiflerin toplamına bölünür) ve altı F1 değerinin ortalamasını, yani makro F1 değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her tanı için bir satır ve üç değer içeren bir tablo yazdır; makro F1 değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Atriyal fibrilasyonun pozitif öngörü değeri Prompt 1.1'dekinden yüksek olmalı ve VC-2444 numaralı hastanın epikrizi yalnızca kalp yetersizliği için pozitif bulunmalı.  

In [ ]:
# Prompt 1.2 için veri: Promptun tarif ettiği "veri" tablosu
veri = _summaries(te)
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 1.2 kontrolü
teach.check_step("Prompt 1.2", "cevap", globals())

In [ ]:
#@title Prompt 1.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _summaries(te)
terimler = {"KY": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
           "AF": ["atriyal fibrilasyon", "af", "afib"],
           "KAH": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
           "HT": ["hipertansiyon", "ht"], "DM": ["diyabet", "diyabetes mellitus", "dm"],
           "KBH": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"]}


def turkce_kucuk(metin):
    return metin.replace("I", "ı").replace("İ", "i").lower()


def skor(kararlar):
    satirlar = []
    for s in terimler:
        gercek = veri[s].to_numpy() == 1
        tahmin = np.array(kararlar[s]) == 1
        dp, yp, yn = int((tahmin & gercek).sum()), int((tahmin & ~gercek).sum()), int((~tahmin & gercek).sum())
        satirlar.append({"tanı": s, "duyarlılık": round(dp / max(1, dp + yn), 3),
                         "PPV": round(dp / max(1, dp + yp), 3), "F1": round(2 * dp / max(1, 2 * dp + yp + yn), 3)})
    tablo = pd.DataFrame(satirlar)
    print(tablo)
    print("Makro F1:", round(float(tablo["F1"].mean()), 3))
    return tablo

olumsuz = ["yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)"]
olasi = ["şüphe", "olası", "ön tanı"]
yakin = ["babası", "annesi", "kardeşi", "babada", "annede", "soygeçmiş"]


def var_olan(kucuk):
    gecisler = sorted((m.start(), m.end(), s) for s, kelimeler in terimler.items() for k in kelimeler
                      for m in re.finditer(r"(?<!\w)" + re.escape(k) + r"(?!\w)", kucuk))
    bulunan = set()
    for i, (bas, son, s) in enumerate(gecisler):
        sonraki = next((m[0] for m in gecisler[i + 1:] if m[2] != s), len(kucuk))
        sinirlar = [x for x in (kucuk.find(".", son), kucuk.find(";", son), sonraki, len(kucuk)) if x >= son]
        sonra = kucuk[son:min(sinirlar)]
        cumle_basi = max(kucuk.rfind(". ", 0, bas), kucuk.rfind(": ", 0, bas)) + 1
        once = kucuk[cumle_basi:bas]
        if any(k in once for k in yakin):
            continue
        if any(k in sonra for k in olumsuz + olasi):
            continue
        bulunan.add(s)
    return bulunan


kararlar = {s: [] for s in terimler}
for metin in veri["epikriz"]:
    bulunan = var_olan(turkce_kucuk(metin))
    for s in terimler:
        kararlar[s].append(int(s in bulunan))
tablo = skor(kararlar)
cevap = round(float(tablo["F1"].mean()), 3)
teach.check_step("Prompt 1.2", "cevap", globals())

**Sonuç ne gösteriyor?** Makro F1 değeri anahtar kelime kuralındaki 0,666 değerinden 0,856 değerine yükselir. Pozitif öngörü değerleri 1'e ulaşır; kalan hatalar kaçırılan tanılardır, yani epikrizin adını hiç anmadığı tanılar.

#### Prompt 1.3: İlaçlar ve laboratuvar değerleri

**Amaç.** Bu promptla hastalığın adı olmayan kanıtlar eklenir: İlaçlar, eGFR ve HbA1c.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Epikrizlerde altı kronik tanıyı, epikriz bu tanıların adını anmadığında ama onlar için verilen bir ilacı ya da onları tanımlayan bir laboratuvar değerini andığında da bulmak istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; sentetik bir kayıt sistemi olan sanal kliniğin A hastanesine ait test kümesindeki 600 epikrizi tutar. Her satır bir hastadır: "epikriz" sütunu Türkçe epikrizi tutar; "KY", "AF", "KAH", "HT", "DM" ve "KBH" adlı altı sütun 0 ya da 1 değerini alır; kayıt sisteminde sırasıyla kalp yetersizliği, atriyal fibrilasyon, koroner arter hastalığı, hipertansiyon, diyabetes mellitus ve kronik böbrek hastalığı kayıtlıysa değer 1, değilse 0 olur.  
> Görev: Her metni Türkçe kurallarına göre küçük harfe çevir: Önce I'yı ı, İ'yi i yap, ardından öteki harfleri küçült. Her tanının şu kelimelerini tam kelime olarak ara; böylece hiçbir kelime daha uzun bir kelimenin içinde bulunmaz: Kalp yetersizliği için "kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky" ve "kky"; atriyal fibrilasyon için "atriyal fibrilasyon", "af" ve "afib"; koroner arter hastalığı için "koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü" ve "akut koroner sendrom"; hipertansiyon için "hipertansiyon" ve "ht"; diyabetes mellitus için "diyabet", "diyabetes mellitus" ve "dm"; kronik böbrek hastalığı için "kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh" ve "kby". Bir geçişten sonra, cümlesinin sonuna ya da başka bir tanının bir sonraki geçişine kadar şu kelimelerden biri geliyorsa geçiş sayılmaz: "yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı" ya da "(-)" (olumsuz); "şüphe", "olası" ya da "ön tanı" (olası). Aynı cümlede geçişten önce şu kelimelerden biri duruyorsa da geçiş sayılmaz: "babası", "annesi", "kardeşi", "babada", "annede" ya da "soygeçmiş" (bir yakın). Geçişlerinden en az biri sayılan tanı var kabul edilir. Epikriz bir tanının ilaçlarından birini tam kelime ya da bir kelimenin başı olarak anıyorsa da tanı var kabul edilir: Kalp yetersizliği için "furosemid" ya da "sakubitril"; atriyal fibrilasyon için "apiksaban", "rivaroksaban", "edoksaban" ya da "dabigatran"; koroner arter hastalığı için "klopidogrel", "tikagrelor", "pkg", "kabg" ya da "stent"; hipertansiyon için "amlodipin" ya da "hidroklorotiyazid"; diyabetes mellitus için "metformin", "gliklazid", "insülin" ya da "sitagliptin". "egfr" ifadesinden sonraki ilk sayı 60'ın altındaysa kronik böbrek hastalığı, "hba1c" ifadesinden ve isteğe bağlı bir yüzde işaretinden sonraki ilk sayı 6,5 ya da üstündeyse diyabetes mellitus vardır; bu sayılarda ondalık ayırıcı virgüldür. Altı kararı kayıt sistemindeki altı sütunla karşılaştır; her tanı için duyarlılığı, pozitif öngörü değerini ve F1 değerini (doğru pozitiflerin iki katı, doğru pozitiflerin iki katı ile yanlış pozitiflerin ve yanlış negatiflerin toplamına bölünür) ve altı F1 değerinin ortalamasını, yani makro F1 değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her tanı için bir satır ve üç değer içeren bir tablo yazdır; makro F1 değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Kronik böbrek hastalığının duyarlılığı 1 olmalı ve VC-2444 numaralı hastanın epikrizi kalp yetersizliği, hipertansiyon ve kronik böbrek hastalığı için pozitif bulunmalı.  

In [ ]:
# Prompt 1.3 için veri: Promptun tarif ettiği "veri" tablosu
veri = _summaries(te)
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 1.3 kontrolü
teach.check_step("Prompt 1.3", "cevap", globals())

In [ ]:
#@title Prompt 1.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _summaries(te)
terimler = {"KY": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
           "AF": ["atriyal fibrilasyon", "af", "afib"],
           "KAH": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
           "HT": ["hipertansiyon", "ht"], "DM": ["diyabet", "diyabetes mellitus", "dm"],
           "KBH": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"]}


def turkce_kucuk(metin):
    return metin.replace("I", "ı").replace("İ", "i").lower()


def skor(kararlar):
    satirlar = []
    for s in terimler:
        gercek = veri[s].to_numpy() == 1
        tahmin = np.array(kararlar[s]) == 1
        dp, yp, yn = int((tahmin & gercek).sum()), int((tahmin & ~gercek).sum()), int((~tahmin & gercek).sum())
        satirlar.append({"tanı": s, "duyarlılık": round(dp / max(1, dp + yn), 3),
                         "PPV": round(dp / max(1, dp + yp), 3), "F1": round(2 * dp / max(1, 2 * dp + yp + yn), 3)})
    tablo = pd.DataFrame(satirlar)
    print(tablo)
    print("Makro F1:", round(float(tablo["F1"].mean()), 3))
    return tablo

olumsuz = ["yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)"]
olasi = ["şüphe", "olası", "ön tanı"]
yakin = ["babası", "annesi", "kardeşi", "babada", "annede", "soygeçmiş"]
ilaclar = {"KY": ["furosemid", "sakubitril"], "AF": ["apiksaban", "rivaroksaban", "edoksaban", "dabigatran"],
           "KAH": ["klopidogrel", "tikagrelor", "pkg", "kabg", "stent"], "HT": ["amlodipin", "hidroklorotiyazid"],
           "DM": ["metformin", "gliklazid", "insülin", "sitagliptin"]}


def var_olan(kucuk):
    gecisler = sorted((m.start(), m.end(), s) for s, kelimeler in terimler.items() for k in kelimeler
                      for m in re.finditer(r"(?<!\w)" + re.escape(k) + r"(?!\w)", kucuk))
    bulunan = set()
    for i, (bas, son, s) in enumerate(gecisler):
        sonraki = next((m[0] for m in gecisler[i + 1:] if m[2] != s), len(kucuk))
        sinirlar = [x for x in (kucuk.find(".", son), kucuk.find(";", son), sonraki, len(kucuk)) if x >= son]
        sonra = kucuk[son:min(sinirlar)]
        cumle_basi = max(kucuk.rfind(". ", 0, bas), kucuk.rfind(": ", 0, bas)) + 1
        if any(k in kucuk[cumle_basi:bas] for k in yakin):
            continue
        if any(k in sonra for k in olumsuz + olasi):
            continue
        bulunan.add(s)
    for s, kelimeler in ilaclar.items():
        if any(re.search(r"(?<!\w)" + re.escape(k), kucuk) for k in kelimeler):
            bulunan.add(s)
    egfr = re.search(r"egfr\s*%?\s*(\d+(?:,\d+)?)", kucuk)
    if egfr and float(egfr.group(1).replace(",", ".")) < 60:
        bulunan.add("KBH")
    hba1c = re.search(r"hba1c\s*%?\s*(\d+(?:,\d+)?)", kucuk)
    if hba1c and float(hba1c.group(1).replace(",", ".")) >= 6.5:
        bulunan.add("DM")
    return bulunan


kararlar = {s: [] for s in terimler}
for metin in veri["epikriz"]:
    bulunan = var_olan(turkce_kucuk(metin))
    for s in terimler:
        kararlar[s].append(int(s in bulunan))
tablo = skor(kararlar)
cevap = round(float(tablo["F1"].mean()), 3)
teach.check_step("Prompt 1.3", "cevap", globals())

**Sonuç ne gösteriyor?** Makro F1 0,98 değerine ulaşır. Diyabet ve kronik böbrek hastalığı artık her hastada bulunur. En zayıf tanı hipertansiyon olarak kalır: "esansiyel hipertansiyon" ve "hipertansyon" kelime listesinde yoktur; ikisini de anmayan epikrizlerdeki ilaçlar, yani ADE inhibitörleri ve anjiyotensin reseptör blokerleri ise kalp yetersizliğinde de verilir.

#### Prompt 1.4: Epikrizlerden öğrenen bir model: TF-IDF ve lojistik regresyon

**Amaç.** Bu promptla A hastanesinin epikrizleriyle bir model eğitilir ve A ile B hastanelerinde değerlendirilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Epikrizlerde altı kronik tanıyı örneklerden öğrenerek bulan bir model eğitmek ve bu modelin kendi hastanesinde ve başka bir üslupla yazan başka bir hastanede ne kadar iyi çalıştığını öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; sentetik bir kayıt sistemi olan sanal kliniğin 3.600 epikrizini tutar. Her satır bir hastadır: "küme" sütunu epikrizin kümesini verir ("eğitim": 1.800 epikriz; "doğrulama": 600; "test": 600; bu üçü A hastanesindendir; "dış": B hastanesinden, (+) ve (-) işaretli kısa bir üslupla yazılmış 600 epikriz). "epikriz" sütunu Türkçe epikrizi tutar; "KY", "AF", "KAH", "HT", "DM" ve "KBH" adlı altı sütun 0 ya da 1 değerini alır; kayıt sisteminde sırasıyla kalp yetersizliği, atriyal fibrilasyon, koroner arter hastalığı, hipertansiyon, diyabetes mellitus ve kronik böbrek hastalığı kayıtlıysa değer 1, değilse 0 olur.  
> Görev: Her metni Türkçe kurallarına göre küçük harfe çevir: Önce I'yı ı, İ'yi i yap, ardından öteki harfleri küçült. Her epikrizi, kelimelerinin ve yan yana duran kelime çiftlerinin TF-IDF ağırlıklarından oluşan bir vektöre dönüştür; kelime, harf ve rakamlardan oluşan kesintisiz bir dizidir ve yalnızca en az iki eğitim epikrizinde geçen terimler tutulur. Terimleri ve ağırlıklarını yalnızca eğitim epikrizlerinden öğren; bir terimin sıklığı olarak sayısının bir fazlasının logaritmasını kullan ve her vektörü 1 uzunluğuna ölçekle. Her tanı için eğitim epikrizlerinde ters düzenlileştirme gücü 10 olan bir lojistik regresyon eğit. Doğrulama epikrizlerinde her tanı için olasılık eşiğini, 0,05'ten 0,95'e 0,05'lik adımlarla, F1 değerinin en yüksek olduğu değer olarak seç. Bu eşiklerle her tanının F1 değerini ve makro F1 değerini test epikrizlerinde ve dış epikrizlerde hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her tanının test ve dış epikrizlerdeki F1 değerini gösteren bir tablo yazdır; dış epikrizlerdeki makro F1 değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Terimler yalnızca eğitim epikrizlerinden, eşikler yalnızca doğrulama epikrizlerinden gelmeli; test epikrizlerindeki makro F1 değeri 0,9'un üstünde olmalı.  

In [ ]:
# Prompt 1.4 için veri: Promptun tarif ettiği "veri" tablosu
veri = _summaries(np.ones(len(texts), dtype=bool), with_set=True)
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 1.4 kontrolü
teach.check_step("Prompt 1.4", "cevap", globals())

In [ ]:
#@title Prompt 1.4 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _summaries(np.ones(len(texts), dtype=bool), with_set=True)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

sutunlar = ["KY", "AF", "KAH", "HT", "DM", "KBH"]
kumeler = {k: veri[veri["küme"] == k] for k in ("eğitim", "doğrulama", "test", "dış")}
vektorlestirici = TfidfVectorizer(preprocessor=lambda t: t.replace("I", "ı").replace("İ", "i").lower(), token_pattern=r"(?u)\b\w+\b",
                             ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X = {"eğitim": vektorlestirici.fit_transform(kumeler["eğitim"]["epikriz"])}
for k in ("doğrulama", "test", "dış"):
    X[k] = vektorlestirici.transform(kumeler[k]["epikriz"])


def f1(gercek, tahmin):
    gercek, tahmin = np.asarray(gercek) == 1, np.asarray(tahmin)
    dp = int((tahmin & gercek).sum())
    return 2 * dp / max(1, 2 * dp + int((tahmin & ~gercek).sum()) + int((~tahmin & gercek).sum()))


izgara = np.round(np.arange(0.05, 0.96, 0.05), 2)
satirlar = []
for s in sutunlar:
    model = LogisticRegression(C=10, max_iter=2000).fit(X["eğitim"], kumeler["eğitim"][s])
    p_dog = model.predict_proba(X["doğrulama"])[:, 1]
    esik = float(izgara[int(np.argmax([f1(kumeler["doğrulama"][s], p_dog >= t) for t in izgara]))])
    satir = {"tanı": s, "eşik": esik}
    for k in ("test", "dış"):
        satir[f"F1 {k}"] = round(f1(kumeler[k][s], model.predict_proba(X[k])[:, 1] >= esik), 3)
    satirlar.append(satir)
tablo = pd.DataFrame(satirlar)
print(tablo)
print("Makro F1, test:", round(float(tablo["F1 test"].mean()), 3))
cevap = round(float(tablo["F1 dış"].mean()), 3)
print("Makro F1, dış:", cevap)
teach.check_step("Prompt 1.4", "cevap", globals())

**Sonuç ne gösteriyor?** Model A hastanesinin test kümesinde neredeyse kusursuzdur, B hastanesinde ise yalnızca 0,708 değerine ulaşır. En ağır terimleri A hastanesinin yazma biçiminden, EKG cümlesinden bir kalıptan ya da bir dozdan gelir; B hastanesinin "AF (-)" gibi ifadeler içeren kısa üslubu ise modele hastalık sandığı kelimeler verir.

**«Durak 1 · Prompt ile uygulama» tamamlandı.** Sırada hemen aşağıdaki «Durak 1 · Hazır uygulama» bölümü var. Durağı burada bırakacaksanız [ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#last-two-hours-advanced-clinical-nlp-applications) ve okumaya «Son 2 saat: Gelişmiş klinik NLP uygulamaları» bölümünden devam edin.

### Durak 1 · Hazır uygulama

Bu bölüm sanal kliniğin rapor arşivini okur, altı tanı için bir anahtar kelime kuralı ve ilaçlarla laboratuvar değerlerini de kullanan bir durum kuralı kurar, A hastanesinin epikrizleri üzerinde lojistik regresyonlu bir TF-IDF modeli eğitir ve hepsini B hastanesinde değerlendirir. Bölüm lemmatizasyon ve arşivden öğrenilen kelime vektörleriyle sona erer.

Hücreleri yukarıdan aşağıya çalıştırın. *Python notu* hücreleri kodu açıklar, *Gözlemle ve düşün* hücreleri ise sonuçlar üzerine sorular sorar.

**Başlangıç hücresi.** Aşağıdaki hücre, «Durak 1 · Hazır uygulama» bölümü için kurulumu, sonraki hücrelerin kullandığı veri çağrılarını ve önceki hücrelerin kodlarını yineler; böylece kodlar hata vermeden çalışır. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Durak 1 · Hazır uygulama» bölümü
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# alıştırma promptlarının tabloları (etkileşimli labın F bölümü)
from vclinic.week06 import alistirma_tablosu

# önceki hücrelerden yinelenen kod: Aşağıdaki hücrelerin kullandığı veriler ve kodlar
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

#### Sanal kliniğin rapor arşivi

*Ders sayfasında: [Veri olarak metin: Sanal kliniğin rapor arşivi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#text-as-data-the-report-archive-of-the-virtual-clinic)*

Sanal klinik, bir kayıt sistemindeki 3.600 hastanın her biri için bir epikriz yazar: A hastanesinden Türkçe ve İngilizce, tam cümlelerle yazılmış 3.000 epikriz ve B hastanesinden kısa bir üslupla yazılmış 600 epikriz. Kayıt sistemi her hastanın altı tanısını bilir; böylece haftanın her yöntemi bu bilgiyle karşılaştırılabilir.

**Formüllerden koda.** Tabloda her hasta için bir satır vardır; `text_tr` Türkçe epikrizdir, `HF`, `AF`, `CAD`, `HT`, `DM` ve `CKD` adlı altı sütun ise 1 ya da 0'dır. Bu sütunlar sırasıyla kalp yetersizliğini, atriyal fibrilasyonu, koroner arter hastalığını, hipertansiyonu, diyabetes mellitusu ve kronik böbrek hastalığını gösterir; kısaltmalar hastalıkların İngilizce adlarından gelir. `split` sütunu kümeyi verir: `training` eğitim, `validation` doğrulama, `test` test, `external` ise dış kümedir. Bir tanının bir kümedeki prevalansı, sütununun ortalamasıdır: `archive[archive["split"] == "test"]["AF"].mean()`.

In [ ]:
archive = reports.archive()
LABELS = list(reports.LABELS)
texts = archive["text_tr"].to_numpy()
y = archive[LABELS].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")


def tr_lower(text):
    """Türkçe kurallarına göre küçük harf: I, ı olur; İ, i olur ve metnin uzunluğu değişmez."""
    return text.replace("I", "ı").replace("İ", "i").lower()


def words(text):
    """Bir metnin küçük harfli kelimeleri: Harf ve rakam dizileri."""
    return re.findall(r"\w+", tr_lower(text))

print(archive.shape)
print(archive.groupby("split")[LABELS].sum())
case = int(np.where(archive["patient_id"] == "VC-2444")[0][0])
print()
print(texts[case])

**Python notu.** `reports.archive()` arşivi birkaç saniyede yazar ve bir tablo olarak döndürür. `groupby("split")[LABELS].sum()` her kümede her tanıya sahip hastaları sayar. `texts`, `y` ve `split`, sütunların dizi biçimidir; `tr`, `va`, `te` ve `ex` adlı dört maske ise önceki haftalarda olduğu gibi eğitim, doğrulama, test ve dış kümelerin satırlarını seçer. Kod hücresindeki `words` fonksiyonu bir metnin kelimelerini `tr_lower` uygulandıktan sonra döndürür.

In [ ]:
n_words = np.array([len(words(t)) for t in texts])
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.hist(n_words[tr | va | te], bins=40, alpha=0.7, label="A hastanesi")
ax.hist(n_words[ex], bins=40, alpha=0.7, label="B hastanesi")
ax.set_xlabel("epikriz başına kelime")
ax.set_ylabel("epikriz")
ax.legend()
plt.show()
print("B hastanesi, ilk epikriz:", texts[ex][0])

**Gözlemle ve düşün.** Vakanın epikrizini ve B hastanesinden bir epikrizi okuyun. Her birinde altı tanıdan hangilerini, hangi kelimelerden bulabiliyorsunuz? Hastalıkların adlarını arayan bir arama bunlardan hangilerini kaçırırdı?

**Alıştırma A.** A hastanesinin test kümesindeki hastalardan kaçının kayıt sisteminde altı tanıdan en az üçü vardır?

In [ ]:
veri = alistirma_tablosu("A")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma A", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int((y[te].sum(axis=1) >= 3).sum())
teach.check("Alıştırma A", cevap)

#### Bir anahtar kelime kuralı ve tuzakları

*Ders sayfasında: [Önce bir kural: Kelime aramak ve tuzakları](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#a-rule-first-searching-for-words-and-its-traps)*

İlk kural, bir tanının kelimelerinden biri Türkçe küçük harfe çevrilmiş epikrizin herhangi bir yerinde tam kelime olarak geçiyorsa o tanıyı var sayar. Kural her tanı için duyarlılık, pozitif öngörü değeri ve F1 ile, altı tanının tamamı için ise makro F1 ile değerlendirilir.

**Formüllerden koda.** $F_1 = 2\,DP/(2\,DP + YP + YN)$: `se_ppv_f1` fonksiyonunun üçüncü değeri. Makro F1, `label_table` tablosundaki `F1` sütununun ortalamasıdır. Tam kelime, `PATTERNS` içindeki `(?<!\w)(...)(?!\w)` kalıbıyla aranır.

**Başlangıç hücresi.** Aşağıdaki hücre, «Bir anahtar kelime kuralı ve tuzakları» kısmı için kurulumu, sonraki hücrelerin kullandığı veri çağrılarını ve önceki hücrelerin kodlarını yineler; böylece kodlar hata vermeden çalışır. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Bir anahtar kelime kuralı ve tuzakları» kısmı
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# alıştırma promptlarının tabloları (etkileşimli labın F bölümü)
from vclinic.week06 import alistirma_tablosu

# önceki hücrelerden yinelenen kod: Aşağıdaki hücrelerin kullandığı veriler ve kodlar
import re
archive = reports.archive()
LABELS = list(reports.LABELS)
texts = archive["text_tr"].to_numpy()
y = archive[LABELS].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
def tr_lower(text):
    """Türkçe kurallarına göre küçük harf: I, ı olur; İ, i olur ve metnin uzunluğu değişmez."""
    return text.replace("I", "ı").replace("İ", "i").lower()
case = int(np.where(archive["patient_id"] == "VC-2444")[0][0])

**Ders sayfasında:** [Çözümlü örnek: Anahtar kelime kuralının atriyal fibrilasyon için F1 değeri](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#worked-example-f1-of-the-keyword-rule-for-atrial-fibrillation). Aşağıdaki hücre anahtar kelime kuralının altı tanı için duyarlılığını, pozitif öngörü değerini ve F1 değerini hesaplar.

In [ ]:
def se_ppv_f1(y_true, y_pred):
    """Evet ya da hayır biçimindeki bir kararın duyarlılığı, pozitif öngörü değeri ve F1 değeri."""
    y_true, y_pred = np.asarray(y_true, dtype=bool), np.asarray(y_pred, dtype=bool)
    tp, fp, fn = int((y_pred & y_true).sum()), int((y_pred & ~y_true).sum()), int((~y_pred & y_true).sum())
    se = tp / max(1, tp + fn)
    ppv = tp / max(1, tp + fp)
    f1 = 2 * tp / max(1, 2 * tp + fp + fn)
    return se, ppv, f1


def label_table(y_true, y_pred):
    """Her tanı için bir satır: Prevalans, duyarlılık, PPV ve F1; son satır ortalama F1'dir (makro F1)."""
    rows = []
    for k, label in enumerate(LABELS):
        se, ppv, f1 = se_ppv_f1(y_true[:, k], y_pred[:, k])
        rows.append([label, int(y_true[:, k].sum()), round(se, 3), round(ppv, 3), round(f1, 3)])
    table = pd.DataFrame(rows, columns=["tanı", "tanılı", "duyarlılık", "PPV", "F1"])
    print("Makro F1:", round(float(table["F1"].mean()), 3))
    return table


TERMS = {
    "HF": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
    "AF": ["atriyal fibrilasyon", "af", "afib"],
    "CAD": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
    "HT": ["hipertansiyon", "ht"],
    "DM": ["diyabet", "diyabetes mellitus", "dm"],
    "CKD": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"],
}
PATTERNS = {label: re.compile(r"(?<!\w)(" + "|".join(sorted(map(re.escape, terms), key=len, reverse=True)) + r")(?!\w)")
            for label, terms in TERMS.items()}


def find_terms(text):
    """Bir tanının metindeki her geçişi: (başlangıç, bitiş, tanı), metindeki sırasıyla."""
    low = tr_lower(text)
    found = [(m.start(), m.end(), label) for label, pattern in PATTERNS.items() for m in pattern.finditer(low)]
    return sorted(found)


def keyword_labels(text):
    """Bir tanının kelimelerinden biri raporun herhangi bir yerinde geçiyorsa tanı var sayılır."""
    present = {label for _, _, label in find_terms(text)}
    return [int(label in present) for label in LABELS]

keyword_test = np.array([keyword_labels(t) for t in texts[te]])
print(label_table(y[te], keyword_test))
print()
print("VC-2444 vakası, anahtar kelime kuralı:", dict(zip(LABELS, keyword_labels(texts[case]))))
print("VC-2444 vakası, kayıt sistemi:     ", dict(zip(LABELS, y[case])))

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c133)*

**Python notu.** `TERMS`, bir tanıdan o tanının kelime listesine giden bir sözlüktür. `PATTERNS` her tanı için kelimeleri "ya da" anlamına gelen `|` ile tek bir düzenli ifadede birleştirir; daha uzun kelimeler önce gelir, böylece "kalp yetmezliği" içindeki daha kısa bir kelimeye üstün gelir. `find_terms` her eşleşmeyi (başlangıç, bitiş, tanı) olarak döndürür; `keyword_labels` ise bulunan tanılar kümesini 0 ya da 1 değerlerinden oluşan altı sayılık bir listeye dönüştürür. `label_table` altı sütunlu iki diziyi karşılaştırır ve tabloyu pandas ile kurar.

In [ ]:
# Kuralın atriyal fibrilasyon olmadan AF dediği yerler
k = LABELS.index("AF")
false_af = np.where(te)[0][(keyword_test[:, k] == 1) & (y[te, k] == 0)]
print(len(false_af), "AF için yanlış alarm; üç cümle:")
for i in false_af[:3]:
    start, end, _ = next(m for m in find_terms(texts[i]) if m[2] == "AF")
    print("  ...", texts[i][max(0, start - 20):end + 30], "...")

**Gözlemle ve düşün.** Üç cümle olumsuzluk tuzağını göstermektedir. Koroner arter hastalığı için yanlış alarmlara da bakın ("AF" yerine "CAD" yazın). Orada hangi başka tuzak görülüyor?

**Alıştırma B.** Anahtar kelime kuralının A hastanesinin test kümesinde diyabetes mellitus (`DM`) için pozitif öngörü değeri kaçtır? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("B")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma B", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
k = LABELS.index("DM")
cevap = round(se_ppv_f1(y[te, k], keyword_test[:, k])[1], 3)
teach.check("Alıştırma B", cevap)

#### Durum, ilaçlar ve laboratuvar değerleri

*Ders sayfasında: [Olumsuzluk, aile öyküsü ve belirsizlik: Bir geçişin durumu](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#negation-family-history-and-uncertainty-the-assertion-of-a-mention)*

Bir tanının her geçişi bir durum alır. Geçişten sonra, cümlenin sonuna ya da başka bir tanının bir sonraki geçişine kadar gelen bir olumsuzluk ya da belirsizlik tetikleyicisi geçişi yok ya da olası yapar; cümlede daha önce geçen bir yakın akraba ise geçişi aile öyküsüne verir. Ardından ilaçlar ve iki laboratuvar değeri, metnin adını anmadığı tanıları ekler.

**Formüllerden koda.** Kapsam, geçişin sonundan dört konumdan ilkine kadar uzanan metindir: Bir sonraki nokta, bir sonraki noktalı virgül, `next_other(...)` ve metnin sonu; bunlar `assertion` fonksiyonundadır. $w$ karakterlik bir pencere bunun yerine `end + window` kullanır. Laboratuvar eşikleri `rule_labels` içindeki `egfr < 60` ve `hba1c >= 6.5` koşullarıdır.

**Başlangıç hücresi.** Aşağıdaki hücre, «Durum, ilaçlar ve laboratuvar değerleri» kısmı için kurulumu, sonraki hücrelerin kullandığı veri çağrılarını ve önceki hücrelerin kodlarını yineler; böylece kodlar hata vermeden çalışır. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Durum, ilaçlar ve laboratuvar değerleri» kısmı
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# alıştırma promptlarının tabloları (etkileşimli labın F bölümü)
from vclinic.week06 import alistirma_tablosu

# önceki hücrelerden yinelenen kod: Aşağıdaki hücrelerin kullandığı veriler ve kodlar
import re
archive = reports.archive()
LABELS = list(reports.LABELS)
texts = archive["text_tr"].to_numpy()
y = archive[LABELS].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
def tr_lower(text):
    """Türkçe kurallarına göre küçük harf: I, ı olur; İ, i olur ve metnin uzunluğu değişmez."""
    return text.replace("I", "ı").replace("İ", "i").lower()
case = int(np.where(archive["patient_id"] == "VC-2444")[0][0])
def se_ppv_f1(y_true, y_pred):
    """Evet ya da hayır biçimindeki bir kararın duyarlılığı, pozitif öngörü değeri ve F1 değeri."""
    y_true, y_pred = np.asarray(y_true, dtype=bool), np.asarray(y_pred, dtype=bool)
    tp, fp, fn = int((y_pred & y_true).sum()), int((y_pred & ~y_true).sum()), int((~y_pred & y_true).sum())
    se = tp / max(1, tp + fn)
    ppv = tp / max(1, tp + fp)
    f1 = 2 * tp / max(1, 2 * tp + fp + fn)
    return se, ppv, f1
def label_table(y_true, y_pred):
    """Her tanı için bir satır: Prevalans, duyarlılık, PPV ve F1; son satır ortalama F1'dir (makro F1)."""
    rows = []
    for k, label in enumerate(LABELS):
        se, ppv, f1 = se_ppv_f1(y_true[:, k], y_pred[:, k])
        rows.append([label, int(y_true[:, k].sum()), round(se, 3), round(ppv, 3), round(f1, 3)])
    table = pd.DataFrame(rows, columns=["tanı", "tanılı", "duyarlılık", "PPV", "F1"])
    print("Makro F1:", round(float(table["F1"].mean()), 3))
    return table
TERMS = {
    "HF": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
    "AF": ["atriyal fibrilasyon", "af", "afib"],
    "CAD": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
    "HT": ["hipertansiyon", "ht"],
    "DM": ["diyabet", "diyabetes mellitus", "dm"],
    "CKD": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"],
}
PATTERNS = {label: re.compile(r"(?<!\w)(" + "|".join(sorted(map(re.escape, terms), key=len, reverse=True)) + r")(?!\w)")
            for label, terms in TERMS.items()}
def find_terms(text):
    """Bir tanının metindeki her geçişi: (başlangıç, bitiş, tanı), metindeki sırasıyla."""
    low = tr_lower(text)
    found = [(m.start(), m.end(), label) for label, pattern in PATTERNS.items() for m in pattern.finditer(low)]
    return sorted(found)

**Ders sayfasında:** [Çözümlü örnek: "KY (-) HT (+) DM (-)" ifadesinde pencere ve kapsam](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#worked-example-a-window-and-a-scope-in-ky-ht-dm). Aşağıdaki hücre her geçişin durumunu aynı kapsamla belirler.

In [ ]:
NEGATION = ["yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)"]
POSSIBLE = ["şüphe", "olası", "ön tanı"]
FAMILY = ["babası", "annesi", "kardeşi", "babada", "annede", "soygeçmiş"]


def assertion(text, start, end, next_start, window=None):
    """Bir geçişin durumu: var (present), yok (negated), olası (possible) ya da aile öyküsü (family).

    Geçişten sonraki kapsam, cümle sonunda ya da başka bir tanının bir sonraki geçişinin başladığı yerde (next_start) biter; window verilirse bunun yerine o kadar karakter sonra biter."""
    low = tr_lower(text)
    if window is None:
        stop = min(x for x in (low.find(".", end), low.find(";", end), next_start, len(low)) if x >= end)
    else:
        stop = min(len(low), end + window)
    after = low[end:stop]
    sentence_start = max(low.rfind(". ", 0, start), low.rfind(": ", 0, start)) + 1
    before = low[sentence_start:start]
    if any(w in before for w in FAMILY):
        return "family"
    if any(w in after for w in NEGATION):
        return "negated"
    if any(w in after for w in POSSIBLE):
        return "possible"
    return "present"


def next_other(mentions, i, text):
    """Başka bir tanının bir sonraki geçişinin başladığı yer; yoksa metnin sonu."""
    return next((m[0] for m in mentions[i + 1:] if m[2] != mentions[i][2]), len(text))


def assertion_labels(text, window=None):
    """Geçişlerinden en az biri var durumunda olan tanı var sayılır."""
    mentions = find_terms(text)
    present = set()
    for i, (start, end, label) in enumerate(mentions):
        if assertion(text, start, end, next_other(mentions, i, text), window) == "present":
            present.add(label)
    return [int(label in present) for label in LABELS]

mentions = find_terms(texts[case])
for i, (start, end, label) in enumerate(mentions):
    print(f"{label:4} {texts[case][start:end]:28} → {assertion(texts[case], start, end, next_other(mentions, i, texts[case]))}")
b = np.where(ex)[0][1]
print()
print(texts[b][:60])
for i, (start, end, label) in enumerate(find_terms(texts[b])):
    ms = find_terms(texts[b])
    print(f"{label:4} kapsam: {assertion(texts[b], start, end, next_other(ms, i, texts[b])):8} 40 karakterlik pencere: {assertion(texts[b], start, end, 0, window=40)}")

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c143)*

**Python notu.** `assertion` metnin Türkçe küçük harfli biçimi üzerinde çalışır. `low.find(".", end)` geçişten sonraki ilk noktanın konumunu, nokta yoksa −1 değerini verir; liste üreteci geçişten sonra gelen konumları tutar, `min` ise en yakın olanı alır. `low.rfind(". ", 0, start)` cümlenin başını geriye doğru arar. `any(w in after for w in NEGATION)` kapsamda herhangi bir tetikleyicinin geçip geçmediğini sorar. İkinci epikrizde "ht" ifadesinden sonraki 40 karakterlik pencere bir sonraki tanının (-) işaretine ulaşır.

In [ ]:
EVIDENCE = {
    "HF": ["furosemid", "sakubitril"],
    "AF": ["apiksaban", "rivaroksaban", "edoksaban", "dabigatran"],
    "CAD": ["klopidogrel", "tikagrelor", "pkg", "kabg", "stent"],
    "HT": ["amlodipin", "hidroklorotiyazid"],
    "DM": ["metformin", "gliklazid", "insülin", "sitagliptin"],
}


def lab_value(text, name):
    """Bir laboratuvar adından sonraki ilk sayı, örneğin egfr 54 ya da hba1c %7,8; yoksa None."""
    m = re.search(name + r"\s*%?\s*(\d+(?:,\d+)?)", tr_lower(text))
    return float(m.group(1).replace(",", ".")) if m else None


def rule_labels(text, window=None):
    """Durumlarıyla birlikte geçişler, ilaçlar ve iki laboratuvar değeri."""
    labels = assertion_labels(text, window)
    low = tr_lower(text)
    for k, label in enumerate(LABELS):
        if any(re.search(r"(?<!\w)" + re.escape(w), low) for w in EVIDENCE.get(label, [])):
            labels[k] = 1
    egfr, hba1c = lab_value(text, "egfr"), lab_value(text, "hba1c")
    if egfr is not None and egfr < 60:
        labels[LABELS.index("CKD")] = 1
    if hba1c is not None and hba1c >= 6.5:
        labels[LABELS.index("DM")] = 1
    return labels

for name, mask in (("test, A hastanesi", te), ("dış, B hastanesi", ex)):
    print(name)
    print(label_table(y[mask], np.array([rule_labels(t) for t in texts[mask]])))
print("VC-2444 vakası:", dict(zip(LABELS, rule_labels(texts[case]))))

**Python notu.** `EVIDENCE` bir tanıya işaret eden ilaçları listeler. `(?<!\w)` + ad kalıbı bir ilacı tam kelime ya da bir kelimenin başı olarak bulur; böylece "insülin" "insülin glarjin" ile de eşleşir. `lab_value` bir addan sonraki ilk sayıyı, isteğe bağlı yüzde işareti ve Türkçe ondalık virgülüyle birlikte okur; ad yoksa `None` döndürür. `if egfr is not None and egfr < 60:` iki koşulu birlikte denetler. `rule_labels`, `assertion_labels` sonucundan başlar ve kanıtları bunun üzerine ekler.

**Gözlemle ve düşün.** Kural A hastanesinde yaklaşık her yedi hastadan birinde hipertansiyonu kaçırır. Bu epikrizlerden birkaçını yazdırın. Hangi kelimeleri ya da ilaçları eklerdiniz ve eklediklerinizden hangileri yanlış alarm da üretirdi?

**Alıştırma C.** Dersin kuralı A hastanesinin 600 test epikrizindeki tanı geçişlerinden kaçını yok (negated) sayar?

In [ ]:
veri = alistirma_tablosu("C")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma C", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = 0
for t in texts[te]:
    ms = find_terms(t)
    cevap += sum(assertion(t, s, e, next_other(ms, i, t)) == "negated" for i, (s, e, l) in enumerate(ms))
teach.check("Alıştırma C", cevap)

#### TF-IDF ve lojistik regresyon

*Ders sayfasında: [Kelime torbası ve TF-IDF: Vektör olarak bir epikriz](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#bag-of-words-and-tf-idf)*

Bir model tanıları eğitim epikrizlerinden öğrenir. Her epikriz, kelimelerinin ve kelime çiftlerinin TF-IDF ağırlıklarından oluşan bir vektöre dönüşür; her tanı için bir lojistik regresyon her terim için bir ağırlık öğrenir ve eşiği doğrulama epikrizlerinde seçilir.

**Formüllerden koda.** $\mathrm{idf}_t = \ln((1 + N)/(1 + \mathrm{df}_t)) + 1$: `vectorizer.idf_`. `sublinear_tf=True` ile bir terimin sıklığı $1 + \ln \mathrm{tf}$ olarak girer ve her vektör 1 uzunluğuna ölçeklenir. Tek kelimeler ve kelime çiftleri: `ngram_range=(1, 2)`. En az iki eğitim epikrizinde geçen terimler: `min_df=2`. Sözlük ve ağırlıklar yalnızca eğitim epikrizlerinde `fit_transform` ile öğrenilir ve ötekilere `transform` ile uygulanır.

**Başlangıç hücresi.** Aşağıdaki hücre, «TF-IDF ve lojistik regresyon» kısmı için kurulumu, sonraki hücrelerin kullandığı veri çağrılarını ve önceki hücrelerin kodlarını yineler; böylece kodlar hata vermeden çalışır. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «TF-IDF ve lojistik regresyon» kısmı
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# alıştırma promptlarının tabloları (etkileşimli labın F bölümü)
from vclinic.week06 import alistirma_tablosu

# önceki hücrelerden yinelenen kod: Aşağıdaki hücrelerin kullandığı veriler ve kodlar
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
archive = reports.archive()
LABELS = list(reports.LABELS)
texts = archive["text_tr"].to_numpy()
y = archive[LABELS].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
def tr_lower(text):
    """Türkçe kurallarına göre küçük harf: I, ı olur; İ, i olur ve metnin uzunluğu değişmez."""
    return text.replace("I", "ı").replace("İ", "i").lower()
def words(text):
    """Bir metnin küçük harfli kelimeleri: Harf ve rakam dizileri."""
    return re.findall(r"\w+", tr_lower(text))
def se_ppv_f1(y_true, y_pred):
    """Evet ya da hayır biçimindeki bir kararın duyarlılığı, pozitif öngörü değeri ve F1 değeri."""
    y_true, y_pred = np.asarray(y_true, dtype=bool), np.asarray(y_pred, dtype=bool)
    tp, fp, fn = int((y_pred & y_true).sum()), int((y_pred & ~y_true).sum()), int((~y_pred & y_true).sum())
    se = tp / max(1, tp + fn)
    ppv = tp / max(1, tp + fp)
    f1 = 2 * tp / max(1, 2 * tp + fp + fn)
    return se, ppv, f1
def label_table(y_true, y_pred):
    """Her tanı için bir satır: Prevalans, duyarlılık, PPV ve F1; son satır ortalama F1'dir (makro F1)."""
    rows = []
    for k, label in enumerate(LABELS):
        se, ppv, f1 = se_ppv_f1(y_true[:, k], y_pred[:, k])
        rows.append([label, int(y_true[:, k].sum()), round(se, 3), round(ppv, 3), round(f1, 3)])
    table = pd.DataFrame(rows, columns=["tanı", "tanılı", "duyarlılık", "PPV", "F1"])
    print("Makro F1:", round(float(table["F1"].mean()), 3))
    return table

**Ders sayfasında:** [Çözümlü örnek: Üç kısa epikrizin TF-IDF değerleri](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#worked-example-tf-idf-of-three-short-summaries). Aşağıdaki hücre aynı üç epikrizin ağırlıklarını hesaplar.

In [ ]:
# Dersteki çözümlü örnek: Üç kısa epikriz
mini = ["Atriyal fibrilasyon izlenmedi.", "Atriyal fibrilasyon. Apiksaban başlandı.", "Hipertansiyon. Amlodipin başlandı."]
mini_vectorizer = TfidfVectorizer(preprocessor=tr_lower, token_pattern=r"(?u)\b\w+\b")
X_mini = mini_vectorizer.fit_transform(mini)
print(pd.DataFrame(X_mini.toarray().round(3), columns=mini_vectorizer.get_feature_names_out()))
print(dict(zip(mini_vectorizer.get_feature_names_out(), mini_vectorizer.idf_.round(3))))

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c154)*

**Python notu.** `TfidfVectorizer` dersteki hesabın tamamını yapar: Terimleri `token_pattern` kalıbıyla, burada `tr_lower` uygulandıktan sonraki her harf ve rakam dizisi olarak bulur, sayar, idf ile çarpar ve her satırı 1 uzunluğuna ölçekler. `fit_transform` yalnızca 0 olmayan değerleri saklayan *seyrek* bir matris döndürür; `.toarray()` bunu yazdırmak için olağan bir diziye dönüştürür. İlk satır, çözümlü örnekteki gibi 0,518; 0,518 ve 0,681 değerlerini verir.

In [ ]:
vectorizer = TfidfVectorizer(preprocessor=tr_lower, token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train = vectorizer.fit_transform(texts[tr])
X_val, X_test, X_ext = vectorizer.transform(texts[va]), vectorizer.transform(texts[te]), vectorizer.transform(texts[ex])
models, thresholds = {}, {}
GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)
for k, label in enumerate(LABELS):
    models[label] = LogisticRegression(C=10, max_iter=2000).fit(X_train, y[tr, k])
    p_val = models[label].predict_proba(X_val)[:, 1]
    f1_val = [se_ppv_f1(y[va, k], p_val >= t)[2] for t in GRID]
    thresholds[label] = float(GRID[int(np.argmax(f1_val))])


def tfidf_labels(X):
    """Her tanı için evet ya da hayır: Modelinin olasılığı, doğrulamada seçilen eşikle karşılaştırılır."""
    return np.column_stack([(models[label].predict_proba(X)[:, 1] >= thresholds[label]).astype(int) for label in LABELS])

print(X_train.shape, "terim;", "eşikler:", thresholds)
for name, X, mask in (("test, A hastanesi", X_test, te), ("dış, B hastanesi", X_ext, ex)):
    print(name)
    print(label_table(y[mask], tfidf_labels(X)))

**Python notu.** Döngü her tanı için `X_train` ve o tanının sütunu olan `y[tr, k]` üzerinde bir `LogisticRegression` eğitir. `C=10`, 2. haftadaki düzenlileştirme gücünün tersidir: Daha büyük bir C ağırlıkların büyümesine izin verir. `predict_proba(X_val)[:, 1]` her doğrulama epikrizi için tanının olasılığını verir; 0,05'ten 0,95'e uzanan ızgarada F1 değeri en yüksek olan eşik `thresholds` içinde saklanır. `tfidf_labels` altı modeli ve eşiği herhangi bir matrise uygular.

In [ ]:
names = vectorizer.get_feature_names_out()
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, label in zip(axes, ["AF", "HT", "CKD"]):
    w = models[label].coef_[0]
    top = np.argsort(w)[-8:]
    ax.barh(names[top], w[top], color="#1F5F8B")
    ax.set_title(label)
plt.tight_layout()
plt.show()

**Gözlemle ve düşün.** Atriyal fibrilasyon, hipertansiyon ve kronik böbrek hastalığının en ağır terimlerine bakın. Bir kardiyolog bunlardan hangilerini kanıt olarak kabul eder, hangileri yalnızca A hastanesinin nasıl yazdığını anlatır? "egfr 57" neden ağırlık taşır ve eGFR değeri 56 olan bir hastaya ne olur?

**Alıştırma D.** Modeli yalnızca tek kelimelerle, yani `ngram_range=(1, 1)` ile ve geri kalan her şeyi değiştirmeden tekrarlayın. Modelin B hastanesinin epikrizlerindeki makro F1 değeri kaçtır? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("D")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma D", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
v1 = TfidfVectorizer(preprocessor=tr_lower, token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 1), min_df=2, sublinear_tf=True)
X1_train, X1_val, X1_ext = v1.fit_transform(texts[tr]), v1.transform(texts[va]), v1.transform(texts[ex])
f1s = []
for k, label in enumerate(LABELS):
    m = LogisticRegression(C=10, max_iter=2000).fit(X1_train, y[tr, k])
    p_val = m.predict_proba(X1_val)[:, 1]
    t = GRID[int(np.argmax([se_ppv_f1(y[va, k], p_val >= g)[2] for g in GRID]))]
    f1s.append(se_ppv_f1(y[ex, k], m.predict_proba(X1_ext)[:, 1] >= t)[2])
cevap = round(float(np.mean(f1s)), 3)
teach.check("Alıştırma D", cevap)

**Lemmatizasyon: Bir kelime, birçok biçim.** *Ders sayfasında: [Tokenizasyon, lemmatizasyon](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#tokens-cutting-text-into-pieces)*

Türkçe bir köke ekler ekler; bu yüzden "yetmezlik", "yetmezliği" ve "yetmezliğinin" kelime torbasında üç ayrı sütundur. Bir *lemmatizer* her biçimi sözlükteki biçimine, yani *lemmasına* geri götürür. Aşağıdaki hücreler arşivin klinik kelimeleri için eğitimin küçük bir lemmatizer'ını kullanır. Lemmatizer kelimenin başında sözlüğündeki en uzun gövdeyi arar ve kalan kısmı ancak Türkçe eklerden oluşan bir zincirse kabul eder. Sondaki k'nin ünlüden önce ğ'ye dönüştüğünü ve "ritim" ile "kayıt" kelimelerinin bir ünlü düşürdüğünü ("ritmi", "kaydı") de bilir. Bu bölümdeki TF-IDF modeli ardından lemmalar üzerinde yeniden eğitilir.

**Formüllerden koda.** Gövdeler: Her lemmadan, yumuşamış biçiminden ve ünlüsü düşmüş biçiminden kurulan `STEMS`. Kelimenin geri kalanı: `SUFFIX.fullmatch(...)`. Lemmalardan oluşan epikrizler: `lemma_text`.

**Ders sayfasında:** [Çözümlü örnek: "yetmezliğinin" kelimesinin lemması](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#worked-example-the-lemma-of-yetmezli-inin). Aşağıdaki hücre aynı kelimenin ve arşivdeki öteki kelimelerin lemmasını bulur.

In [ ]:
LEMMAS = ["tedavi", "öykü", "poliklinik", "kontrol", "ritim", "hastalık", "fraksiyon", "fonksiyon", "ağrı", "darlık", "yetersizlik",
          "yetmezlik", "enfarktüs", "servis", "anjiyografi", "ödem", "yakınma", "ölçüm", "kardeş", "anne", "baba", "derivasyon",
          "yükselme", "kayıt", "cerrahi", "bacak", "yıl", "tanı", "şüphe", "segment", "proksimal", "distal", "stent", "girişim", "kalp", "damar",
          "diyabet", "hipertansiyon", "fibrilasyon", "arter", "muayene", "sendrom", "ventrikül", "aritmi", "böbrek", "ekokardiyografi"]
SOFT = {"k": "ğ", "p": "b", "ç": "c", "t": "d"}          # yetmezlik → yetmezliği
DROPPED = {"ritim": "ritm", "kayıt": "kayd"}             # ritim → ritmi, kayıt → kaydı
SUFFIX = re.compile(r"(?:l[ae]r)?(?:s?[ıiuü]|l[ae]r[ıi])?(?:n?d[ae]n?|n?t[ae]n?|y?[ae]|n[ae]|y?l[ae]|n?[ıiuü]n|y?[ıiuü]|n[ıiuü]|[ıiuü]?yl[ae]|[ıiuü]?nd[ae]|ki)?")
STEMS = {}
for lem in LEMMAS:
    STEMS[lem] = lem
    STEMS[DROPPED.get(lem, lem)] = lem
    if lem[-1] in SOFT:
        STEMS[lem[:-1] + SOFT[lem[-1]]] = lem
STEM_ORDER = sorted(STEMS, key=len, reverse=True)


def lemma(word):
    """Küçük harfli bir kelimenin lemması: Kelimenin başladığı en uzun sözlük gövdesi; kalan kısım Türkçe eklerden oluşan bir zincir olmalıdır."""
    for stem in STEM_ORDER:
        if word.startswith(stem) and SUFFIX.fullmatch(word[len(stem):]):
            return STEMS[stem]
    return word


def lemma_text(text):
    """Her kelimesi lemmasıyla değiştirilmiş bir metin."""
    return " ".join(lemma(w) for w in words(text))

for w in ["yetmezliğinin", "ritmi", "kaydında", "anjiyografide", "darlığa", "segmentinde", "hastane"]:
    print(f"{w:16} → {lemma(w)}")
word_forms = Counter(w for t in texts[tr] for w in words(t))
print("Eğitim epikrizlerindeki kelime biçimleri:", len(word_forms),
      "→ lemma:", len({lemma(w) for w in word_forms}))

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c164)*

**Python notu.** `sorted(STEMS, key=len, reverse=True)` en uzun gövdeleri başa koyar; böylece "hastalığ" "hasta"dan önce denenir ve "hastalığı" "hastalık" olur. `SUFFIX.fullmatch(rest)` yalnızca kelimenin geri kalanının tamamı ek kalıbıyla eşleştiğinde doğrudur: İsteğe bağlı bir çoğul eki (`l[ae]r`), isteğe bağlı bir iyelik eki ve isteğe bağlı bir hâl eki. Hiçbir gövdeyle eşleşmeyen kelime olduğu gibi kalır; bu yüzden "hastane" "hasta"ya kesilmez.

In [ ]:
lemma_texts = np.array([lemma_text(t) for t in texts])
lemma_vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
L_train = lemma_vectorizer.fit_transform(lemma_texts[tr])
L_val, L_test, L_ext = (lemma_vectorizer.transform(lemma_texts[m]) for m in (va, te, ex))
lemma_f1 = {"test": [], "external": []}
for k, label in enumerate(LABELS):
    lemma_model = LogisticRegression(C=10, max_iter=2000).fit(L_train, y[tr, k])
    p_val = lemma_model.predict_proba(L_val)[:, 1]
    t_best = GRID[int(np.argmax([se_ppv_f1(y[va, k], p_val >= g)[2] for g in GRID]))]
    for part, L_part, mask in (("test", L_test, te), ("external", L_ext, ex)):
        lemma_f1[part].append(se_ppv_f1(y[mask, k], lemma_model.predict_proba(L_part)[:, 1] >= t_best)[2])

macro_words = [round(float(np.mean([se_ppv_f1(y[m, k], tfidf_labels(X)[:, k])[2] for k in range(len(LABELS))])), 3) for X, m in ((X_test, te), (X_ext, ex))]
print("Terimler:", X_train.shape[1], "kelime →", L_train.shape[1], "lemma")
print("Makro F1, kelimeler:", dict(zip(["test", "dış"], macro_words)))
print("Makro F1, lemmalar:", {"test": round(float(np.mean(lemma_f1["test"])), 3), "dış": round(float(np.mean(lemma_f1["external"])), 3)})

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c166)*

**Gözlemle ve düşün.** Arşiv sentetiktir: Üreteci kelimelerin çoğunu yalnızca birkaç biçimde yazar; bu yüzden lemmatizasyon sözlüğü pek değiştirmez. Gerçek raporlar aynı kelimenin çok daha fazla biçimini kullanır. Kliniğinizdeki gerçek bir epikrizin hangi kelimelerini lemmatizer'ın sözlüğüne ilk olarak eklerdiniz?

#### Arşivden kelime vektörleri

*Ders sayfasında: [Kelime vektörleri: Kelimelerin anlamı çevrelerinden gelir](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#word-vectors-meaning-from-the-company-of-words)*

Kelimeler vektörlerini çevrelerindeki kelimelerden alır. Kod, iki kelimenin eğitim epikrizlerinde ne sıklıkla birlikte geçtiğini sayar, sayımları PMI ile rastlantıyla karşılaştırır, pozitif değerleri tutar ve her satırı 50 sayıya kısaltır. İki bağlam karşılaştırılır: Dört kelimelik bir pencere ve bütün epikriz.

**Formüllerden koda.** $\mathrm{PMI}(a, b) = \ln(P(a, b)/(P(a)P(b)))$: `np.log(co / total) - np.log(np.outer(p_word, p_word))`. PPMI: `np.maximum(pmi, 0)`. 50 ana yön: `np.linalg.svd(ppmi)` ve `U[:, :50] * S[:50]`. $\cos(\mathbf{u}, \mathbf{v})$: `vectors @ vectors[i]`; çünkü her vektörün uzunluğu 1'dir.

**Başlangıç hücresi.** Aşağıdaki hücre, «Arşivden kelime vektörleri» kısmı için kurulumu, sonraki hücrelerin kullandığı veri çağrılarını ve önceki hücrelerin kodlarını yineler; böylece kodlar hata vermeden çalışır. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Arşivden kelime vektörleri» kısmı
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# alıştırma promptlarının tabloları (etkileşimli labın F bölümü)
from vclinic.week06 import alistirma_tablosu

# önceki hücrelerden yinelenen kod: Aşağıdaki hücrelerin kullandığı veriler ve kodlar
import re
from collections import Counter
archive = reports.archive()
texts = archive["text_tr"].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
def tr_lower(text):
    """Türkçe kurallarına göre küçük harf: I, ı olur; İ, i olur ve metnin uzunluğu değişmez."""
    return text.replace("I", "ı").replace("İ", "i").lower()
def words(text):
    """Bir metnin küçük harfli kelimeleri: Harf ve rakam dizileri."""
    return re.findall(r"\w+", tr_lower(text))

**Ders sayfasında:** [Çözümlü örnek: Küçük sayılarla PMI ve kosinüs](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#worked-example-pmi-and-cosine-with-small-numbers). Aşağıdaki hücre arşivdeki kelimelerin PPMI değerlerini hesaplar, satırlarını tekil değer ayrışımıyla kısaltır ve vektörleri kosinüsleriyle karşılaştırır.

In [ ]:
def word_vectors(token_lists, window=None, min_count=5, dimensions=50):
    """Birlikte geçişten kelime vektörleri: PPMI ve SVD. window=4 ile iki kelime en fazla dört kelime arayla durduklarında, window=None ile aynı raporda geçtiklerinde birlikte geçmiş sayılır."""
    counts = Counter(w for ws in token_lists for w in ws)
    vocab = sorted(w for w, c in counts.items() if c >= min_count and not w.isdigit())
    index = {w: i for i, w in enumerate(vocab)}
    co = np.zeros((len(vocab), len(vocab)))
    for ws in token_lists:
        ids = [index.get(w) for w in ws]
        if window is None:
            present = sorted({i for i in ids if i is not None})
            co[np.ix_(present, present)] += 1
            continue
        for i, a in enumerate(ids):
            if a is None:
                continue
            for j in range(max(0, i - window), min(len(ids), i + window + 1)):
                if j != i and ids[j] is not None:
                    co[a, ids[j]] += 1
    np.fill_diagonal(co, 0)
    total = co.sum()
    p_word = co.sum(axis=1) / total
    with np.errstate(divide="ignore"):
        pmi = np.log(co / total) - np.log(np.outer(p_word, p_word))
    ppmi = np.where(co > 0, np.maximum(pmi, 0), 0.0)
    U, S, _ = np.linalg.svd(ppmi)
    vectors = U[:, :dimensions] * S[:dimensions]
    return vocab, vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


def nearest(word, vocab, vectors, k=5):
    """Vektörü, verilen kelimenin vektörüyle en büyük kosinüsü yapan k kelime."""
    cos = vectors @ vectors[vocab.index(word)]
    order = [i for i in np.argsort(-cos) if vocab[i] != word][:k]
    return [(vocab[i], round(float(cos[i]), 3)) for i in order]

train_tokens = [words(t) for t in texts[tr]]
vocab_w, vectors_w = word_vectors(train_tokens, window=4)
vocab_r, vectors_r = word_vectors(train_tokens, window=None)
print(len(vocab_w), "vektörü olan kelime")
for word in ["apiksaban", "hipertansiyon", "furosemid"]:
    print(f"{word:14} pencere:", nearest(word, vocab_w, vectors_w, 4))
    print(f"{'':14} epikriz:", nearest(word, vocab_r, vectors_r, 4))

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c171)*

**Python notu.** `Counter` bütün eğitim epikrizlerinin kelimelerini sayar; sözlük en az beş kez geçen ve sayı olmayan kelimeleri tutar. `co`, her kelime için bir satır ve bir sütun içeren kare bir tablodur; iç içe iki döngü, aralarında en fazla `window` kelime bulunan her kelime çifti için 1 ekler. `window=None` olduğunda `np.ix_(present, present)` aynı epikrizdeki her kelime çiftine bir seferde 1 ekler. `np.errstate(divide="ignore")` 0'ın logaritması için verilen uyarıyı susturur; bu değerler daha sonra 0 yapılır.

In [ ]:
show = ["apiksaban", "rivaroksaban", "dabigatran", "düzensiz", "metformin", "gliklazid", "furosemid", "spironolakton", "dekompanse",
        "amlodipin", "ramipril", "hipertansiyon", "hipertansyon", "ht", "egfr", "kreatinin"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, (vocab_x, vectors_x, title) in zip(axes, [(vocab_w, vectors_w, "dört kelimelik pencere"),
                                                  (vocab_r, vectors_r, "bütün epikriz")]):
    M = vectors_x[[vocab_x.index(w) for w in show]]
    M = M - M.mean(axis=0)
    xy = M @ np.linalg.svd(M, full_matrices=False)[2][:2].T
    ax.scatter(xy[:, 0], xy[:, 1], s=12, color="#9B1B1B")
    for (a, b), w in zip(xy, show):
        ax.annotate(w, (a, b), fontsize=8)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

**Gözlemle ve düşün.** Pencereyle antikoagülanlar bir arada ve "düzensiz" kelimesinden uzakta durur; bütün epikriz bağlam olarak alındığında "düzensiz" onlara katılır. Yanlış yazılmış bir ilacı tanıması gereken bir modele bu iki uzaydan hangisi, ilaçlardan bir tanıyı tahmin etmesi gereken bir modele hangisi yardım ederdi?

**Alıştırma E.** Bağlam olarak bütün epikriz alındığında "apiksaban" ve "düzensiz" kelimelerinin vektörleri arasındaki kosinüs kaçtır? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("E")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma E", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(vectors_r[vocab_r.index("apiksaban")] @ vectors_r[vocab_r.index("düzensiz")]), 3)
teach.check("Alıştırma E", cevap)

**1. durak tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#last-two-hours-advanced-clinical-nlp-applications) ve okumaya «Son 2 saat: Gelişmiş klinik NLP uygulamaları» bölümünden devam edin. Bir sonraki uygulama durağında ders sayfası sizi yine bu Jupyter Notebook'a getirir.

## Durak 2: Mini proje: NER pipeline ile STEMI, AF ve kalp yetersizliği

*Ders sayfasında: [2. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#stop-2)*

Bu durak oturumun mini projesidir. Burada epikrizleri Hugging Face Hub'dan alınan, önceden eğitilmiş Türkçe bir dil modeli okur. Modelin tokenizer'ı önce B hastanesinin büyük harflerinin bir modele ne yaptığını gösterir. Model ardından tanıları, ilaçları ve işlemleri bulmak için ince ayarla eğitilir; bulduğu varlıklar, 1. duraktaki durum kuralı ve ilaçlarla birlikte B hastanesinin epikrizlerindeki altı tanıyı verir. Son adım, epikrizlerdeki akut koroner sendromdan ve ST yükselmesinden okunan STEMI'yi ekler.

**Durağın sonunda:** B hastanesinin epikrizleri yazıldığı gibi verildiğinde bilinmeyen token'ların oranını (yüzde 19,5), ince ayarlı modelin test kümesindeki varlık F1 değerini (yaklaşık 0,981) modelin varlıklarından B hastanesinde altı tanının makro F1 değerini (yaklaşık 0,754) ve A hastanesinin test kümesinde STEMI'nin F1 değerini (1,0) bulacaksınız.

Durak üç bölümden oluşur: **Python temelleri**, **Prompt ile uygulama** ve **Hazır uygulama**. Bölümleri bu sırayla izleyin. Python biliyorsanız promptlarla, yalnızca sonucu görmek istiyorsanız hazır uygulamayla başlayabilirsiniz. Her bölüm kendi başlangıç hücresiyle açılır.

### Durak 2 · Python temelleri

*Ders sayfasında: [Named Entity Recognition (NER): İlaç, tanı, prosedür](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#named-entities-diagnoses-drugs-and-procedures)*

Bu durağın hazır uygulaması arşivin token'ları üzerinde üç küçük harf biçimini dener, bir epikrizin her token'ına ait olduğu varlığın etiketini verir, varlıkları tam eşleşmeye göre değerlendirir ve yanlış yazılmış bir geçişi sözlüğün en benzer kelimesine bağlar. Aşağıdaki örnekler bu yapıları kısa metinler üzerinde tanıtır: Fonksiyonlardan oluşan bir sözlük, bir metindeki kelimelerin konumları, varlık kümeleri ve iki metnin benzerliği.

Hücreleri sırayla çalıştırın. Bir örneğin altındaki *Python notu* yeni yapıyı açıklar. Alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerdedir.

**Başlangıç hücresi.** Aşağıdaki hücre, «Durak 2 · Python temelleri» bölümü için kurulumu, sonraki hücrelerin kullandığı veri çağrılarını ve önceki hücrelerin kodlarını yineler; böylece kodlar hata vermeden çalışır. «Durak 1 · Hazır uygulama» bölümünde yazılan fonksiyonlar, onları aynen saklayan ders paketinden alınır; bu fonksiyonların kodları o bölümdedir. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Durak 2 · Python temelleri» bölümü
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# önceki bölümlerin fonksiyonları: Ders paketi onları aynen saklar
from vclinic.week06 import tr_lower

# önceki hücrelerden yinelenen kod: Aşağıdaki hücrelerin kullandığı veriler ve kodlar
import difflib
import re

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
forms = {"yazıldığı gibi": lambda t: t, "Python'un lower() metodu": str.lower, "Türkçe küçük harf": tr_lower}
word = "KALP YETMEZLİĞİ"
for name, f in forms.items():
    print(f"{name:26} {f(word)}  ({len(f(word))} karakter)")

**Python notu.** Bir sözlük değer olarak fonksiyon da tutabilir. `lambda t: t`, girdisini değiştirmeden döndüren adsız küçük bir fonksiyon yazar; `str.lower` ise Python'un kendi metodudur ve burada bir fonksiyon gibi kullanılır. Döngü her fonksiyonu aynı kelimeye uygular: Python'un `lower()` metodu İ harfini iki karaktere çevirdiği için kelime bir karakter uzar, dersin Türkçe küçük harf fonksiyonu ise uzunluğu korur. Hazır uygulama üç biçimi aynı şekilde bütün arşivin token'ları üzerinde karşılaştırır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
text = "Kalp yetmezliği için furosemid başlandı."
entities = [{"start": 0, "end": 15, "label": "DIAG"}, {"start": 21, "end": 30, "label": "DRUG"}]
offsets = [m.span() for m in re.finditer(r"\w+", text)]   # her kelimenin (başlangıç, bitiş) konumu
tags = []
for start, end in offsets:
    tag = "O"
    for e in entities:
        if e["start"] <= start < e["end"]:
            tag = ("B-" if start == e["start"] else "I-") + e["label"]
    tags.append(tag)
for (start, end), tag in zip(offsets, tags):
    print(f"{text[start:end]:12} {start:2}-{end:2}  {tag}")

**Python notu.** `m.span()` bir eşleşmenin ilk karakterinin konumunu ve son karakterinden sonraki karakterin konumunu verir; böylece `text[start:end]` kelimeyi metinden yeniden keser. Bir varlık da aynı biçimde, başlangıcı, bitişi ve türüyle saklanır. Döngü her kelimeye, ilk karakterinin içinde bulunduğu varlığın etiketini verir: Varlığın ilk kelimesi için B-, onu izleyen kelimeler için I-, hiçbir varlığa girmeyen kelimeler için O. Bunlar BIO düzeninin etiketleridir; hazır uygulamadaki `encode` fonksiyonu da modelin token'larını aynı biçimde etiketler.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
gold = {(0, 15, "DIAG"), (21, 30, "DRUG"), (45, 52, "PROC")}                    # açıklanmış varlıklar
found = {(0, 15, "DIAG"), (21, 30, "DRUG"), (45, 49, "PROC"), (60, 64, "DIAG")}   # bir modelin bulduğu varlıklar
tp, fp, fn = len(gold & found), len(found - gold), len(gold - found)
print("DP", tp, "YP", fp, "YN", fn)
print("kesinlik", round(tp / (tp + fp), 3), "duyarlılık", round(tp / (tp + fn), 3), "F1", round(2 * tp / (2 * tp + fp + fn), 3))

**Python notu.** Bir *küme* her değeri bir kez, süslü parantezler arasında tutar; buradaki değerler, her biri başlangıç, bitiş ve türden oluşan bir demet olan varlıklardır. `gold & found` iki kümede de bulunan varlıkları, yani doğru pozitifleri tutar; `found - gold` yalnızca modelin bulduğu varlıkları, `gold - found` da modelin kaçırdığı varlıkları verir. Bir varlık ancak başlangıcı, bitişi ve türü birlikte uyuştuğunda sayılır: Modelin 52 yerine 49. karakterde bitirdiği işlem hem yanlış pozitif hem de yanlış negatiftir. Hazır uygulamadaki `span_scores` fonksiyonu varlıkları aynı biçimde sayar.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
dictionary = ["hipertansiyon", "diyabet", "kalp yetmezliği"]
for mention in ["hipertansyon", "diyebet", "nefes darlığı"]:
    score, best = max((difflib.SequenceMatcher(None, mention, w).ratio(), w) for w in dictionary)
    print(f"{mention:14} → {best:16} {score:.2f}", "bağlandı" if score >= 0.8 else "bağlanmadı")

**Python notu.** `difflib.SequenceMatcher(None, a, b).ratio()` iki metni harf harf karşılaştırır: Aynı metin için 1 verir; harfler ne kadar çok farklıysa değer o kadar küçülür. `max(...)` içindeki üreteç sözlükteki her kelime için bir (skor, kelime) çifti oluşturur. `max` çiftleri ilk değerlerine göre karşılaştırdığı için en benzer kelimeyi skoruyla birlikte döndürür. Yanlış yazılmış bir geçiş ancak skor 0,8'e ulaştığında bir tanıya bağlanır; bu yüzden "nefes darlığı" hiçbir tanıya bağlanmaz. Hazır uygulamadaki `normalise` fonksiyonu modelin bulduğu varlıkları altı tanıya aynı biçimde bağlar.

**Alıştırma 2.1.** "Akut koroner sendrom nedeniyle koroner anjiyografi yapıldı." cümlesinde "Akut koroner sendrom" tanısı 0 ile 20, "koroner anjiyografi" işlemi de 31 ile 50 arasındaki karakterleri kaplar. Cümlenin kaç kelimesi I- etiketi alır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 2.1", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
text = "Akut koroner sendrom nedeniyle koroner anjiyografi yapıldı."
entities = [{"start": 0, "end": 20, "label": "DIAG"}, {"start": 31, "end": 50, "label": "PROC"}]
cevap = 0
for start, end in [m.span() for m in re.finditer(r"\w+", text)]:
    for e in entities:
        if e["start"] < start < e["end"]:
            cevap += 1
teach.check("Alıştırma 2.1", cevap)

**Alıştırma 2.2.** Bir epikrizin açıklanmış varlıkları (0, 15, "DIAG"), (21, 30, "DRUG"), (40, 52, "DIAG") ve (60, 70, "PROC") demetleridir. Bir model (0, 15, "DIAG"), (21, 30, "DRUG"), (40, 48, "DIAG") ve (75, 84, "DRUG") varlıklarını buluyor. Tam eşleşmeye göre varlıkların F1 değeri nedir? Üç ondalık basamağa yuvarlayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 2.2", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
gold = {(0, 15, "DIAG"), (21, 30, "DRUG"), (40, 52, "DIAG"), (60, 70, "PROC")}
found = {(0, 15, "DIAG"), (21, 30, "DRUG"), (40, 48, "DIAG"), (75, 84, "DRUG")}
tp, fp, fn = len(gold & found), len(found - gold), len(gold - found)
cevap = round(2 * tp / (2 * tp + fp + fn), 3)
teach.check("Alıştırma 2.2", cevap)

**«Durak 2 · Python temelleri» tamamlandı.** Sırada hemen aşağıdaki «Durak 2 · Prompt ile uygulama» bölümü var. Durağı burada bırakacaksanız [ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#summaries-and-large-language-models-as-clinical-assistants) ve okumaya «Radyoloji raporu otomatik özetleme» bölümünden devam edin.

### Durak 2 · Prompt ile uygulama

Bu bölümde dört prompt vardır: «Önceden eğitilmiş Türkçe bir modelin token'ları», «Varlıklar için modele ince ayar», «Varlıklardan B hastanesindeki altı tanıya» ve «Seyirden ve EKG'den STEMI». Onarım promptu ve hata mesajı için prompt, [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-06/PROMPTLAR.md) ve 1. durağın promptlarında yer alır.

Her promptta dört adım vardır. Promptu kopyalayıp YZ asistanınıza gönderin. Altındaki veri hücresini çalıştırın: Bu hücre, promptun tarif ettiği "veri" tablosunu hazırlar ve gösterir. Asistanın kodunu boş hücreye yapıştırıp çalıştırın. Ardından kontrol hücresini çalıştırın; bu hücre *doğru* ya da *henüz değil* yazar. Asistanın kodu kontrolden geçmezse *Hazır kod* hücresini çalıştırıp devam edin.

**Başlangıç hücresi.** Aşağıdaki hücre, «Durak 2 · Prompt ile uygulama» bölümü için kurulumu ve bu bölümün veri çağrılarını yineler; böylece kodlar hata vermeden çalışır. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Durak 2 · Prompt ile uygulama» bölümü
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

import torch
from vclinic import reports
import re

if "archive" not in globals():
    archive = reports.archive()
    LABELS = list(reports.LABELS)
    texts = archive["text_tr"].to_numpy()
    y = archive[LABELS].to_numpy()
    split = archive["split"].to_numpy()
    tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")


    def tr_lower(text):
        """Türkçe kurallarına göre küçük harf: I, ı olur; İ, i olur ve metnin uzunluğu değişmez."""
        return text.replace("I", "ı").replace("İ", "i").lower()


    def words(text):
        """Bir metnin küçük harfli kelimeleri: Harf ve rakam dizileri."""
        return re.findall(r"\w+", tr_lower(text))
_COLS = ["KY", "AF", "KAH", "HT", "DM", "KBH"]
_SETS = {"training": "eğitim", "validation": "doğrulama", "test": "test", "external": "dış"}


def _summaries(mask, with_set=False, with_entities=False):
    """Bir kümenin epikrizleri ve kayıt sistemindeki altı tanı; promptların tarif ettiği biçimde."""
    t = pd.DataFrame({"epikriz": texts[mask]})
    if with_set:
        t.insert(0, "küme", [_SETS[s] for s in split[mask]])
    if with_entities:
        t["varlıklar"] = [[(e["start"], e["end"], e["label"]) for e in es] for es in archive["entities_tr"][mask]]
    for k, c in enumerate(_COLS):
        t[c] = y[mask, k]
    return t.reset_index(drop=True)


def _course_ner():
    """Dersin modelinin dış epikrizlerde bulduğu varlıklar (bir kez eğitilir, birkaç dakika)."""
    if "_ner_cache" in globals():
        return _ner_cache
    from transformers import AutoTokenizer, AutoModelForTokenClassification
    torch.manual_seed(2026)
    np.random.seed(2026)
    MODEL = "ytu-ce-cosmos/turkish-tiny-bert-uncased"
    tokenizer = AutoTokenizer.from_pretrained(MODEL)

    TAGS = ["O", "B-DIAG", "I-DIAG", "B-DRUG", "I-DRUG", "B-PROC", "I-PROC"]
    MAX_LENGTH = 384


    def encode(text, entities):
        """Bir raporun token'ları ve her birinin etiketi: Bir varlığın ilk token'ı B-, ötekiler I-, dışındakiler O; özel token'lar -100."""
        enc = tokenizer(tr_lower(text), return_offsets_mapping=True, truncation=True, max_length=MAX_LENGTH)
        tags = []
        for start, end in enc["offset_mapping"]:
            if start == end:
                tags.append(-100)
                continue
            tag = "O"
            for e in entities:
                if e["start"] <= start < e["end"]:
                    tag = ("B-" if start == e["start"] else "I-") + e["label"]
                    break
            tags.append(TAGS.index(tag))
        return enc["input_ids"], tags, enc["offset_mapping"]


    def spans(tag_ids, offsets):
        """Token'ların etiketlerinden varlıklar: Raporun karakterleri cinsinden (başlangıç, bitiş, tür)."""
        found, current = [], None
        for t, (start, end) in zip(tag_ids, offsets):
            if start == end:
                continue
            tag = TAGS[t]
            if tag.startswith("B-") or (tag.startswith("I-") and (current is None or current[2] != tag[2:])):
                if current:
                    found.append(tuple(current))
                current = [start, end, tag[2:]]
            elif tag.startswith("I-"):
                current[1] = end
            else:
                if current:
                    found.append(tuple(current))
                current = None
        if current:
            found.append(tuple(current))
        return found


    def batches(items, size, shuffle=False):
        """Eğitim ve tahmin için aynı uzunluğa tamamlanmış rapor grupları."""
        order = np.random.permutation(len(items)) if shuffle else np.arange(len(items))
        for s in range(0, len(items), size):
            group = [items[i] for i in order[s:s + size]]
            width = max(len(g[0]) for g in group)
            ids = torch.tensor([g[0] + [0] * (width - len(g[0])) for g in group])
            tags = torch.tensor([g[1] + [-100] * (width - len(g[1])) for g in group])
            mask = torch.tensor([[1] * len(g[0]) + [0] * (width - len(g[0])) for g in group])
            yield ids, tags, mask

    EPOCHS, BATCH, LEARNING_RATE, SEED = 4, 16, 3e-4, 2026
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    train_items = [encode(t, e)[:2] for t, e in zip(texts[tr], archive["entities_tr"][tr])]
    ner_model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(TAGS))
    optimizer = torch.optim.AdamW(ner_model.parameters(), lr=LEARNING_RATE)
    for epoch in range(EPOCHS):
        ner_model.train()
        total = 0.0
        for ids, tags, mask in batches(train_items, BATCH, shuffle=True):
            loss = ner_model(input_ids=ids, attention_mask=mask, labels=tags).loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(ids)
        print(f"epoch {epoch + 1}: ortalama kayıp {total / len(train_items):.4f}")


    def predict_spans(text_list):
        """Modelin bir listedeki her raporda bulduğu varlıklar."""
        ner_model.eval()
        out = []
        with torch.no_grad():
            for text in text_list:
                ids, _, offsets = encode(text, [])
                logits = ner_model(input_ids=torch.tensor([ids])).logits[0]
                out.append(spans(logits.argmax(-1).tolist(), offsets))
        return out
    globals()["_ner_cache"] = predict_spans(list(texts[ex]))
    return _ner_cache

print(len(archive), "epikriz; B hastanesi:", int(ex.sum()))

#### Prompt 2.1: Önceden eğitilmiş Türkçe bir modelin token'ları

**Amaç.** Bu promptla önceden eğitilmiş Türkçe bir modelin B hastanesinin epikrizlerini nasıl token'lara böldüğü ve harflerin büyük ya da küçük olmasının neyi değiştirdiği görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Kısmen büyük harflerle yazılmış epikrizlerde önceden eğitilmiş Türkçe bir dil modeli kullanmak ve modelin bu epikrizlerin kelimelerini bilip bilmediğini öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; sentetik bir kayıt sistemi olan sanal kliniğin B hastanesine ait, büyük harfli kısaltmalar içeren kısa bir üslupla Türkçe yazılmış 600 epikrizi tutar. Her satır bir hastadır ve "epikriz" sütunu epikrizi tutar.  
> Görev: "ytu-ce-cosmos/turkish-tiny-bert-uncased" modelinin tokenizer'ını Hugging Face Hub'dan yükle. Her epikrizi üç kez token'lara böl: Yazıldığı gibi, programlama dilinin olağan küçük harf işleminden sonra ve Türkçe küçük harf işleminden sonra (önce I, ı; İ, i olur, ardından öteki harfler küçülür). Üçünün her biri için baştaki ve sondaki özel token'lar hariç token'ları say ve bunların yüzde kaçının bilinmeyen token olduğunu hesapla. İlk epikrizin token'larını da üç biçimde göster.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Üç biçim için token sayısını ve bilinmeyen token'ların yüzdesini yazdır; yazıldığı gibi verilen metindeki yüzdeyi bir ondalık basamakla "cevap" adıyla sakla.  
> Test: Türkçe küçük harf işleminden sonra hiç bilinmeyen token kalmamalı; ilk epikriz olağan küçük harf işleminden sonra Türkçe işlemden sonrakinden daha çok ya da aynı sayıda token vermeli.  

In [ ]:
# Prompt 2.1 için veri: Promptun tarif ettiği "veri" tablosu
veri = pd.DataFrame({"epikriz": texts[ex]})
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 2.1 kontrolü
teach.check_step("Prompt 2.1", "cevap", globals())

In [ ]:
#@title Prompt 2.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = pd.DataFrame({"epikriz": texts[ex]})
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ytu-ce-cosmos/turkish-tiny-bert-uncased")
bicimler = {"yazıldığı gibi": lambda t: t, "olağan küçük harf": lambda t: t.lower(),
          "Türkçe küçük harf": lambda t: t.replace("I", "ı").replace("İ", "i").lower()}
oranlar = {}
for ad, f in bicimler.items():
    kimlikler = [tokenizer(f(t))["input_ids"][1:-1] for t in veri["epikriz"]]
    n = sum(len(i) for i in kimlikler)
    bilinmeyen = sum(i.count(tokenizer.unk_token_id) for i in kimlikler)
    oranlar[ad] = 100 * bilinmeyen / n
    print(f"{ad}: {n} token, {bilinmeyen} [UNK] ({oranlar[ad]:.1f} %)")
    print("   ", tokenizer.tokenize(f(veri["epikriz"].iloc[0]))[:25])
cevap = round(oranlar["yazıldığı gibi"], 1)
teach.check_step("Prompt 2.1", "cevap", globals())

**Sonuç ne gösteriyor?** Yazıldığı gibi verildiğinde B hastanesinin token'larının yüzde 19,5 kadarı bilinmeyendir; çünkü büyük ve küçük harf ayrımı yapmayan modelin sözlüğünde büyük harf yoktur. Olağan küçük harf işlemi bilinmeyen token'ları ortadan kaldırır ama i'den sonra bir nokta eklediği için İ içeren her kelimeyi yanlış böler; sözlükteki kelimeleri yalnızca Türkçe küçük harf işlemi verir.

#### Prompt 2.2: Varlıklar için modele ince ayar

**Amaç.** Bu promptla önceden eğitilmiş modele tanıları, ilaçları ve işlemleri etiketlemesi öğretilir. Eğitim birkaç dakika sürer.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Küçük ve önceden eğitilmiş Türkçe bir dil modelini epikrizlerdeki tanıları, ilaçları ve işlemleri bulacak biçimde uyarlamak ve modelin bunları görmediği epikrizlerde ne kadar doğru bulduğunu öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; sentetik bir kayıt sistemi olan sanal klinikten 3.000 epikriz tutar. Her satır bir hastadır: "küme" sütunu kümeyi verir ("eğitim": A hastanesinden 1.800 epikriz; "test": A hastanesinden 600; "dış": B hastanesinden 600). "epikriz" sütunu Türkçe epikrizi tutar; "varlıklar" sütunu epikrizdeki açıklanmış varlıkları, 0'dan başlayarak sayılan (ilk karakter, son karakter artı bir, tür) üçlüleri hâlinde listeler. Türler tanı için "DIAG", ilaç için "DRUG", işlem için "PROC" olarak yazılmıştır; "KY", "AF", "KAH", "HT", "DM" ve "KBH" adlı altı sütun 0 ya da 1 değerini alır; kayıt sisteminde sırasıyla kalp yetersizliği, atriyal fibrilasyon, koroner arter hastalığı, hipertansiyon, diyabetes mellitus ve kronik böbrek hastalığı kayıtlıysa değer 1, değilse 0 olur.  
> Görev: "ytu-ce-cosmos/turkish-tiny-bert-uncased" modelini Hugging Face Hub'dan, her token'a yedi etiketten birini veren bir katmanla kullan: Varlığın dışında ya da bir tanının, bir ilacın veya bir işlemin başlangıcı ya da içi. Her epikrizi Türkçe küçük harfe çevir (önce I, ı; İ, i olur); bu işlem karakterlerin konumlarını korur. Epikrizi en fazla 384 token'a böl. Bir token, ilk karakterinin içinde bulunduğu varlığın etiketini alır: Varlığı başlatıyorsa başlangıç etiketini, başlatmıyorsa iç etiketini. Modele eğitim epikrizleriyle 4 epoch boyunca, karıştırılmış 16 epikrizlik gruplar hâlinde, AdamW optimizasyon algoritması ve 0,0003 öğrenme hızıyla ince ayar yap. Ardından test ve dış epikrizleri etiketle, her varlığın token'larını yeniden bir karakter parçasına birleştir ve varlıkların kesinliğini, duyarlılığını ve F1 değerini hesapla; tahmin edilen bir varlık ancak türü, ilk karakteri ve son karakteri açıklanmış bir varlıkla eşleştiğinde sayılır. Sonucun tekrarlanabilmesi için seed değerini 2026 olarak sabitle.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her epoch'un kaybını ve test ile dış epikrizlerde bütün varlıklar ve her tür için kesinliği, duyarlılığı ve F1 değerini gösteren bir tablo yazdır; test epikrizlerinde bütün varlıkların F1 değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Kayıp ilk epoch'tan son epoch'a düşmeli ve F1 değeri 0 ile 1 arasında olmalı.  

In [ ]:
# Prompt 2.2 için veri: Promptun tarif ettiği "veri" tablosu
veri = _summaries(tr | te | ex, with_set=True, with_entities=True)
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 2.2 kontrolü
teach.check_step("Prompt 2.2", "cevap", globals())

In [ ]:
#@title Prompt 2.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _summaries(tr | te | ex, with_set=True, with_entities=True)
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

torch.manual_seed(2026)
np.random.seed(2026)
etiketler = ["O", "B-DIAG", "I-DIAG", "B-DRUG", "I-DRUG", "B-PROC", "I-PROC"]
tokenizer = AutoTokenizer.from_pretrained("ytu-ce-cosmos/turkish-tiny-bert-uncased")


def kodla(metin, varliklar):
    kod = tokenizer(metin.replace("I", "ı").replace("İ", "i").lower(), return_offsets_mapping=True, truncation=True, max_length=384)
    etiket_no = []
    for a, b in kod["offset_mapping"]:
        if a == b:
            etiket_no.append(-100)
            continue
        etiket = next((("B-" if a == s else "I-") + t for s, e, t in varliklar if s <= a < e), "O")
        etiket_no.append(etiketler.index(etiket))
    return kod["input_ids"], etiket_no, kod["offset_mapping"]


def parcalara(etiket_nolari, konumlar):
    bulunan, guncel = [], None
    for t, (a, b) in zip(etiket_nolari, konumlar):
        if a == b:
            continue
        etiket = etiketler[t]
        if etiket.startswith("B-") or (etiket.startswith("I-") and (guncel is None or guncel[2] != etiket[2:])):
            if guncel:
                bulunan.append(tuple(guncel))
            guncel = [a, b, etiket[2:]]
        elif etiket.startswith("I-"):
            guncel[1] = b
        else:
            if guncel:
                bulunan.append(tuple(guncel))
            guncel = None
    if guncel:
        bulunan.append(tuple(guncel))
    return bulunan


egitim = veri[veri["küme"] == "eğitim"]
ogeler = [kodla(t, e)[:2] for t, e in zip(egitim["epikriz"], egitim["varlıklar"])]
model = AutoModelForTokenClassification.from_pretrained("ytu-ce-cosmos/turkish-tiny-bert-uncased", num_labels=len(etiketler))
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
for epoch in range(4):
    model.train()
    sira = np.random.permutation(len(ogeler))
    toplam = 0.0
    for bas in range(0, len(sira), 16):
        grup = [ogeler[i] for i in sira[bas:bas + 16]]
        uzunluk = max(len(g[0]) for g in grup)
        kimlikler = torch.tensor([g[0] + [0] * (uzunluk - len(g[0])) for g in grup])
        etk = torch.tensor([g[1] + [-100] * (uzunluk - len(g[1])) for g in grup])
        maske = torch.tensor([[1] * len(g[0]) + [0] * (uzunluk - len(g[0])) for g in grup])
        kayip = model(input_ids=kimlikler, attention_mask=maske, labels=etk).loss
        optimizer.zero_grad()
        kayip.backward()
        optimizer.step()
        toplam += kayip.item() * len(grup)
    print("epoch", epoch + 1, "kayıp", round(toplam / len(ogeler), 4))
model.eval()
satirlar = []
for k in ("test", "dış"):
    parca = veri[veri["küme"] == k]
    tahmin = []
    with torch.no_grad():
        for t in parca["epikriz"]:
            kimlikler, _, konumlar = kodla(t, [])
            tahmin.append(parcalara(model(input_ids=torch.tensor([kimlikler])).logits[0].argmax(-1).tolist(), konumlar))
    for tur in (None, "DIAG", "DRUG", "PROC"):
        tp = fp = fn = 0
        for g, p in zip(parca["varlıklar"], tahmin):
            g = {tuple(e) for e in g if tur in (None, e[2])}
            p = {e for e in p if tur in (None, e[2])}
            tp, fp, fn = tp + len(g & p), fp + len(p - g), fn + len(g - p)
        satirlar.append({"küme": k, "tür": tur or "hepsi", "kesinlik": round(tp / max(1, tp + fp), 3),
                         "duyarlılık": round(tp / max(1, tp + fn), 3), "F1": round(2 * tp / max(1, 2 * tp + fp + fn), 3)})
tablo = pd.DataFrame(satirlar)
print(tablo)
cevap = float(tablo.query("tür == 'hepsi'")["F1"].iloc[0])
teach.check_step("Prompt 2.2", "cevap", globals())

**Sonuç ne gösteriyor?** İnce ayarlı model test kümesindeki varlıkları yaklaşık 0,981 F1 değeriyle bulur. Buradaki hatalar varlıkların sınırlarındadır; "koroner anjiyografida" örneğinde olduğu gibi Türkçe bir ek bir işlemin adına yapışır. Model B hastanesinin epikrizlerinde ilaçların çoğunu bulur, ancak tanıların az bir kısmını bulabilir ve hiçbir işlemi bulamaz: B hastanesi bunları kısa üslubunda kısaltmalarla yazar.

#### Prompt 2.3: Varlıklardan B hastanesindeki altı tanıya

**Amaç.** Bu promptla modelin varlıkları, 1. duraktaki durum kuralı ve ilaçlarla B hastanesinin her epikrizi için altı tanıya dönüştürülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Bir dil modelinin başka bir hastanenin epikrizlerinde bulduğu tanıları ve ilaçları her hastanın altı kronik tanısına dönüştürmek ve bunun ne kadar iyi çalıştığını öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; sentetik bir kayıt sistemi olan sanal kliniğin B hastanesine ait, kısa bir üslupla Türkçe yazılmış 600 epikrizi tutar. Her satır bir hastadır: "epikriz" sütunu epikrizi tutar; "varlıklar" sütunu epikrizdeki açıklanmış varlıkları, "model varlıkları" sütunu ise A hastanesinin eğitim epikrizleriyle ince ayar yapılmış küçük Türkçe dil modelinin epikrizde bulduğu varlıkları listeler. İki sütunda da varlıklar, 0'dan başlayarak sayılan (ilk karakter, son karakter artı bir, tür) üçlüleri hâlinde yazılmıştır; türler tanı için "DIAG", ilaç için "DRUG" ve işlem için "PROC" olarak gösterilir. "KY", "AF", "KAH", "HT", "DM" ve "KBH" adlı altı sütun 0 ya da 1 değerini alır; kayıt sisteminde sırasıyla kalp yetersizliği, atriyal fibrilasyon, koroner arter hastalığı, hipertansiyon, diyabetes mellitus ve kronik böbrek hastalığı kayıtlıysa değer 1, değilse 0 olur.  
> Görev: "model varlıkları" sütununu kullan. Her tanı varlığını, 1. duraktaki gibi kelimelerine göre altı tanıdan birine ver ve durumunu 1. duraktaki kuralla belirle; varlıklarından biri var durumunda olan tanı var kabul edilir. 1. duraktaki gibi ilaç varlıklarının ve eGFR ile HbA1c değerlerinin gösterdiği tanıları ekle. Altı kararı kayıt sistemindeki altı sütunla karşılaştır; her tanı için duyarlılığı, pozitif öngörü değerini ve F1 değerini (doğru pozitiflerin iki katı, doğru pozitiflerin iki katı ile yanlış pozitiflerin ve yanlış negatiflerin toplamına bölünür) ve altı F1 değerinin ortalamasını, yani makro F1 değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her tanı için bir satır ve üç değer içeren bir tablo yazdır; makro F1 değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her değer 0 ile 1 arasında olmalı; hiç varlığı olmayan bir epikriz, laboratuvar değerlerinden gelebilen kronik böbrek hastalığı ve diyabet dışında her tanı için negatif olmalı.  

In [ ]:
# Prompt 2.3 için veri: Promptun tarif ettiği "veri" tablosu
veri = _summaries(ex, with_entities=True)
veri.insert(2, "model varlıkları", _course_ner())   # ilk seferde birkaç dakika sürer: Dersin modeline bir kez ince ayar yapılır
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 2.3 kontrolü
teach.check_step("Prompt 2.3", "cevap", globals())

In [ ]:
#@title Prompt 2.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _summaries(ex, with_entities=True)
veri.insert(2, "model varlıkları", _course_ner())   # ilk seferde birkaç dakika sürer: Dersin modeline bir kez ince ayar yapılır
terimler = {"KY": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
           "AF": ["atriyal fibrilasyon", "af", "afib"],
           "KAH": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
           "HT": ["hipertansiyon", "ht"], "DM": ["diyabet", "diyabetes mellitus", "dm"],
           "KBH": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"]}


def turkce_kucuk(metin):
    return metin.replace("I", "ı").replace("İ", "i").lower()


def skor(kararlar):
    satirlar = []
    for s in terimler:
        gercek = veri[s].to_numpy() == 1
        tahmin = np.array(kararlar[s]) == 1
        dp, yp, yn = int((tahmin & gercek).sum()), int((tahmin & ~gercek).sum()), int((~tahmin & gercek).sum())
        satirlar.append({"tanı": s, "duyarlılık": round(dp / max(1, dp + yn), 3),
                         "PPV": round(dp / max(1, dp + yp), 3), "F1": round(2 * dp / max(1, 2 * dp + yp + yn), 3)})
    tablo = pd.DataFrame(satirlar)
    print(tablo)
    print("Makro F1:", round(float(tablo["F1"].mean()), 3))
    return tablo

import difflib

dis = veri
tahmin = veri["model varlıkları"]
olumsuz = ["yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)", "şüphe", "olası", "ön tanı"]
yakin = ["babası", "annesi", "kardeşi", "babada", "annede", "soygeçmiş"]
ilaclar = {"KY": ["furosemid", "sakubitril"], "AF": ["apiksaban", "rivaroksaban", "edoksaban", "dabigatran"],
           "KAH": ["klopidogrel", "tikagrelor", "pkg", "kabg", "stent"], "HT": ["amlodipin", "hidroklorotiyazid"],
           "DM": ["metformin", "gliklazid", "insülin", "sitagliptin"]}
tum_kelimeler = [(k, s) for s, kelimeler in terimler.items() for k in kelimeler]


def tanisi(gecis):
    for k, s in sorted(tum_kelimeler, key=lambda x: -len(x[0])):
        if re.search(r"(?<!\w)" + re.escape(k) + r"(?!\w)", gecis):
            return s
    en_iyi = max((difflib.SequenceMatcher(None, gecis, k).ratio(), s) for k, s in tum_kelimeler)
    return en_iyi[1] if en_iyi[0] >= 0.8 else None


kararlar = {s: [] for s in terimler}
for metin, varliklar in zip(dis["epikriz"], tahmin):
    kucuk = turkce_kucuk(metin)
    gecisler = sorted((s0, e0, tanisi(kucuk[s0:e0])) for s0, e0, tur in varliklar if tur == "DIAG")
    gecisler = [m for m in gecisler if m[2]]
    bulunan = set()
    for i, (s0, e0, s) in enumerate(gecisler):
        sonraki = next((m[0] for m in gecisler[i + 1:] if m[2] != s), len(kucuk))
        sinir = min(x for x in (kucuk.find(".", e0), kucuk.find(";", e0), sonraki, len(kucuk)) if x >= e0)
        cumle_basi = max(kucuk.rfind(". ", 0, s0), kucuk.rfind(": ", 0, s0)) + 1
        if any(w in kucuk[cumle_basi:s0] for w in yakin) or any(w in kucuk[e0:sinir] for w in olumsuz):
            continue
        bulunan.add(s)
    for s0, e0, tur in varliklar:
        if tur in ("DRUG", "PROC"):
            bulunan.update(s for s, kelimeler in ilaclar.items()
                           if any(re.search(r"(?<!\w)" + re.escape(w), kucuk[s0:e0]) for w in kelimeler))
    egfr = re.search(r"egfr\s*%?\s*(\d+(?:,\d+)?)", kucuk)
    if egfr and float(egfr.group(1).replace(",", ".")) < 60:
        bulunan.add("KBH")
    hba1c = re.search(r"hba1c\s*%?\s*(\d+(?:,\d+)?)", kucuk)
    if hba1c and float(hba1c.group(1).replace(",", ".")) >= 6.5:
        bulunan.add("DM")
    for s in terimler:
        kararlar[s].append(int(s in bulunan))
tablo = skor(kararlar)
cevap = round(float(tablo["F1"].mean()), 3)
teach.check_step("Prompt 2.3", "cevap", globals())

**Sonuç ne gösteriyor?** Modelin varlıklarından elde edilen altı tanı B hastanesinde yaklaşık 0,754 makro F1 değerine ulaşır; 1. duraktaki TF-IDF modelinde bu değer 0,708 idi. Sonucu iki tür hata düşürür. Model tanı kısaltmalarının çoğunu ve bütün işlemleri kaçırır. Ayrıca bir varlık "KBY (-)" örneğinde olduğu gibi işareti de içine aldığında durum kuralı varlıktan sonra "(-)" bulamaz ve yok sayılması gereken hastalığı var kabul eder. Modelin varlıkları, dolayısıyla makro F1 değeri de bir eğitimden ötekine değişir: Dersin farklı seed değerleriyle yapılan altı çalıştırmasında makro F1 değeri 0,613 ile 0,792 arasında kalır. Bu yüzden denetim, dersin sonucunun çevresinde geniş bir aralığı kabul eder.

#### Prompt 2.4: Seyirden ve EKG'den STEMI

**Amaç.** Bu promptla epikrizlerin akut koroner sendrom ve ST segmenti hakkında yazdıklarından STEMI geçiren hastalar bulunur ve sonuç iki hastanede değerlendirilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: İki hastanenin epikrizlerinde ST yükselmeli miyokard enfarktüsü geçiren hastaları bulmak ve basit bir kuralın onları ne kadar güvenilir biçimde bulduğunu öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; A hastanesinin test kümesine ait, tam cümlelerle yazılmış 600 epikrizi ve B hastanesine ait, kısa bir üslupla yazılmış 600 epikrizi, yani Türkçe yazılmış toplam 1.200 epikrizi tutar. Her satır bir hastadır: "küme" sütunu kümeyi ("test" ya da "dış"), "epikriz" sütunu epikrizi verir; "STEMI" sütunu sanal kliniğin kayıt sistemi ST yükselmeli miyokard enfarktüsü kaydettiyse 1, kaydetmediyse 0'dır.  
> Görev: Her epikrizi Türkçe kurallarına göre küçük harfe çevir: Önce I'yı ı, İ'yi i yap, ardından öteki harfleri küçült. Bir epikriz akut koroner sendromdan, yani "akut koroner sendrom" ifadesinden ya da tam kelime olarak "aks" kısaltmasından söz ediyorsa ve aynı cümlede ardından "dışlandı", "yok", "izlenmedi" ya da "saptanmadı" kelimelerinden biri gelmiyorsa, ayrıca "st yükselmesi" ifadesini de içeriyorsa ona STEMI de. Sonucu "STEMI" sütunuyla karşılaştır ve her küme için STEMI geçiren hasta sayısını, bulunan hasta sayısını, duyarlılığı, pozitif öngörü değerini ve F1 değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her küme için bir satır ve bu değerleri içeren bir tablo yazdır; "test" kümesinin F1 değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her değer 0 ile 1 arasında olmalı ve her kümedeki STEMI geçiren hasta sayısı o kümedeki "STEMI" sütununun toplamına eşit olmalı.  

In [ ]:
# Prompt 2.4 için veri: Promptun tarif ettiği "veri" tablosu
from vclinic import make_cohort
_reg = make_cohort(len(archive)).set_index("patient_id")
_m = te | ex
veri = pd.DataFrame({"küme": [_SETS[s] for s in split[_m]], "epikriz": texts[_m],
                     "STEMI": (_reg.loc[archive["patient_id"][_m], "stemi_territory"] != "none").astype(int).to_numpy()})
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 2.4 kontrolü
teach.check_step("Prompt 2.4", "cevap", globals())

In [ ]:
#@title Prompt 2.4 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
from vclinic import make_cohort
_reg = make_cohort(len(archive)).set_index("patient_id")
_m = te | ex
veri = pd.DataFrame({"küme": [_SETS[s] for s in split[_m]], "epikriz": texts[_m],
                     "STEMI": (_reg.loc[archive["patient_id"][_m], "stemi_territory"] != "none").astype(int).to_numpy()})
satirlar = []
for kume, parca in veri.groupby("küme", sort=False):
    bulunan = []
    for metin in parca["epikriz"]:
        kucuk = metin.replace("I", "ı").replace("İ", "i").lower()
        aks = False
        for e in re.finditer(r"akut koroner sendrom|(?<!\w)aks(?!\w)", kucuk):
            son = kucuk.find(".", e.end())
            kalan = kucuk[e.end():son if son >= 0 else len(kucuk)]
            if not any(k in kalan for k in ["dışlandı", "yok", "izlenmedi", "saptanmadı"]):
                aks = True
        bulunan.append(aks and "st yükselmesi" in kucuk)
    gercek = parca["STEMI"].to_numpy() == 1
    tahmin = np.array(bulunan)
    dp, yp, yn = int((tahmin & gercek).sum()), int((tahmin & ~gercek).sum()), int((~tahmin & gercek).sum())
    satirlar.append({"küme": kume, "STEMI": int(gercek.sum()), "bulunan": int(tahmin.sum()),
                     "duyarlılık": round(dp / max(1, dp + yn), 3), "PPV": round(dp / max(1, dp + yp), 3),
                     "F1": round(2 * dp / max(1, 2 * dp + yp + yn), 3)})
tablo = pd.DataFrame(satirlar)
print(tablo)
cevap = float(tablo.loc[tablo["küme"] == "test", "F1"].iloc[0])
teach.check_step("Prompt 2.4", "cevap", globals())

**Sonuç ne gösteriyor?** Kural A hastanesindeki her STEMI'yi bulur: Bu hastanenin epikrizleri seyirde akut koroner sendromu, EKG'de ise ST yükselmesi olan derivasyonları adlandırır. B hastanesi kısa seyrinde yalnızca "AKS" yazar ve ST segmenti hakkında hiçbir şey yazmaz; bu yüzden orada ne bir kural ne de bir model STEMI bulabilir. Bir raporda bulunmayan bilgi o rapordan çıkarılamaz.

**«Durak 2 · Prompt ile uygulama» tamamlandı.** Sırada hemen aşağıdaki «Durak 2 · Hazır uygulama» bölümü var. Durağı burada bırakacaksanız [ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#summaries-and-large-language-models-as-clinical-assistants) ve okumaya «Radyoloji raporu otomatik özetleme» bölümünden devam edin.

### Durak 2 · Hazır uygulama

Bu bölüm dersin küçük Türkçe BERT modelini ve BioBERT'in tokenizer'ını Hugging Face Hub'dan yükler, Türkçe modele epikrizlerdeki varlıklar için ince ayar yapar, modelin bulduğu varlıkları altı tanıya dönüştürür ve mini projeyi, yani NER pipeline ile STEMI, AF ve kalp yetersizliğini çalıştırır.

Hücreleri yukarıdan aşağıya çalıştırın. *Python notu* hücreleri kodu açıklar, *Gözlemle ve düşün* hücreleri ise sonuçlar üzerine sorular sorar.

**Başlangıç hücresi.** Aşağıdaki hücre, «Durak 2 · Hazır uygulama» bölümü için kurulumu, sonraki hücrelerin kullandığı veri çağrılarını ve önceki hücrelerin kodlarını yineler; böylece kodlar hata vermeden çalışır. «Durak 1 · Hazır uygulama» bölümünde yazılan fonksiyonlar, onları aynen saklayan ders paketinden alınır; bu fonksiyonların kodları o bölümdedir. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Durak 2 · Hazır uygulama» bölümü
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# önceki bölümlerin fonksiyonları: Ders paketi onları aynen saklar
from vclinic.week06 import tr_lower, words, se_ppv_f1, next_other, lab_value

# alıştırma promptlarının tabloları (etkileşimli labın F bölümü)
from vclinic.week06 import alistirma_tablosu

# önceki hücrelerden yinelenen kod: Aşağıdaki hücrelerin kullandığı veriler ve kodlar
import difflib
import re
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
archive = reports.archive()
LABELS = list(reports.LABELS)
texts = archive["text_tr"].to_numpy()
y = archive[LABELS].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
case = int(np.where(archive["patient_id"] == "VC-2444")[0][0])
def label_table(y_true, y_pred):
    """Her tanı için bir satır: Prevalans, duyarlılık, PPV ve F1; son satır ortalama F1'dir (makro F1)."""
    rows = []
    for k, label in enumerate(LABELS):
        se, ppv, f1 = se_ppv_f1(y_true[:, k], y_pred[:, k])
        rows.append([label, int(y_true[:, k].sum()), round(se, 3), round(ppv, 3), round(f1, 3)])
    table = pd.DataFrame(rows, columns=["tanı", "tanılı", "duyarlılık", "PPV", "F1"])
    print("Makro F1:", round(float(table["F1"].mean()), 3))
    return table
TERMS = {
    "HF": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
    "AF": ["atriyal fibrilasyon", "af", "afib"],
    "CAD": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
    "HT": ["hipertansiyon", "ht"],
    "DM": ["diyabet", "diyabetes mellitus", "dm"],
    "CKD": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"],
}
PATTERNS = {label: re.compile(r"(?<!\w)(" + "|".join(sorted(map(re.escape, terms), key=len, reverse=True)) + r")(?!\w)")
            for label, terms in TERMS.items()}
def find_terms(text):
    """Bir tanının metindeki her geçişi: (başlangıç, bitiş, tanı), metindeki sırasıyla."""
    low = tr_lower(text)
    found = [(m.start(), m.end(), label) for label, pattern in PATTERNS.items() for m in pattern.finditer(low)]
    return sorted(found)
NEGATION = ["yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)"]
POSSIBLE = ["şüphe", "olası", "ön tanı"]
FAMILY = ["babası", "annesi", "kardeşi", "babada", "annede", "soygeçmiş"]
def assertion(text, start, end, next_start, window=None):
    """Bir geçişin durumu: var (present), yok (negated), olası (possible) ya da aile öyküsü (family).

    Geçişten sonraki kapsam, cümle sonunda ya da başka bir tanının bir sonraki geçişinin başladığı yerde (next_start) biter; window verilirse bunun yerine o kadar karakter sonra biter."""
    low = tr_lower(text)
    if window is None:
        stop = min(x for x in (low.find(".", end), low.find(";", end), next_start, len(low)) if x >= end)
    else:
        stop = min(len(low), end + window)
    after = low[end:stop]
    sentence_start = max(low.rfind(". ", 0, start), low.rfind(": ", 0, start)) + 1
    before = low[sentence_start:start]
    if any(w in before for w in FAMILY):
        return "family"
    if any(w in after for w in NEGATION):
        return "negated"
    if any(w in after for w in POSSIBLE):
        return "possible"
    return "present"
EVIDENCE = {
    "HF": ["furosemid", "sakubitril"],
    "AF": ["apiksaban", "rivaroksaban", "edoksaban", "dabigatran"],
    "CAD": ["klopidogrel", "tikagrelor", "pkg", "kabg", "stent"],
    "HT": ["amlodipin", "hidroklorotiyazid"],
    "DM": ["metformin", "gliklazid", "insülin", "sitagliptin"],
}

#### Önceden eğitilmiş Türkçe bir model

*Ders sayfasında: [BERT / BioBERT: Hugging Face Hub üzerinden yükleme](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#pretrained-language-models-learning-a-language-before-a-task)*

Dersin küçük Türkçe BERT modelinde iki katman, token başına 128 sayı ve 32.000 parçalık bir sözlük vardır. Bu bölümde arşivin token'ları üç tür küçük harf işlemiyle sayılır, modelin büyüklüğüne bakılır ve modele klinik cümlelerdeki boşlukları doldurtulur.

**Formüllerden koda.** Bilinmeyen token'ların oranı, `tokenizer.unk_token_id` sayısının bütün token'lara bölümüdür. Modelin parametreleri: `sum(p.numel() for p in model.parameters())`. Maskeli dil modeli: Cümlede [MASK] bulunan `pipeline("fill-mask", model=MODEL)`.

In [ ]:
MODEL = "ytu-ce-cosmos/turkish-tiny-bert-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

forms = {"yazıldığı gibi": lambda t: t, "Python'un lower() metodu": str.lower, "Türkçe küçük harf": tr_lower}
for name, f in forms.items():
    for hospital, mask in (("A", tr), ("B", ex)):
        ids = [tokenizer(f(t))["input_ids"][1:-1] for t in texts[mask]]
        n = sum(len(i) for i in ids)
        unknown = sum(i.count(tokenizer.unk_token_id) for i in ids)
        print(f"{name:26} hastane {hospital}: {n / len(ids):6.1f} token/epikriz, [UNK] {100 * unknown / n:5.1f} %")
for word in ["KALP YETMEZLİĞİ", "PROKSİMAL", "hipertansyon"]:
    print(word, "→", tokenizer.tokenize(word), tokenizer.tokenize(word.lower()), tokenizer.tokenize(tr_lower(word)))

**Python notu.** `tokenizer(text)["input_ids"][1:-1]` ilk ve son token'ı, yani [CLS] ve [SEP] token'larını atar. `i.count(tokenizer.unk_token_id)` bir epikrizdeki bilinmeyen token'ları sayar. `tokenizer.tokenize(word)` parçaları sayı yerine metin olarak verir. `forms` sözlüğü üç fonksiyon tutar; böylece aynı döngü üçünü de dener. `str.lower` Python'un kendi metodudur ve burada bir fonksiyon olarak kullanılmıştır.

In [ ]:
fill = pipeline("fill-mask", model=MODEL)
print("Modelin parametreleri:", sum(p.numel() for p in fill.model.parameters()))
for sentence in ["hasta [MASK] ağrısı yakınmasıyla başvurdu.", "ekg: sinüs [MASK].", "taburcu tedavisi: [MASK] 40 mg."]:
    print(sentence, [(g["token_str"], round(g["score"], 3)) for g in fill(sentence)])
for word in ["göğüs", "ritmi", "furosemid"]:
    print(word, "→", tokenizer.tokenize(word))

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c184)*

**Gözlemle ve düşün.** Model boşlukları genel Türkçe metinlerden öğrendikleriyle doldurur. Ağrı cümlesindeki tahminler arasında "göğüs" kaçıncı sıradadır? Kardiyoloji raporlarıyla önceden eğitilmiş bir model onu neden ilk sıraya koyabilir? Bir [MASK] tek bir token'ın yerini tutar: Model EKG cümlesi için neden "ritmi", ilaç cümlesi için neden "furosemid" önermez?

**Alıştırma F.** B hastanesinin bir epikrizinde Türkçe küçük harfe çevirmeden sonra, iki özel token hariç ortalama kaç token vardır? Ortalamayı bir ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("F")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma F", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(np.mean([len(tokenizer(tr_lower(t))["input_ids"]) - 2 for t in texts[ex]])), 1)
teach.check("Alıştırma F", cevap)

**Hugging Face Hub'dan BioBERT.** *Ders sayfasında: [BERT / BioBERT: Hugging Face Hub üzerinden yükleme](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#pretrained-language-models-learning-a-language-before-a-task)*

BioBERT, biyomedikal makalelerle eğitimi sürdürülmüş İngilizce BERT modelidir. Aşağıdaki hücre Hugging Face Hub'dan yalnızca birkaç yüz kilobaytlık tokenizer'ını yükler ve BioBERT ile eğitimin Türkçe modelinin İngilizce ve Türkçe birer klinik cümleyi ve arşivin epikrizlerini nasıl böldüğünü karşılaştırır.

**Formüllerden koda.** Hub'daki her model adıyla yüklenir: `AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")`. BioBERT büyük harfleri korur: `do_lower_case=False`. Kelime başına parça: Bir epikrizin token sayısının kelime sayısına bölümü.

In [ ]:
bio_tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2", do_lower_case=False)

print("Sözlük: BioBERT", len(bio_tokenizer), "· eğitimin Türkçe modeli", len(tokenizer))
for sentence in ["Immunoglobulin levels were normal.", "Kalp yetmezliğinin nedeni koroner arter hastalığıdır."]:
    print(sentence)
    print("   BioBERT:", bio_tokenizer.tokenize(sentence))
    print("   Türkçe model:", tokenizer.tokenize(tr_lower(sentence)))
pieces = lambda tok, f: float(np.mean([len(tok.tokenize(f(t))) / len(words(t)) for t in texts[te][:200]]))
print("Kelime başına parça, 200 test epikrizi: BioBERT", round(pieces(bio_tokenizer, lambda t: t), 2),
      "· Türkçe model", round(pieces(tokenizer, tr_lower), 2))

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c190)*

**Python notu.** `AutoTokenizer.from_pretrained(name)` tokenizer'ın dosyalarını bir kez indirir ve bilgisayardaki bir klasörde saklar; ikinci çağrı dosyaları oradan okur. `len(tokenizer)` sözlüğün büyüklüğüdür. `lambda tok, f: ...` adsız küçük bir fonksiyon tanımlar; burada bir tokenizer ve bir küçük harf fonksiyonu alır, böylece iki model birer satırla karşılaştırılır.

**Gözlemle ve düşün.** BioBERT tıbbı İngilizce makalelerden öğrenmiştir, ama sözlüğü İngilizce BERT'in sözlüğüdür. "Immunoglobulin" kelimesini ve Türkçe bir klinik cümleyi nasıl böler? Türkçe bir kelimenin parçalarının bir model için anlam taşıması için modelin neden çok daha fazla Türkçe metinle eğitilmesi gerekir?

#### Varlıklar için modele ince ayar

*Ders sayfasında: [Varlıklar için önceden eğitilmiş bir modele ince ayar yapmak](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#fine-tuning-a-pretrained-model-for-entities)*

Önceden eğitilmiş modele her token'a yedi etiketten birini veren bir katman eklenir ve bütün model açıklamalı eğitim epikrizleriyle dört epoch boyunca eğitilir. Modelin bulduğu varlıklar test kümesinde ve B hastanesinde tam eşleşmeye göre değerlendirilir.

**Formüllerden koda.** BIO düzeninin etiketleri: `TAGS`. Bir token, ilk karakterinin içinde bulunduğu varlığın etiketini alır: `encode` içindeki `e["start"] <= start < e["end"]`. Kayıp, etiketi olan token'lar üzerinden hesaplanan çapraz entropidir; özel token'lar ve dolgu −100 alır ve kayıp bunları dışarıda bırakır. Varlıkların kesinliği, duyarlılığı ve F1 değeri: `span_scores`.

**Başlangıç hücresi.** Aşağıdaki hücre, «Varlıklar için modele ince ayar» kısmı için kurulumu, sonraki hücrelerin kullandığı veri çağrılarını ve önceki hücrelerin kodlarını yineler; böylece kodlar hata vermeden çalışır. «Durak 1 · Hazır uygulama» bölümünde yazılan fonksiyonlar, onları aynen saklayan ders paketinden alınır; bu fonksiyonların kodları o bölümdedir. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Varlıklar için modele ince ayar» kısmı
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# önceki bölümlerin fonksiyonları: Ders paketi onları aynen saklar
from vclinic.week06 import tr_lower

# alıştırma promptlarının tabloları (etkileşimli labın F bölümü)
from vclinic.week06 import alistirma_tablosu

# önceki hücrelerden yinelenen kod: Aşağıdaki hücrelerin kullandığı veriler ve kodlar
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
archive = reports.archive()
texts = archive["text_tr"].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
case = int(np.where(archive["patient_id"] == "VC-2444")[0][0])
MODEL = "ytu-ce-cosmos/turkish-tiny-bert-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

**Ders sayfasında:** [Çözümlü örnek: Neredeyse doğru bir varlık](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#worked-example-an-entity-that-is-almost-right). Aşağıdaki hücreler modelin bulduğu varlıkları aynı biçimde, tam eşleşmeye göre değerlendirir.

In [ ]:
TAGS = ["O", "B-DIAG", "I-DIAG", "B-DRUG", "I-DRUG", "B-PROC", "I-PROC"]
MAX_LENGTH = 384


def encode(text, entities):
    """Bir raporun token'ları ve her birinin etiketi: Bir varlığın ilk token'ı B-, ötekiler I-, dışındakiler O; özel token'lar -100."""
    enc = tokenizer(tr_lower(text), return_offsets_mapping=True, truncation=True, max_length=MAX_LENGTH)
    tags = []
    for start, end in enc["offset_mapping"]:
        if start == end:
            tags.append(-100)
            continue
        tag = "O"
        for e in entities:
            if e["start"] <= start < e["end"]:
                tag = ("B-" if start == e["start"] else "I-") + e["label"]
                break
        tags.append(TAGS.index(tag))
    return enc["input_ids"], tags, enc["offset_mapping"]


def spans(tag_ids, offsets):
    """Token'ların etiketlerinden varlıklar: Raporun karakterleri cinsinden (başlangıç, bitiş, tür)."""
    found, current = [], None
    for t, (start, end) in zip(tag_ids, offsets):
        if start == end:
            continue
        tag = TAGS[t]
        if tag.startswith("B-") or (tag.startswith("I-") and (current is None or current[2] != tag[2:])):
            if current:
                found.append(tuple(current))
            current = [start, end, tag[2:]]
        elif tag.startswith("I-"):
            current[1] = end
        else:
            if current:
                found.append(tuple(current))
            current = None
    if current:
        found.append(tuple(current))
    return found


def batches(items, size, shuffle=False):
    """Eğitim ve tahmin için aynı uzunluğa tamamlanmış rapor grupları."""
    order = np.random.permutation(len(items)) if shuffle else np.arange(len(items))
    for s in range(0, len(items), size):
        group = [items[i] for i in order[s:s + size]]
        width = max(len(g[0]) for g in group)
        ids = torch.tensor([g[0] + [0] * (width - len(g[0])) for g in group])
        tags = torch.tensor([g[1] + [-100] * (width - len(g[1])) for g in group])
        mask = torch.tensor([[1] * len(g[0]) + [0] * (width - len(g[0])) for g in group])
        yield ids, tags, mask


def span_scores(gold, predicted, kind=None):
    """Varlıkların tam eşleşmeye göre kesinliği, duyarlılığı ve F1 değeri; kind tek bir türle sınırlar."""
    tp = fp = fn = 0
    for g, p in zip(gold, predicted):
        g = {(e[0], e[1], e[2]) for e in g if kind is None or e[2] == kind}
        p = {e for e in p if kind is None or e[2] == kind}
        tp, fp, fn = tp + len(g & p), fp + len(p - g), fn + len(g - p)
    precision, recall = tp / max(1, tp + fp), tp / max(1, tp + fn)
    return round(precision, 3), round(recall, 3), round(2 * tp / max(1, 2 * tp + fp + fn), 3)

ids, tags, offsets = encode(texts[case], archive["entities_tr"][case])
for t, tag, (s, e) in list(zip(tokenizer.convert_ids_to_tokens(ids), tags, offsets))[30:48]:
    print(f"{t:14} {TAGS[tag] if tag >= 0 else '':8} {texts[case][s:e]}")

**Python notu.** `encode`, tokenizer'ı `return_offsets_mapping=True` ile çağırır; bu ayar her token için ilk ve son karakterinin metindeki konumlarını döndürür. `tr_lower` metnin uzunluğunu koruduğu için bu konumlar özgün epikrizin konumlarıdır. `spans` tahmin edilen etiketleri sırayla dolaşır ve bir B- etiketiyle onu izleyen I- etiketlerini tek bir varlıkta birleştirir. `batches` bir gruptaki epikrizleri 0 token'ıyla aynı uzunluğa tamamlar ve modele hangi token'ların gerçek olduğunu söyleyen `mask` dizisini kurar.

In [ ]:
EPOCHS, BATCH, LEARNING_RATE, SEED = 4, 16, 3e-4, 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
train_items = [encode(t, e)[:2] for t, e in zip(texts[tr], archive["entities_tr"][tr])]
ner_model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(TAGS))
optimizer = torch.optim.AdamW(ner_model.parameters(), lr=LEARNING_RATE)
for epoch in range(EPOCHS):
    ner_model.train()
    total = 0.0
    for ids, tags, mask in batches(train_items, BATCH, shuffle=True):
        loss = ner_model(input_ids=ids, attention_mask=mask, labels=tags).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(ids)
    print(f"epoch {epoch + 1}: ortalama kayıp {total / len(train_items):.4f}")


def predict_spans(text_list):
    """Modelin bir listedeki her raporda bulduğu varlıklar."""
    ner_model.eval()
    out = []
    with torch.no_grad():
        for text in text_list:
            ids, _, offsets = encode(text, [])
            logits = ner_model(input_ids=torch.tensor([ids])).logits[0]
            out.append(spans(logits.argmax(-1).tolist(), offsets))
    return out

results = {}
for name, mask in (("test", te), ("dış", ex)):
    gold = [[(e["start"], e["end"], e["label"]) for e in es] for es in archive["entities_tr"][mask]]
    predicted = predict_spans(list(texts[mask]))
    results[name] = predicted
    print(name, {kind or "hepsi": span_scores(gold, predicted, kind) for kind in (None, "DIAG", "DRUG", "PROC")})

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c198)*

**Python notu.** Eğitim döngüsü 3-5. haftalardakiyle aynıdır; üç fark vardır. Model, yedi çıktısı olan yeni bir son katmanla `from_pretrained` üzerinden hazır gelir. Batch, modelin adlarıyla aldığı bir tensör sözlüğüdür (`input_ids`, `attention_mask` ve `labels`); `labels` verildiğinde model kaybı kendisi hesaplar (`.loss`). Optimizasyon algoritması AdamW'dir; Transformer modellerine uygun bir ağırlık azaltımı içeren Adam'dır. `torch.no_grad()` tahmin sırasında gradyanları kapatır. `torch.manual_seed(SEED)` ve `np.random.seed(SEED)` yeni katmanın rastgele başlangıcını ve grupların sırasını sabitler; böylece aynı bilgisayardaki ikinci bir çalıştırma aynı sonucu verir.

**Gözlemle ve düşün.** Modelin B hastanesinde kaçırdığı ya da farklı sınırlarla bulduğu bazı varlıkları yazdırın. Bunlar A hastanesinin hiç yazmadığı kelimeler mi, yoksa sınırları bir ekle değişen varlıklar mı?

**Gözlemle ve düşün.** `SEED` değerini başka bir sayı yapın ve modeli yeniden eğitin. Hangi F1 değeri daha çok değişir: Test kümesininki mi, B hastanesininki mi? Bu durum, yalnızca öğrendiği hastanede sınanan bir model hakkında ne söyler?

**Alıştırma G.** 1.800 eğitim epikrizinin token'larından kaçı `encode` ile bir başlangıç etiketi, yani B-DIAG, B-DRUG ya da B-PROC alır?

In [ ]:
veri = alistirma_tablosu("G")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma G", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = 0
for t, es in zip(texts[tr], archive["entities_tr"][tr]):
    cevap += sum(1 for x in encode(t, es)[1] if x >= 0 and TAGS[x].startswith("B-"))
teach.check("Alıştırma G", cevap)

#### Varlıklardan tanılara

*Ders sayfasında: [Adı geçmeyen kanıt: İlaçlar ve laboratuvar değerleri](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#evidence-without-a-name-drugs-and-laboratory-values)*

Modelin varlıkları her epikrizin altı tanısına dönüşür: Bir tanı varlığı kelimelerine göre ya da yanlış yazılmışsa sözlükteki en benzer kelimeye göre altı tanıdan birine bağlanır; durumu «Durak 1 · Hazır uygulama» bölümündeki kuraldan gelir; ilaç ve işlem varlıkları ile iki laboratuvar değeri geri kalanı ekler.

**Formüllerden koda.** İki kelimenin benzerliği, `normalise` içindeki `difflib.SequenceMatcher` oranıdır ve 0 ile 1 arasındadır. Açıklanmış varlıklar işlem hattının üst sınırını verir: `entity_labels(text, gold)`. Modelin varlıkları ise sonucunu verir.

**Başlangıç hücresi.** Aşağıdaki hücre, «Varlıklardan tanılara» kısmı için kurulumu, sonraki hücrelerin kullandığı veri çağrılarını ve önceki hücrelerin kodlarını yineler; böylece kodlar hata vermeden çalışır. «Durak 1 · Hazır uygulama» bölümünde yazılan fonksiyonlar, onları aynen saklayan ders paketinden alınır; bu fonksiyonların kodları o bölümdedir. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Varlıklardan tanılara» kısmı
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# önceki bölümlerin fonksiyonları: Ders paketi onları aynen saklar
from vclinic.week06 import tr_lower, se_ppv_f1, next_other, lab_value

# alıştırma promptlarının tabloları (etkileşimli labın F bölümü)
from vclinic.week06 import alistirma_tablosu

# önceki hücrelerden yinelenen kod: Aşağıdaki hücrelerin kullandığı veriler ve kodlar
import difflib
import re
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
archive = reports.archive()
LABELS = list(reports.LABELS)
texts = archive["text_tr"].to_numpy()
y = archive[LABELS].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
def label_table(y_true, y_pred):
    """Her tanı için bir satır: Prevalans, duyarlılık, PPV ve F1; son satır ortalama F1'dir (makro F1)."""
    rows = []
    for k, label in enumerate(LABELS):
        se, ppv, f1 = se_ppv_f1(y_true[:, k], y_pred[:, k])
        rows.append([label, int(y_true[:, k].sum()), round(se, 3), round(ppv, 3), round(f1, 3)])
    table = pd.DataFrame(rows, columns=["tanı", "tanılı", "duyarlılık", "PPV", "F1"])
    print("Makro F1:", round(float(table["F1"].mean()), 3))
    return table
TERMS = {
    "HF": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
    "AF": ["atriyal fibrilasyon", "af", "afib"],
    "CAD": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
    "HT": ["hipertansiyon", "ht"],
    "DM": ["diyabet", "diyabetes mellitus", "dm"],
    "CKD": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"],
}
PATTERNS = {label: re.compile(r"(?<!\w)(" + "|".join(sorted(map(re.escape, terms), key=len, reverse=True)) + r")(?!\w)")
            for label, terms in TERMS.items()}
def find_terms(text):
    """Bir tanının metindeki her geçişi: (başlangıç, bitiş, tanı), metindeki sırasıyla."""
    low = tr_lower(text)
    found = [(m.start(), m.end(), label) for label, pattern in PATTERNS.items() for m in pattern.finditer(low)]
    return sorted(found)
NEGATION = ["yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)"]
POSSIBLE = ["şüphe", "olası", "ön tanı"]
FAMILY = ["babası", "annesi", "kardeşi", "babada", "annede", "soygeçmiş"]
def assertion(text, start, end, next_start, window=None):
    """Bir geçişin durumu: var (present), yok (negated), olası (possible) ya da aile öyküsü (family).

    Geçişten sonraki kapsam, cümle sonunda ya da başka bir tanının bir sonraki geçişinin başladığı yerde (next_start) biter; window verilirse bunun yerine o kadar karakter sonra biter."""
    low = tr_lower(text)
    if window is None:
        stop = min(x for x in (low.find(".", end), low.find(";", end), next_start, len(low)) if x >= end)
    else:
        stop = min(len(low), end + window)
    after = low[end:stop]
    sentence_start = max(low.rfind(". ", 0, start), low.rfind(": ", 0, start)) + 1
    before = low[sentence_start:start]
    if any(w in before for w in FAMILY):
        return "family"
    if any(w in after for w in NEGATION):
        return "negated"
    if any(w in after for w in POSSIBLE):
        return "possible"
    return "present"
EVIDENCE = {
    "HF": ["furosemid", "sakubitril"],
    "AF": ["apiksaban", "rivaroksaban", "edoksaban", "dabigatran"],
    "CAD": ["klopidogrel", "tikagrelor", "pkg", "kabg", "stent"],
    "HT": ["amlodipin", "hidroklorotiyazid"],
    "DM": ["metformin", "gliklazid", "insülin", "sitagliptin"],
}
MODEL = "ytu-ce-cosmos/turkish-tiny-bert-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
TAGS = ["O", "B-DIAG", "I-DIAG", "B-DRUG", "I-DRUG", "B-PROC", "I-PROC"]
MAX_LENGTH = 384
def encode(text, entities):
    """Bir raporun token'ları ve her birinin etiketi: Bir varlığın ilk token'ı B-, ötekiler I-, dışındakiler O; özel token'lar -100."""
    enc = tokenizer(tr_lower(text), return_offsets_mapping=True, truncation=True, max_length=MAX_LENGTH)
    tags = []
    for start, end in enc["offset_mapping"]:
        if start == end:
            tags.append(-100)
            continue
        tag = "O"
        for e in entities:
            if e["start"] <= start < e["end"]:
                tag = ("B-" if start == e["start"] else "I-") + e["label"]
                break
        tags.append(TAGS.index(tag))
    return enc["input_ids"], tags, enc["offset_mapping"]
def spans(tag_ids, offsets):
    """Token'ların etiketlerinden varlıklar: Raporun karakterleri cinsinden (başlangıç, bitiş, tür)."""
    found, current = [], None
    for t, (start, end) in zip(tag_ids, offsets):
        if start == end:
            continue
        tag = TAGS[t]
        if tag.startswith("B-") or (tag.startswith("I-") and (current is None or current[2] != tag[2:])):
            if current:
                found.append(tuple(current))
            current = [start, end, tag[2:]]
        elif tag.startswith("I-"):
            current[1] = end
        else:
            if current:
                found.append(tuple(current))
            current = None
    if current:
        found.append(tuple(current))
    return found
def batches(items, size, shuffle=False):
    """Eğitim ve tahmin için aynı uzunluğa tamamlanmış rapor grupları."""
    order = np.random.permutation(len(items)) if shuffle else np.arange(len(items))
    for s in range(0, len(items), size):
        group = [items[i] for i in order[s:s + size]]
        width = max(len(g[0]) for g in group)
        ids = torch.tensor([g[0] + [0] * (width - len(g[0])) for g in group])
        tags = torch.tensor([g[1] + [-100] * (width - len(g[1])) for g in group])
        mask = torch.tensor([[1] * len(g[0]) + [0] * (width - len(g[0])) for g in group])
        yield ids, tags, mask
def span_scores(gold, predicted, kind=None):
    """Varlıkların tam eşleşmeye göre kesinliği, duyarlılığı ve F1 değeri; kind tek bir türle sınırlar."""
    tp = fp = fn = 0
    for g, p in zip(gold, predicted):
        g = {(e[0], e[1], e[2]) for e in g if kind is None or e[2] == kind}
        p = {e for e in p if kind is None or e[2] == kind}
        tp, fp, fn = tp + len(g & p), fp + len(p - g), fn + len(g - p)
    precision, recall = tp / max(1, tp + fp), tp / max(1, tp + fn)
    return round(precision, 3), round(recall, 3), round(2 * tp / max(1, 2 * tp + fp + fn), 3)
EPOCHS, BATCH, LEARNING_RATE, SEED = 4, 16, 3e-4, 2026
np.random.seed(SEED)
train_items = [encode(t, e)[:2] for t, e in zip(texts[tr], archive["entities_tr"][tr])]
ner_model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(TAGS))
optimizer = torch.optim.AdamW(ner_model.parameters(), lr=LEARNING_RATE)
for epoch in range(EPOCHS):
    ner_model.train()
    total = 0.0
    for ids, tags, mask in batches(train_items, BATCH, shuffle=True):
        loss = ner_model(input_ids=ids, attention_mask=mask, labels=tags).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(ids)
    print(f"epoch {epoch + 1}: ortalama kayıp {total / len(train_items):.4f}")
def predict_spans(text_list):
    """Modelin bir listedeki her raporda bulduğu varlıklar."""
    ner_model.eval()
    out = []
    with torch.no_grad():
        for text in text_list:
            ids, _, offsets = encode(text, [])
            logits = ner_model(input_ids=torch.tensor([ids])).logits[0]
            out.append(spans(logits.argmax(-1).tolist(), offsets))
    return out
results = {}
for name, mask in (("test", te), ("dış", ex)):
    gold = [[(e["start"], e["end"], e["label"]) for e in es] for es in archive["entities_tr"][mask]]
    predicted = predict_spans(list(texts[mask]))
    results[name] = predicted
    print(name, {kind or "hepsi": span_scores(gold, predicted, kind) for kind in (None, "DIAG", "DRUG", "PROC")})

In [ ]:
ALL_TERMS = [(t, label) for label, terms in TERMS.items() for t in terms]
DRUG_EVIDENCE = {w: label for label, ws in EVIDENCE.items() for w in ws}


def normalise(mention):
    """Bir geçişin tanısı: İçerdiği bir sözlük kelimesinin tanısı; yoksa en benzer sözlük kelimesinin tanısı (benzerlik en az 0,8 ise)."""
    found = find_terms(mention)
    if found:
        return found[0][2]
    low = tr_lower(mention)
    score, label = max((difflib.SequenceMatcher(None, low, t).ratio(), lab) for t, lab in ALL_TERMS)
    return label if score >= 0.8 else None


def entity_labels(text, entities):
    """Bir raporun altı tanısı, varlıklarından: Durumlarıyla birlikte tanılar, ilaçlar ve işlemler ile iki laboratuvar değeri. Her varlık (başlangıç, bitiş, tür) biçimindedir."""
    mentions = sorted((s, e, normalise(text[s:e])) for s, e, kind in entities if kind == "DIAG")
    mentions = [m for m in mentions if m[2]]
    present = set()
    for i, (s, e, label) in enumerate(mentions):
        if assertion(text, s, e, next_other(mentions, i, text)) == "present":
            present.add(label)
    for s, e, kind in entities:
        if kind in ("DRUG", "PROC"):
            low = tr_lower(text[s:e])
            present.update(label for w, label in DRUG_EVIDENCE.items() if re.search(r"(?<!\w)" + re.escape(w), low))
    egfr, hba1c = lab_value(text, "egfr"), lab_value(text, "hba1c")
    if egfr is not None and egfr < 60:
        present.add("CKD")
    if hba1c is not None and hba1c >= 6.5:
        present.add("DM")
    return [int(label in present) for label in LABELS]

for word in ["hipertansyon", "koroner arter hastaligi", "diyebet", "kalp yetmezligi", "nefes darlığı"]:
    print(f"{word:26} → {normalise(word)}")
for name, mask, key in (("test", te, "test"), ("dış", ex, "dış")):
    gold = [[(e["start"], e["end"], e["label"]) for e in es] for es in archive["entities_tr"][mask]]
    print(name, "açıklanmış varlıklar:")
    print(label_table(y[mask], np.array([entity_labels(t, g) for t, g in zip(texts[mask], gold)])))
    print(name, "modelin varlıkları:")
    print(label_table(y[mask], np.array([entity_labels(t, p) for t, p in zip(texts[mask], results[key])])))

**Python notu.** `difflib.SequenceMatcher(None, a, b).ratio()` iki karakter dizisini harf harf karşılaştırır ve aynı metin için 1 verir. (skor, tanı) çiftleri üzerinde `max(...)` en yüksek skorlu çifti döndürür; tanı ancak skor 0,8'e ulaştığında tutulur, böylece "nefes darlığı" hiçbir tanıya bağlanmaz. `entity_labels` metni aramak yerine varlıkların konumlarını kullanır; böylece aynı fonksiyon hem açıklamalarla hem de tahminlerle çalışır.

**Gözlemle ve düşün.** B hastanesinin iki tablosunu karşılaştırın. Modelin varlıklarıyla hangi tanıların pozitif öngörü değeri düşer? B hastanesinin birkaç epikrizinde modelin varlıklarını yazdırın ve son karakterlerine bakın.

**Alıştırma H.** `entity_labels` açıklanmış varlıkları kullandığında B hastanesinin epikrizlerindeki makro F1 değeri kaçtır? Üç ondalık basamakla saklayın; bu değer işlem hattının üst sınırıdır.

In [ ]:
veri = alistirma_tablosu("H")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma H", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
gold = [[(e["start"], e["end"], e["label"]) for e in es] for es in archive["entities_tr"][ex]]
P = np.array([entity_labels(t, g) for t, g in zip(texts[ex], gold)])
cevap = round(float(np.mean([se_ppv_f1(y[ex, k], P[:, k])[2] for k in range(len(LABELS))])), 3)
teach.check("Alıştırma H", cevap)

**Mini proje: NER pipeline ile STEMI, AF ve kalp yetersizliği.** *Ders sayfasında: [Mini proje: Sentetik Türkçe epikrizlerden tanı (STEMI, AF, KY) otomatik çıkarımı: NER pipeline](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#mini-project-automatic-extraction-of-diagnoses-stemi-af-hf-from-synthetic-turkish-discharge-summaries-an-ner-pipeline)*

Mini projenin işlem hattı, ince ayarlı modelin varlıklarından üç tanı okur: AF ve kalp yetersizliği yukarıdaki hücrede olduğu gibi bulunur; STEMI ise akut koroner sendrom var durumundaysa ve epikriz ST yükselmesi bildiriyorsa verilir. Sanal kliniğin kayıt sistemi hangi hastaların STEMI geçirdiğini bildiği için işlem hattı iki hastanede de değerlendirilebilir.

**Formüllerden koda.** STEMI: Var durumunda bir "akut koroner sendrom" ya da "AKS" varlığı ve Türkçe küçük harfle "st yükselmesi"; bunlar `stemi_af_hf` içindedir. Gerçek durum: `registry["stemi_territory"] != "none"`.

In [ ]:
from vclinic import make_cohort

ACS_WORDS = ("akut koroner sendrom", "aks")
ST_UP = re.compile(r"st\s+yükselmesi")


def stemi_af_hf(text, entities):
    """Bir epikrizin STEMI, AF ve kalp yetersizliği durumu, varlıklarından: AF ve kalp yetersizliği entity_labels ile; akut koroner sendrom var durumundaysa ve metin ST yükselmesi bildiriyorsa STEMI."""
    six = dict(zip(LABELS, entity_labels(text, entities)))
    mentions = sorted((s, e, normalise(text[s:e])) for s, e, kind in entities if kind == "DIAG")
    mentions = [m for m in mentions if m[2]]
    acs = any(tr_lower(text[s:e]).strip() in ACS_WORDS and assertion(text, s, e, next_other(mentions, i, text)) == "present"
              for i, (s, e, label) in enumerate(mentions))
    return [int(acs and ST_UP.search(tr_lower(text)) is not None), six["AF"], six["HF"]]


registry = make_cohort(len(archive)).set_index("patient_id")
stemi_true = (registry.loc[archive["patient_id"], "stemi_territory"] != "none").astype(int).to_numpy()
truth3 = np.column_stack([stemi_true, y[:, LABELS.index("AF")], y[:, LABELS.index("HF")]])

rows = []
for name, mask, key in (("test, A hastanesi", te, "test"), ("dış, B hastanesi", ex, "dış")):
    P = np.array([stemi_af_hf(t, p) for t, p in zip(texts[mask], results[key])])
    for k, d in enumerate(["STEMI", "AF", "KY"]):
        se, ppv, f1 = se_ppv_f1(truth3[mask, k], P[:, k])
        rows.append({"küme": name, "tanı": d, "hasta": int(truth3[mask, k].sum()),
                     "duyarlılık": round(se, 3), "PPV": round(ppv, 3), "F1": round(f1, 3)})
mini_table = pd.DataFrame(rows)
print(mini_table.to_string(index=False))

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c215)*

**Python notu.** `make_cohort(len(archive))` arşivdeki 3.600 hastanın kayıt sistemini yeniden kurar; `.set_index("patient_id")` ve `.loc[archive["patient_id"]]` satırlarını epikrizlerin sırasına koyar. `np.column_stack` 0 ve 1 değerlerinden oluşan üç sütunu gerçek durum tablosunda birleştirir. Bir üreteç üzerinde `any(...)`, akut koroner sendromun var durumundaki ilk geçişinde durur.

**Gözlemle ve düşün.** İşlem hattı B hastanesinde hiç STEMI bulamaz. B hastanesinde STEMI geçirmiş bir hastanın epikrizini okuyun: Hastane seyir ve EKG hakkında ne yazmıştır? Herhangi bir işlem hattının STEMI'yi bulabilmesi için bir kardiyolog hangi cümleyi eklerdi?

**2. durak tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#summaries-and-large-language-models-as-clinical-assistants) ve okumaya «Radyoloji raporu otomatik özetleme» bölümünden devam edin. Bir sonraki uygulama durağında ders sayfası sizi yine bu Jupyter Notebook'a getirir.

## Durak 3: Anjiyografi raporları: Darlık derecesi ve damar

*Ders sayfasında: [3. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#stop-3)*

Burada sanal kliniğin anjiyografi raporları bir kayıt sisteminin okuyacağı gibi okunur. Dersteki işlem hattı bir rapordaki arterleri, segmentleri ve darlık derecelerini bulur, her bölgenin en büyük darlığını tutar ve hastalıklı damarları sayar. İşlem hattı önce tek bir rapora, sonra A hastanesinin test raporlarına, son olarak da büyük harflerle yazılmış B hastanesi raporlarına uygulanır.

**Durağın sonunda:** VC-2914 numaralı hastanın raporundaki hastalıklı damar sayısını (3) ve hastalıklı damar sayısının doğru bulunduğu raporların oranını A hastanesinin test kümesinde (0,959) ve B hastanesinde (1,0) bulacaksınız.

Durak üç bölümden oluşur: **Python temelleri**, **Prompt ile uygulama** ve **Hazır uygulama**. Bölümleri bu sırayla izleyin. Python biliyorsanız promptlarla, yalnızca sonucu görmek istiyorsanız hazır uygulamayla başlayabilirsiniz. Her bölüm kendi başlangıç hücresiyle açılır.

### Durak 3 · Python temelleri

*Ders sayfasında: [Anjiyografi raporları: Darlık derecesi ve damar](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#angiography-reports-the-degree-of-stenosis-and-the-vessel)*

Bu durağın hazır uygulaması bir anjiyografi raporunu cümlelere böler, her darlık derecesini bir sayı ya da bir aralık olarak okur, her bölgenin en büyük değerini tutar ve bu değeri dersin kategorilerinden birine koyar. Aşağıdaki örnekler bu yapıları kısa bir rapor üzerinde tanıtır: Bir kalıba göre bölme, bir düzenli ifadenin grupları, tek satırda kurulan bir sözlük ve art arda gelen koşullar.

Hücreleri sırayla çalıştırın. Bir örneğin altındaki *Python notu* yeni yapıyı açıklar. Alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerdedir.

**Başlangıç hücresi.** Aşağıdaki hücre, «Durak 3 · Python temelleri» bölümü için kurulumu, sonraki hücrelerin kullandığı veri çağrılarını ve önceki hücrelerin kodlarını yineler; böylece kodlar hata vermeden çalışır. «Durak 1 · Hazır uygulama» bölümünde yazılan fonksiyonlar, onları aynen saklayan ders paketinden alınır; bu fonksiyonların kodları o bölümdedir. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Durak 3 · Python temelleri» bölümü
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# önceki bölümlerin fonksiyonları: Ders paketi onları aynen saklar
from vclinic.week06 import tr_lower

# önceki hücrelerden yinelenen kod: Aşağıdaki hücrelerin kullandığı veriler ve kodlar
import re

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
report = "Endikasyon: Göğüs ağrısı. LAD proksimalde %80 darlık. 1. diyagonalde %50 darlık. RCA normal."
for sentence in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", report):
    print(tr_lower(sentence).startswith(("endikasyon", "öykü")), "|", sentence)

**Python notu.** `re.split` bir metni bir kalıbın her eşleşmesinde böler. `(?<=\.)` bölünen yerin önünde bir nokta, `(?=[A-ZÇĞİÖŞÜ0-9])` da ardında bir büyük harf ya da rakam bulunmasını ister; ikisi de metinden kesilip atılmaz. Böylece rapor, ardından bir boşluk ve bir büyük harf ya da rakam gelen her noktadan sonra bölünür ve "1. diyagonalde" tek parça kalır. `startswith` bir kelime demeti de alabilir; cümle demetteki kelimelerden biriyle başlıyorsa doğru değerini verir. Hazır uygulama endikasyon ve öykü cümlelerini bu biçimde atlar.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
NUMBER = r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?"
sentence = "lad proksimalde %70-80, mid segmentte yüzde 40 darlık"
for m in re.finditer(NUMBER, sentence):
    print(m.group(0), "→", m.group(1), m.group(2), "→", int(m.group(2) or m.group(1)))

**Python notu.** `re.finditer` kalıbın her eşleşmesini gruplarıyla birlikte döndürür. Kalıp "%" ya da "yüzde" ifadesinden sonra gelen bir sayıyı, `(?:\s?-\s?(\d{1,3}))?` da kısa çizgiden sonra gelebilecek ikinci bir sayıyı bulur; darlık tek bir sayıyla yazıldığında `m.group(2)` değeri `None` olur. `m.group(2) or m.group(1)` ikinci sayı varsa onu, yoksa birinci sayıyı alır: Ders sayfasındaki işlem hattında olduğu gibi bir aralık üst ucuyla sayılır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
best = {t: 0 for t in ["LM", "LAD", "LCX", "RCA"]}
for territory, value in [("LAD", 70), ("RCA", 40), ("LAD", 90), ("LCX", 50)]:
    best[territory] = max(best[territory], value)
print(best)

**Python notu.** `{t: 0 for t in [...]}` bir *sözlük üreteci*dir: Her bölge için 0 değerini taşıyan bir sözlük kurar. Ardından döngü her bölgenin en büyük darlığını tutar; çünkü `max(best[territory], value)` bir değerin yerine yalnızca daha büyük bir değer koyar. LAD'deki ikinci darlık, yani yüzde 90, ilkinin yerini alır. Hazır uygulama her raporun tablosunu bu biçimde, cümle cümle doldurur.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
def category(value):
    return 0 if value < 50 else 1 if value < 70 else 2 if value < 90 else 3 if value < 100 else 4


print([category(v) for v in (30, 60, 75, 95, 100)])
print(sum(best[t] >= 70 for t in ("LAD", "LCX", "RCA")))

**Python notu.** `a if koşul else b` bir koşullu ifadedir ve birkaç tanesi art arda yazılabilir: `category` koşullarını soldan sağa okur ve sağlanan ilk koşulun değerini döndürür. Bunlar ders sayfasındaki kategorilerdir: Yüzde 50'nin altı için 0'dan tıkanıklık için 4'e kadar. `best[t] >= 70` her bölge için `True` ya da `False` verir; `sum` da her `True` değerini 1 olarak sayar. Burada tek bir hastalıklı damar, LAD vardır. Hazır uygulamadaki `n_vessels` fonksiyonu hastalıklı damarları aynı biçimde sayar.

**Alıştırma 3.1.** Bir rapor cümlesi "cx orta segmentte %50-60, distalde %40 darlık" diyor. Bir aralık üst ucuyla sayıldığında bu cümledeki en büyük darlık derecesi yüzde kaçtır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 3.1", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
NUMBER = r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?"
cevap = max(int(m.group(2) or m.group(1)) for m in re.finditer(NUMBER, "cx orta segmentte %50-60, distalde %40 darlık"))
teach.check("Alıştırma 3.1", cevap)

**Alıştırma 3.2.** Bir raporun en büyük darlıkları LM'de yüzde 40, LAD'de yüzde 70, LCX'te yüzde 95 ve RCA'da yüzde 65'tir. Yüzde 70 ve üzerindeki LAD, LCX ve RCA bölgeleri sayıldığında raporda kaç hastalıklı damar vardır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 3.2", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
table = {"LM": 40, "LAD": 70, "LCX": 95, "RCA": 65}
cevap = sum(table[t] >= 70 for t in ("LAD", "LCX", "RCA"))
teach.check("Alıştırma 3.2", cevap)

**«Durak 3 · Python temelleri» tamamlandı.** Sırada hemen aşağıdaki «Durak 3 · Prompt ile uygulama» bölümü var. Durağı burada bırakacaksanız [ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#ai-pair-programming-part-6-reviewing-a-text-pipeline) ve okumaya «YZ asistanıyla çalışmak, 6. bölüm: Bir metin işlem hattını denetlemek» bölümünden devam edin.

### Durak 3 · Prompt ile uygulama

Bu bölümde üç prompt vardır: «Tek bir rapor, cümle cümle», «A hastanesinin bütün test raporları» ve «B hastanesinin büyük harfli raporları». Onarım promptu ve hata mesajı için prompt, [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-06/PROMPTLAR.md) ve 1. durağın promptlarında yer alır.

Her promptta dört adım vardır. Promptu kopyalayıp YZ asistanınıza gönderin. Altındaki veri hücresini çalıştırın: Bu hücre, promptun tarif ettiği "veri" tablosunu hazırlar ve gösterir. Asistanın kodunu boş hücreye yapıştırıp çalıştırın. Ardından kontrol hücresini çalıştırın; bu hücre *doğru* ya da *henüz değil* yazar. Asistanın kodu kontrolden geçmezse *Hazır kod* hücresini çalıştırıp devam edin.

**Başlangıç hücresi.** Aşağıdaki hücre, «Durak 3 · Prompt ile uygulama» bölümü için kurulumu ve bu bölümün veri çağrılarını yineler; böylece kodlar hata vermeden çalışır. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Durak 3 · Prompt ile uygulama» bölümü
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

from vclinic import reports
import re

if "angio" not in globals():
    archive_angio_ready = True
    def tr_lower(text):
        """Türkçe kurallarına göre küçük harf: I, ı olur; İ, i olur ve metnin uzunluğu değişmez."""
        return text.replace("I", "ı").replace("İ", "i").lower()


    def words(text):
        """Bir metnin küçük harfli kelimeleri: Harf ve rakam dizileri."""
        return re.findall(r"\w+", tr_lower(text))

    angio = reports.angiography_archive()
    TERRITORIES = ["LM", "LAD", "LCX", "RCA"]
    VESSELS = {"LM": r"sol ana koroner|lmca", "LAD": r"sol ön inen|\blad\b", "D1": r"birinci diyagonal|1\. diyagonal|\bd1\b",
               "D2": r"ikinci diyagonal|2\. diyagonal|\bd2\b", "LCX": r"sirkumfleks|\bl?cx\b", "OM1": r"birinci obtus|1\. obtus|\bom1\b",
               "OM2": r"ikinci obtus|2\. obtus|\bom2\b", "RCA": r"sağ koroner|\brca\b", "PDA": r"arka inen|posterior desendan|\bpda\b"}
    TERRITORY = {"LM": "LM", "LAD": "LAD", "D1": "LAD", "D2": "LAD", "LCX": "LCX", "OM1": "LCX", "OM2": "LCX", "RCA": "RCA", "PDA": "RCA"}
    WORDS = {"total oklüzyon": 100, "tam tıkanıklık": 100, "to": 100, "subtotal": 99, "kritik": 95,
             "ileri derecede": 80, "orta derecede": 60, "hafif": 30, "plak": 30, "plaklı": 30}
    NUMBER = r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?"
    WORD = r"(?<!\w)(" + "|".join(sorted(WORDS, key=len, reverse=True)) + r")(?!\w)"
    SKIP = ("endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim", "işlem")


    def stenoses(sentence):
        """Bir cümledeki her darlık derecesi, yüzde olarak: Sayılar (bir aralık üst ucuyla sayılır) ve kelimeler."""
        values = [int(m.group(2) or m.group(1)) for m in re.finditer(NUMBER, sentence)]
        values += [WORDS[m.group(1)] for m in re.finditer(WORD, sentence)]
        return values


    def angio_table(text, coreference=True):
        """Bir rapordaki her bölgenin en büyük darlığı. Cümle, büyük harften ya da rakamdan önceki noktada biter; arter adı geçmeyen bir cümle, önceki cümlenin arterine aittir."""
        best = {t: 0 for t in TERRITORIES}
        last = None
        for sentence in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", text):
            sentence = tr_lower(sentence)
            if sentence.startswith(SKIP):
                continue
            found = sorted((m.start(), v) for v, pattern in VESSELS.items() for m in re.finditer(pattern, sentence))
            if found:
                last = found[0][1]
            elif not coreference:
                continue
            if last is None:
                continue
            for value in stenoses(sentence):
                best[TERRITORY[last]] = max(best[TERRITORY[last]], value)
        return best


    def category(value):
        """50'nin altı 0, 50-69 için 1, 70-89 için 2, 90-99 için 3 ve tıkanma için 4."""
        return 0 if value < 50 else 1 if value < 70 else 2 if value < 90 else 3 if value < 100 else 4


    def n_vessels(table):
        """Hastalıklı damar sayısı: Yüzde 70 ya da daha fazla darlığı olan LAD, LCX ve RCA bölgeleri."""
        return sum(table[t] >= 70 for t in ("LAD", "LCX", "RCA"))
_TER = ["LM", "LAD", "CX", "RCA"]


def _reports(mask):
    """Kayıt sistemine göre her bölgenin en büyük darlığı ve hastalıklı damar sayısıyla birlikte anjiyografi raporları."""
    d = angio[mask].reset_index(drop=True)
    t = pd.DataFrame({"rapor": d["text_tr"]})
    for name, col in zip(_TER, ["max_LM", "max_LAD", "max_LCX", "max_RCA"]):
        t[name] = d[col]
    t["hastalıklı damar"] = d["n_vessels"]
    return t
print(len(angio), "anjiyografi raporu")

#### Prompt 3.1: Tek bir rapor, cümle cümle

**Amaç.** Bu promptla tek bir anjiyografi raporu dersteki işlem hattının yaptığı gibi okunur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Türkçe yazılmış bir koroner anjiyografi raporunu okuyan ve her koroner bölgenin en büyük darlığını ve hastalıklı damar sayısını veren bir program istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında tek satırlık bir tablo var: "rapor" sütunu, sentetik bir kayıt sistemi olan sanal kliniğin VC-2914 numaralı hastasının Türkçe yazılmış koroner anjiyografi raporunu tutar.  
> Görev: Raporu cümle cümle oku; bir cümle, ardından bir boşluk ve bir büyük harf ya da bir rakam gelen noktada biter. Her cümleyi Türkçe kurallarına göre küçük harfe çevir (önce I, ı; İ, i olur). "endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim" ya da "işlem" ile başlayan cümleleri atla. Öteki her cümlede, cümlenin andığı ilk arteri bul: Sol ana koroner için "sol ana koroner" ya da "lmca"; LAD için "sol ön inen" ya da "lad", dalları için "birinci diyagonal", "1. diyagonal" ya da "d1" ve "ikinci diyagonal", "2. diyagonal" ya da "d2"; sirkumfleks için "sirkumfleks", "cx" ya da "lcx", dalları için "birinci obtus", "1. obtus" ya da "om1" ve "ikinci obtus", "2. obtus" ya da "om2"; RCA için "sağ koroner" ya da "rca" ve "arka inen", "posterior desendan" ya da "pda". Kısa adlar yalnızca tam kelime olarak sayılır. Arter adı geçmeyen bir cümle, önceki cümlenin arterine aittir. Cümledeki her darlık derecesini oku: "%" ya da "yüzde " ifadesinden sonraki sayı ("%70-80" gibi bir aralık üst ucuyla, yani 80 olarak sayılır) ve "total oklüzyon", "tam tıkanıklık" ya da "to" (100), "subtotal" (99), "kritik" (95), "ileri derecede" (80), "orta derecede" (60) ile "hafif", "plak" ya da "plaklı" (30) kelimeleri. Her değeri arterinin bölgesine ver (dallar ana arterlerinin bölgesine aittir) ve her bölgenin en büyük değerini tut; değeri olmayan bölge 0 alır. Hastalıklı damar sayısı, değeri 70 ya da daha fazla olan LAD, CX ve RCA bölgelerini sayar. Her cümleyi ait olduğu arterle ve içinde bulunan değerlerle birlikte göster.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Cümleleri, LM, LAD, CX ve RCA bölgelerinin en büyük darlığını ve hastalıklı damar sayısını yazdır; hastalıklı damar sayısını "cevap" adıyla sakla.  
> Test: "Proksimal segmentinde ileri derecede darlık saptandı." cümlesi RCA'ya ait olmalı ve kateterlerin numarası olan 6 bir darlık olarak görünmemeli.  

In [ ]:
# Prompt 3.1 için veri: Promptun tarif ettiği "veri" tablosu
veri = pd.DataFrame({"rapor": angio.loc[angio["patient_id"] == "VC-2914", "text_tr"].tolist()})
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 3.1 kontrolü
teach.check_step("Prompt 3.1", "cevap", globals())

In [ ]:
#@title Prompt 3.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = pd.DataFrame({"rapor": angio.loc[angio["patient_id"] == "VC-2914", "text_tr"].tolist()})
arterler = {"LM": r"sol ana koroner|lmca", "LAD": r"sol ön inen|\blad\b", "D1": r"birinci diyagonal|1\. diyagonal|\bd1\b",
           "D2": r"ikinci diyagonal|2\. diyagonal|\bd2\b", "LCX": r"sirkumfleks|\bl?cx\b", "OM1": r"birinci obtus|1\. obtus|\bom1\b",
           "OM2": r"ikinci obtus|2\. obtus|\bom2\b", "RCA": r"sağ koroner|\brca\b", "PDA": r"arka inen|posterior desendan|\bpda\b"}
bolge = {"LM": "LM", "LAD": "LAD", "D1": "LAD", "D2": "LAD", "LCX": "LCX", "OM1": "LCX", "OM2": "LCX", "RCA": "RCA", "PDA": "RCA"}
kelimeler = {"total oklüzyon": 100, "tam tıkanıklık": 100, "to": 100, "subtotal": 99, "kritik": 95,
            "ileri derecede": 80, "orta derecede": 60, "hafif": 30, "plak": 30, "plaklı": 30}
atla = ("endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim", "işlem")


def turkce_kucuk(metin):
    return metin.replace("I", "ı").replace("İ", "i").lower()


def oku(rapor, goster=False, kucult=turkce_kucuk):
    en_buyuk = {"LM": 0, "LAD": 0, "LCX": 0, "RCA": 0}
    son = None
    for cumle in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", rapor):
        kucuk = kucult(cumle)
        if kucuk.startswith(atla):
            continue
        bulunan = sorted((m.start(), v) for v, p in arterler.items() for m in re.finditer(p, kucuk))
        if bulunan:
            son = bulunan[0][1]
        if son is None:
            continue
        degerler = [int(m.group(2) or m.group(1)) for m in re.finditer(r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?", kucuk)]
        degerler += [kelimeler[m.group(1)] for m in re.finditer(r"(?<!\w)(" + "|".join(sorted(kelimeler, key=len, reverse=True)) + r")(?!\w)", kucuk)]
        for v in degerler:
            en_buyuk[bolge[son]] = max(en_buyuk[bolge[son]], v)
        if goster:
            print(f"{son:4} {degerler!s:10} {cumle}")
    return en_buyuk

tablo = oku(veri["rapor"].iloc[0], goster=True)
cevap = sum(tablo[t] >= 70 for t in ("LAD", "LCX", "RCA"))
print(tablo, "· hastalıklı damar:", cevap)
teach.check_step("Prompt 3.1", "cevap", globals())

**Sonuç ne gösteriyor?** Rapor 3 damar hastalığını anlatır: Proksimal LAD'de yüzde 95 darlık, sirkumfleks arterde tıkanma ve proksimal RCA'da ileri derecede darlık. Bu son cümle hiçbir arterin adını anmaz ve önceki cümlenin RCA'sına aittir.

#### Prompt 3.2: A hastanesinin bütün test raporları

**Amaç.** Bu promptla işlem hattı her test raporuna uygulanır ve kayıt sistemiyle karşılaştırılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Koroner anjiyografi raporlarını okuyan bir programın ne sıklıkla kayıt sistemiyle aynı hastalıklı damar sayısını ve aynı darlık derecesini verdiğini öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; sentetik bir kayıt sistemi olan sanal kliniğin A hastanesine ait test kümesindeki Türkçe yazılmış 394 koroner anjiyografi raporunu tutar. Her satır bir hastadır: "rapor" sütunu raporu tutar; "LM", "LAD", "CX" ve "RCA" adlı dört sütun, kayıt sistemine göre sırasıyla sol ana koroner arterin, diyagonal dallarıyla sol ön inen arterin, obtus marjinal dallarıyla sirkumfleks arterin ve arka inen arterle sağ koroner arterin bölgelerindeki en büyük darlığı yüzde olarak verir. "hastalıklı damar" sütunu, yüzde 70 ya da daha fazla darlığı olan LAD, CX ve RCA bölgelerinin sayısını verir.  
> Görev: Raporu cümle cümle oku; bir cümle, ardından bir boşluk ve bir büyük harf ya da bir rakam gelen noktada biter. Her cümleyi Türkçe kurallarına göre küçük harfe çevir (önce I, ı; İ, i olur). "endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim" ya da "işlem" ile başlayan cümleleri atla. Öteki her cümlede, cümlenin andığı ilk arteri bul: Sol ana koroner için "sol ana koroner" ya da "lmca"; LAD için "sol ön inen" ya da "lad", dalları için "birinci diyagonal", "1. diyagonal" ya da "d1" ve "ikinci diyagonal", "2. diyagonal" ya da "d2"; sirkumfleks için "sirkumfleks", "cx" ya da "lcx", dalları için "birinci obtus", "1. obtus" ya da "om1" ve "ikinci obtus", "2. obtus" ya da "om2"; RCA için "sağ koroner" ya da "rca" ve "arka inen", "posterior desendan" ya da "pda". Kısa adlar yalnızca tam kelime olarak sayılır. Arter adı geçmeyen bir cümle, önceki cümlenin arterine aittir. Cümledeki her darlık derecesini oku: "%" ya da "yüzde " ifadesinden sonraki sayı ("%70-80" gibi bir aralık üst ucuyla, yani 80 olarak sayılır) ve "total oklüzyon", "tam tıkanıklık" ya da "to" (100), "subtotal" (99), "kritik" (95), "ileri derecede" (80), "orta derecede" (60) ile "hafif", "plak" ya da "plaklı" (30) kelimeleri. Her değeri arterinin bölgesine ver (dallar ana arterlerinin bölgesine aittir) ve her bölgenin en büyük değerini tut; değeri olmayan bölge 0 alır. Hastalıklı damar sayısı, değeri 70 ya da daha fazla olan LAD, CX ve RCA bölgelerini sayar. Bunu her rapora uygula. Programın ve kayıt sisteminin verdiği her en büyük darlığı beş kategoriden birine koy: 50'nin altı, 50-69, 70-89, 90-99 ve 100. Her bölge için kategorilerin uyuştuğu raporların oranını ve hastalıklı damar sayısının uyuştuğu raporların oranını hesapla. Hastalıklı damar sayısının farklı olduğu raporları göster.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Bölgelere ait dört oranı ve hastalıklı damar sayısına ait oranı yazdır; sonuncusunu üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her oran 0 ile 1 arasında olmalı ve program VC-2914 numaralı hastanın raporunda 3 hastalıklı damar bulmalı.  

In [ ]:
# Prompt 3.2 için veri: Promptun tarif ettiği "veri" tablosu
veri = _reports(angio["split"].to_numpy() == "test")
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 3.2 kontrolü
teach.check_step("Prompt 3.2", "cevap", globals())

In [ ]:
#@title Prompt 3.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _reports(angio["split"].to_numpy() == "test")
arterler = {"LM": r"sol ana koroner|lmca", "LAD": r"sol ön inen|\blad\b", "D1": r"birinci diyagonal|1\. diyagonal|\bd1\b",
           "D2": r"ikinci diyagonal|2\. diyagonal|\bd2\b", "LCX": r"sirkumfleks|\bl?cx\b", "OM1": r"birinci obtus|1\. obtus|\bom1\b",
           "OM2": r"ikinci obtus|2\. obtus|\bom2\b", "RCA": r"sağ koroner|\brca\b", "PDA": r"arka inen|posterior desendan|\bpda\b"}
bolge = {"LM": "LM", "LAD": "LAD", "D1": "LAD", "D2": "LAD", "LCX": "LCX", "OM1": "LCX", "OM2": "LCX", "RCA": "RCA", "PDA": "RCA"}
kelimeler = {"total oklüzyon": 100, "tam tıkanıklık": 100, "to": 100, "subtotal": 99, "kritik": 95,
            "ileri derecede": 80, "orta derecede": 60, "hafif": 30, "plak": 30, "plaklı": 30}
atla = ("endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim", "işlem")


def turkce_kucuk(metin):
    return metin.replace("I", "ı").replace("İ", "i").lower()


def oku(rapor, goster=False, kucult=turkce_kucuk):
    en_buyuk = {"LM": 0, "LAD": 0, "LCX": 0, "RCA": 0}
    son = None
    for cumle in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", rapor):
        kucuk = kucult(cumle)
        if kucuk.startswith(atla):
            continue
        bulunan = sorted((m.start(), v) for v, p in arterler.items() for m in re.finditer(p, kucuk))
        if bulunan:
            son = bulunan[0][1]
        if son is None:
            continue
        degerler = [int(m.group(2) or m.group(1)) for m in re.finditer(r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?", kucuk)]
        degerler += [kelimeler[m.group(1)] for m in re.finditer(r"(?<!\w)(" + "|".join(sorted(kelimeler, key=len, reverse=True)) + r")(?!\w)", kucuk)]
        for v in degerler:
            en_buyuk[bolge[son]] = max(en_buyuk[bolge[son]], v)
        if goster:
            print(f"{son:4} {degerler!s:10} {cumle}")
    return en_buyuk

bulunan = [oku(r) for r in veri["rapor"]]
kategori = lambda v: 0 if v < 50 else 1 if v < 70 else 2 if v < 90 else 3 if v < 100 else 4
for ad, anahtar in zip(["LM", "LAD", "CX", "RCA"], ["LM", "LAD", "LCX", "RCA"]):
    uyum = np.mean([kategori(f[anahtar]) == kategori(v) for f, v in zip(bulunan, veri[ad])])
    print(ad, round(float(uyum), 3))
bulunan_sayi = np.array([sum(f[t] >= 70 for t in ("LAD", "LCX", "RCA")) for f in bulunan])
cevap = round(float(np.mean(bulunan_sayi == veri["hastalıklı damar"].to_numpy())), 3)
print("Hastalıklı damar sayısı doğru:", cevap)
yanlis = np.where(bulunan_sayi != veri["hastalıklı damar"].to_numpy())[0]
for i in yanlis[:3]:
    print("—", veri["rapor"].iloc[i][:300], "...")
teach.check_step("Prompt 3.2", "cevap", globals())

**Sonuç ne gösteriyor?** Hastalıklı damar sayısı test raporlarının 0,959 kadarında doğrudur. Yanlış olduğu raporlar bir arterin adını "Sirkumflex arter" ya da "Sag koroner arter" gibi Türkçe harfler olmadan ya da bir yazım hatasıyla yazar: İşlem hattı arteri tanımaz ve darlığını önceki cümlenin arterine verir.

#### Prompt 3.3: B hastanesinin büyük harfli raporları

**Amaç.** Bu promptla aynı işlem hattı, büyük harflerle ve kısa bir üslupla yazılmış başka bir hastanenin raporlarına uygulanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Bir hastanenin koroner anjiyografi raporlarını okuyan bir programın, büyük harflerle ve kısa bir üslupla yazılmış başka bir hastanenin raporlarını da okuyup okumadığını öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; sentetik bir kayıt sistemi olan sanal kliniğin B hastanesinden gelen ve Türkçe yazılmış 373 koroner anjiyografi raporunu tutar; raporlar büyük harflerle yazılmıştır. Her satır bir hastadır: "rapor" sütunu raporu tutar; "LM", "LAD", "CX" ve "RCA" adlı dört sütun, kayıt sistemine göre sırasıyla sol ana koroner arterin, diyagonal dallarıyla sol ön inen arterin, obtus marjinal dallarıyla sirkumfleks arterin ve arka inen arterle sağ koroner arterin bölgelerindeki en büyük darlığı yüzde olarak verir. "hastalıklı damar" sütunu, yüzde 70 ya da daha fazla darlığı olan LAD, CX ve RCA bölgelerinin sayısını verir.  
> Görev: Raporu cümle cümle oku; bir cümle, ardından bir boşluk ve bir büyük harf ya da bir rakam gelen noktada biter. Her cümleyi Türkçe kurallarına göre küçük harfe çevir (önce I, ı; İ, i olur). "endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim" ya da "işlem" ile başlayan cümleleri atla. Öteki her cümlede, cümlenin andığı ilk arteri bul: Sol ana koroner için "sol ana koroner" ya da "lmca"; LAD için "sol ön inen" ya da "lad", dalları için "birinci diyagonal", "1. diyagonal" ya da "d1" ve "ikinci diyagonal", "2. diyagonal" ya da "d2"; sirkumfleks için "sirkumfleks", "cx" ya da "lcx", dalları için "birinci obtus", "1. obtus" ya da "om1" ve "ikinci obtus", "2. obtus" ya da "om2"; RCA için "sağ koroner" ya da "rca" ve "arka inen", "posterior desendan" ya da "pda". Kısa adlar yalnızca tam kelime olarak sayılır. Arter adı geçmeyen bir cümle, önceki cümlenin arterine aittir. Cümledeki her darlık derecesini oku: "%" ya da "yüzde " ifadesinden sonraki sayı ("%70-80" gibi bir aralık üst ucuyla, yani 80 olarak sayılır) ve "total oklüzyon", "tam tıkanıklık" ya da "to" (100), "subtotal" (99), "kritik" (95), "ileri derecede" (80), "orta derecede" (60) ile "hafif", "plak" ya da "plaklı" (30) kelimeleri. Her değeri arterinin bölgesine ver (dallar ana arterlerinin bölgesine aittir) ve her bölgenin en büyük değerini tut; değeri olmayan bölge 0 alır. Hastalıklı damar sayısı, değeri 70 ya da daha fazla olan LAD, CX ve RCA bölgelerini sayar. Bunu her rapora uygula ve hastalıklı damar sayısının kayıt sistemiyle uyuştuğu raporların oranını hesapla. Ardından Türkçe küçük harf işlemi yerine programlama dilinin olağan küçük harf işlemini kullanarak tekrarla ve oranı yeniden hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İki oranı yazdır; Türkçe küçük harf işlemiyle bulunan oranı üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Türkçe küçük harf işlemiyle bulunan oran, olağan küçük harf işlemiyle bulunandan yüksek olmalı.  

In [ ]:
# Prompt 3.3 için veri: Promptun tarif ettiği "veri" tablosu
veri = _reports(angio["split"].to_numpy() == "external")
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 3.3 kontrolü
teach.check_step("Prompt 3.3", "cevap", globals())

In [ ]:
#@title Prompt 3.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _reports(angio["split"].to_numpy() == "external")
arterler = {"LM": r"sol ana koroner|lmca", "LAD": r"sol ön inen|\blad\b", "D1": r"birinci diyagonal|1\. diyagonal|\bd1\b",
           "D2": r"ikinci diyagonal|2\. diyagonal|\bd2\b", "LCX": r"sirkumfleks|\bl?cx\b", "OM1": r"birinci obtus|1\. obtus|\bom1\b",
           "OM2": r"ikinci obtus|2\. obtus|\bom2\b", "RCA": r"sağ koroner|\brca\b", "PDA": r"arka inen|posterior desendan|\bpda\b"}
bolge = {"LM": "LM", "LAD": "LAD", "D1": "LAD", "D2": "LAD", "LCX": "LCX", "OM1": "LCX", "OM2": "LCX", "RCA": "RCA", "PDA": "RCA"}
kelimeler = {"total oklüzyon": 100, "tam tıkanıklık": 100, "to": 100, "subtotal": 99, "kritik": 95,
            "ileri derecede": 80, "orta derecede": 60, "hafif": 30, "plak": 30, "plaklı": 30}
atla = ("endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim", "işlem")


def turkce_kucuk(metin):
    return metin.replace("I", "ı").replace("İ", "i").lower()


def oku(rapor, goster=False, kucult=turkce_kucuk):
    en_buyuk = {"LM": 0, "LAD": 0, "LCX": 0, "RCA": 0}
    son = None
    for cumle in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", rapor):
        kucuk = kucult(cumle)
        if kucuk.startswith(atla):
            continue
        bulunan = sorted((m.start(), v) for v, p in arterler.items() for m in re.finditer(p, kucuk))
        if bulunan:
            son = bulunan[0][1]
        if son is None:
            continue
        degerler = [int(m.group(2) or m.group(1)) for m in re.finditer(r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?", kucuk)]
        degerler += [kelimeler[m.group(1)] for m in re.finditer(r"(?<!\w)(" + "|".join(sorted(kelimeler, key=len, reverse=True)) + r")(?!\w)", kucuk)]
        for v in degerler:
            en_buyuk[bolge[son]] = max(en_buyuk[bolge[son]], v)
        if goster:
            print(f"{son:4} {degerler!s:10} {cumle}")
    return en_buyuk

gercek = veri["hastalıklı damar"].to_numpy()


def damar_sayisi(kucult):
    return np.array([sum(f[t] >= 70 for t in ("LAD", "LCX", "RCA")) for f in (oku(r, kucult=kucult) for r in veri["rapor"])])


cevap = round(float(np.mean(damar_sayisi(turkce_kucuk) == gercek)), 3)
olagan = round(float(np.mean(damar_sayisi(str.lower) == gercek)), 3)
print("Olağan küçük harf:", olagan)
print("Türkçe küçük harf:", cevap)
teach.check_step("Prompt 3.3", "cevap", globals())

**Sonuç ne gösteriyor?** Türkçe küçük harf işlemiyle hastalıklı damar sayısı, kısa üslubunda yanlış yazılmış arter adı bulunmayan B hastanesi raporlarının 1,0 kadarında doğrudur. Olağan küçük harf işleminde PROKSİMAL, DİSTAL ve MİD kelimelerindeki her İ noktalı bir i'ye dönüşür ve ardından gelen kelimeler artık eşleşmez; ı yerine i'ye dönüşen her büyük I da aynı sorunu yaratır.

**«Durak 3 · Prompt ile uygulama» tamamlandı.** Sırada hemen aşağıdaki «Durak 3 · Hazır uygulama» bölümü var. Durağı burada bırakacaksanız [ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#ai-pair-programming-part-6-reviewing-a-text-pipeline) ve okumaya «YZ asistanıyla çalışmak, 6. bölüm: Bir metin işlem hattını denetlemek» bölümünden devam edin.

### Durak 3 · Hazır uygulama

Bu bölüm iki hastanenin anjiyografi raporlarını cümle cümle okur, her cümlenin arterini ve bütün darlık derecelerini bulur, her bölgenin en büyük değerini tutar; ardından kategorileri ve hastalıklı damar sayısını kayıt sistemiyle karşılaştırır.

Hücreleri yukarıdan aşağıya çalıştırın. *Python notu* hücreleri kodu açıklar, *Gözlemle ve düşün* hücreleri ise sonuçlar üzerine sorular sorar.

**Başlangıç hücresi.** Aşağıdaki hücre, «Durak 3 · Hazır uygulama» bölümü için kurulumu, sonraki hücrelerin kullandığı veri çağrılarını ve önceki hücrelerin kodlarını yineler; böylece kodlar hata vermeden çalışır. «Durak 1 · Hazır uygulama» bölümünde yazılan fonksiyonlar, onları aynen saklayan ders paketinden alınır; bu fonksiyonların kodları o bölümdedir. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Durak 3 · Hazır uygulama» bölümü
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# önceki bölümlerin fonksiyonları: Ders paketi onları aynen saklar
from vclinic.week06 import tr_lower

# alıştırma promptlarının tabloları (etkileşimli labın F bölümü)
from vclinic.week06 import alistirma_tablosu

# önceki hücrelerden yinelenen kod: Aşağıdaki hücrelerin kullandığı veriler ve kodlar
import re

#### Anjiyografi raporları: Darlık ve damar

*Ders sayfasında: [Anjiyografi raporları: Darlık derecesi ve damar](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#angiography-reports-the-degree-of-stenosis-and-the-vessel)*

İşlem hattı bir raporu cümle cümle okur; endikasyon, öykü, ventrikülografi ve girişim cümlelerini atlar, her cümlenin arterini bulur, her darlık derecesini sayı, aralık ya da kelime olarak okur ve her bölgenin en büyük değerini tutar.

**Formüllerden koda.** Bölgeler: `TERRITORY`; dallar ana arterlerinin bölgesine gider. Anlamlı darlık: LM'de 50 ya da üstü, diğerlerinde 70 ya da üstü. Kategoriler: `category`. Hastalıklı damar sayısı: `n_vessels`, yani değeri 70 ya da daha fazla olan LAD, LCX ve RCA bölgeleri. Bir aralık üst ucuyla okunur: `stenoses` içindeki `m.group(2) or m.group(1)`.

In [ ]:
angio = reports.angiography_archive()
TERRITORIES = ["LM", "LAD", "LCX", "RCA"]
VESSELS = {"LM": r"sol ana koroner|lmca", "LAD": r"sol ön inen|\blad\b", "D1": r"birinci diyagonal|1\. diyagonal|\bd1\b",
           "D2": r"ikinci diyagonal|2\. diyagonal|\bd2\b", "LCX": r"sirkumfleks|\bl?cx\b", "OM1": r"birinci obtus|1\. obtus|\bom1\b",
           "OM2": r"ikinci obtus|2\. obtus|\bom2\b", "RCA": r"sağ koroner|\brca\b", "PDA": r"arka inen|posterior desendan|\bpda\b"}
TERRITORY = {"LM": "LM", "LAD": "LAD", "D1": "LAD", "D2": "LAD", "LCX": "LCX", "OM1": "LCX", "OM2": "LCX", "RCA": "RCA", "PDA": "RCA"}
WORDS = {"total oklüzyon": 100, "tam tıkanıklık": 100, "to": 100, "subtotal": 99, "kritik": 95,
         "ileri derecede": 80, "orta derecede": 60, "hafif": 30, "plak": 30, "plaklı": 30}
NUMBER = r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?"
WORD = r"(?<!\w)(" + "|".join(sorted(WORDS, key=len, reverse=True)) + r")(?!\w)"
SKIP = ("endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim", "işlem")


def stenoses(sentence):
    """Bir cümledeki her darlık derecesi, yüzde olarak: Sayılar (bir aralık üst ucuyla sayılır) ve kelimeler."""
    values = [int(m.group(2) or m.group(1)) for m in re.finditer(NUMBER, sentence)]
    values += [WORDS[m.group(1)] for m in re.finditer(WORD, sentence)]
    return values


def angio_table(text, coreference=True):
    """Bir rapordaki her bölgenin en büyük darlığı. Cümle, büyük harften ya da rakamdan önceki noktada biter; arter adı geçmeyen bir cümle, önceki cümlenin arterine aittir."""
    best = {t: 0 for t in TERRITORIES}
    last = None
    for sentence in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", text):
        sentence = tr_lower(sentence)
        if sentence.startswith(SKIP):
            continue
        found = sorted((m.start(), v) for v, pattern in VESSELS.items() for m in re.finditer(pattern, sentence))
        if found:
            last = found[0][1]
        elif not coreference:
            continue
        if last is None:
            continue
        for value in stenoses(sentence):
            best[TERRITORY[last]] = max(best[TERRITORY[last]], value)
    return best


def category(value):
    """50'nin altı 0, 50-69 için 1, 70-89 için 2, 90-99 için 3 ve tıkanma için 4."""
    return 0 if value < 50 else 1 if value < 70 else 2 if value < 90 else 3 if value < 100 else 4


def n_vessels(table):
    """Hastalıklı damar sayısı: Yüzde 70 ya da daha fazla darlığı olan LAD, LCX ve RCA bölgeleri."""
    return sum(table[t] >= 70 for t in ("LAD", "LCX", "RCA"))

angio_case = angio.set_index("patient_id").loc["VC-2914"]
print(angio_case["text_tr"])
print("İşlem hattı:", angio_table(angio_case["text_tr"]), "· kayıt sistemi:", {t: int(angio_case[f"max_{t}"]) for t in TERRITORIES})
for name, part in (("test", "test"), ("dış", "external")):
    d = angio[angio["split"] == part]
    found = [angio_table(t) for t in d["text_tr"]]
    agree = {t: round(float(np.mean([category(f[t]) == c for f, c in zip(found, d[f"cat_{t}"])])), 3) for t in TERRITORIES}
    right = round(float(np.mean([n_vessels(f) == n for f, n in zip(found, d["n_vessels"])])), 3)
    print(name, agree, "· hastalıklı damar sayısı doğru:", right)

**Python notu.** `re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", text)` raporu ardından bir boşluk ve bir büyük harf ya da rakam gelen noktadan böler; böylece "1. diyagonal" tek parça kalır. `sentence.startswith(SKIP)` bir demet alır ve cümle demetteki kelimelerden biriyle başlıyorsa doğru olur. `sorted(...)[0][1]` cümlenin ilk andığı arteri verir. `last`, hiçbir arter anmayan cümleler için önceki cümlenin arterini tutar.

In [ ]:
d = angio[angio["split"] == "test"]
wrong = [(r["patient_id"], angio_table(r["text_tr"]), {t: int(r[f"max_{t}"]) for t in TERRITORIES}) for _, r in d.iterrows()
         if any(category(angio_table(r["text_tr"])[t]) != r[f"cat_{t}"] for t in TERRITORIES)]
print(len(wrong), "bölgesi yanlış olan test raporu; ilki:")
print(wrong[0])
print(angio.set_index("patient_id").loc[wrong[0][0], "text_tr"])

**Gözlemle ve düşün.** İşlem hattının yanıldığı ilk raporu okuyun. Onu hangi cümle yanılttı ve işlem hattında neyi değiştirirdiniz? Değişikliğiniz görmediğiniz raporlarda da işe yarar mıydı?

**Alıştırma I.** Arter adı geçmeyen cümleler atlandığında, yani `angio_table(text, coreference=False)` ile, A hastanesinin test raporlarının ne kadarında hastalıklı damar sayısı doğru bulunur? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("I")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma I", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
d = angio[angio["split"] == "test"]
cevap = round(float(np.mean([n_vessels(angio_table(t, coreference=False)) == n for t, n in zip(d["text_tr"], d["n_vessels"])])), 3)
teach.check("Alıştırma I", cevap)

**3. durak tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#ai-pair-programming-part-6-reviewing-a-text-pipeline) ve okumaya «YZ asistanıyla çalışmak, 6. bölüm: Bir metin işlem hattını denetlemek» bölümünden devam edin. Haftalık proje bu Jupyter Notebook'un sonundadır; ders sayfasındaki «Haftalık proje» bölümü projeyi anlatır.

## Özetler, klinik çalışma uygunluğu ve bir asistanın denetimi

*Ders sayfasında: [Radyoloji raporu otomatik özetleme](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#summaries-and-large-language-models-as-clinical-assistants)*

Vakanın, bir YZ asistanının üslubuyla yazılmış üç kısa özeti verilmiştir. Dersteki denetim her birini epikrizle karşılaştırır: Her sayı raporda geçmeli, her ilaç raporun ilaçlarından biri olmalı ve özetin var dediği her tanı rapor tarafından, var durumunda bir geçiş, bir ilaç ya da bir laboratuvar değeriyle desteklenmelidir.

**Formüllerden koda.** Bir metnin sayıları: `re.findall(r"\d+(?:,\d+)?", ...)`. Arşivin ilaçları: `DRUG_NAMES`, yani her ilaç varlığının ilk kelimesi. Raporun desteklediği tanılar: `rule_labels(report)`; özetin var dediği tanılar: `assertion_labels(summary)`.

Bu bölüm uygulama duraklarından sonra gelir ve kendi başına çalışır: Başlangıç hücresi, hücrelerin kullandığı kurulumu ve veri çağrılarını yineler.

**Başlangıç hücresi.** Aşağıdaki hücre, «Özetler, klinik çalışma uygunluğu ve bir asistanın denetimi» bölümü için kurulumu, sonraki hücrelerin kullandığı veri çağrılarını ve önceki hücrelerin kodlarını yineler; böylece kodlar hata vermeden çalışır. «Durak 1 · Hazır uygulama» ve «Durak 3 · Hazır uygulama» bölümünde yazılan fonksiyonlar, onları aynen saklayan ders paketinden alınır; bu fonksiyonların kodları o bölümdedir. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: «Özetler, klinik çalışma uygunluğu ve bir asistanın denetimi» bölümü
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

# önceki bölümlerin fonksiyonları: Ders paketi onları aynen saklar
from vclinic.week06 import tr_lower, se_ppv_f1, next_other, lab_value, n_vessels

# alıştırma promptlarının tabloları (etkileşimli labın F bölümü)
from vclinic.week06 import alistirma_tablosu

# önceki hücrelerden yinelenen kod: Aşağıdaki hücrelerin kullandığı veriler ve kodlar
import re
archive = reports.archive()
LABELS = list(reports.LABELS)
texts = archive["text_tr"].to_numpy()
y = archive[LABELS].to_numpy()
split = archive["split"].to_numpy()
tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")
case = int(np.where(archive["patient_id"] == "VC-2444")[0][0])
TERMS = {
    "HF": ["kalp yetersizliği", "kalp yetmezliği", "konjestif kalp yetmezliği", "ky", "kky"],
    "AF": ["atriyal fibrilasyon", "af", "afib"],
    "CAD": ["koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü", "akut koroner sendrom"],
    "HT": ["hipertansiyon", "ht"],
    "DM": ["diyabet", "diyabetes mellitus", "dm"],
    "CKD": ["kronik böbrek hastalığı", "kronik böbrek yetmezliği", "kbh", "kby"],
}
PATTERNS = {label: re.compile(r"(?<!\w)(" + "|".join(sorted(map(re.escape, terms), key=len, reverse=True)) + r")(?!\w)")
            for label, terms in TERMS.items()}
def find_terms(text):
    """Bir tanının metindeki her geçişi: (başlangıç, bitiş, tanı), metindeki sırasıyla."""
    low = tr_lower(text)
    found = [(m.start(), m.end(), label) for label, pattern in PATTERNS.items() for m in pattern.finditer(low)]
    return sorted(found)
NEGATION = ["yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)"]
POSSIBLE = ["şüphe", "olası", "ön tanı"]
FAMILY = ["babası", "annesi", "kardeşi", "babada", "annede", "soygeçmiş"]
def assertion(text, start, end, next_start, window=None):
    """Bir geçişin durumu: var (present), yok (negated), olası (possible) ya da aile öyküsü (family).

    Geçişten sonraki kapsam, cümle sonunda ya da başka bir tanının bir sonraki geçişinin başladığı yerde (next_start) biter; window verilirse bunun yerine o kadar karakter sonra biter."""
    low = tr_lower(text)
    if window is None:
        stop = min(x for x in (low.find(".", end), low.find(";", end), next_start, len(low)) if x >= end)
    else:
        stop = min(len(low), end + window)
    after = low[end:stop]
    sentence_start = max(low.rfind(". ", 0, start), low.rfind(": ", 0, start)) + 1
    before = low[sentence_start:start]
    if any(w in before for w in FAMILY):
        return "family"
    if any(w in after for w in NEGATION):
        return "negated"
    if any(w in after for w in POSSIBLE):
        return "possible"
    return "present"
def assertion_labels(text, window=None):
    """Geçişlerinden en az biri var durumunda olan tanı var sayılır."""
    mentions = find_terms(text)
    present = set()
    for i, (start, end, label) in enumerate(mentions):
        if assertion(text, start, end, next_other(mentions, i, text), window) == "present":
            present.add(label)
    return [int(label in present) for label in LABELS]
EVIDENCE = {
    "HF": ["furosemid", "sakubitril"],
    "AF": ["apiksaban", "rivaroksaban", "edoksaban", "dabigatran"],
    "CAD": ["klopidogrel", "tikagrelor", "pkg", "kabg", "stent"],
    "HT": ["amlodipin", "hidroklorotiyazid"],
    "DM": ["metformin", "gliklazid", "insülin", "sitagliptin"],
}
def rule_labels(text, window=None):
    """Durumlarıyla birlikte geçişler, ilaçlar ve iki laboratuvar değeri."""
    labels = assertion_labels(text, window)
    low = tr_lower(text)
    for k, label in enumerate(LABELS):
        if any(re.search(r"(?<!\w)" + re.escape(w), low) for w in EVIDENCE.get(label, [])):
            labels[k] = 1
    egfr, hba1c = lab_value(text, "egfr"), lab_value(text, "hba1c")
    if egfr is not None and egfr < 60:
        labels[LABELS.index("CKD")] = 1
    if hba1c is not None and hba1c >= 6.5:
        labels[LABELS.index("DM")] = 1
    return labels
angio = reports.angiography_archive()
TERRITORIES = ["LM", "LAD", "LCX", "RCA"]
VESSELS = {"LM": r"sol ana koroner|lmca", "LAD": r"sol ön inen|\blad\b", "D1": r"birinci diyagonal|1\. diyagonal|\bd1\b",
           "D2": r"ikinci diyagonal|2\. diyagonal|\bd2\b", "LCX": r"sirkumfleks|\bl?cx\b", "OM1": r"birinci obtus|1\. obtus|\bom1\b",
           "OM2": r"ikinci obtus|2\. obtus|\bom2\b", "RCA": r"sağ koroner|\brca\b", "PDA": r"arka inen|posterior desendan|\bpda\b"}
TERRITORY = {"LM": "LM", "LAD": "LAD", "D1": "LAD", "D2": "LAD", "LCX": "LCX", "OM1": "LCX", "OM2": "LCX", "RCA": "RCA", "PDA": "RCA"}
WORDS = {"total oklüzyon": 100, "tam tıkanıklık": 100, "to": 100, "subtotal": 99, "kritik": 95,
         "ileri derecede": 80, "orta derecede": 60, "hafif": 30, "plak": 30, "plaklı": 30}
NUMBER = r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?"
WORD = r"(?<!\w)(" + "|".join(sorted(WORDS, key=len, reverse=True)) + r")(?!\w)"
SKIP = ("endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim", "işlem")
def stenoses(sentence):
    """Bir cümledeki her darlık derecesi, yüzde olarak: Sayılar (bir aralık üst ucuyla sayılır) ve kelimeler."""
    values = [int(m.group(2) or m.group(1)) for m in re.finditer(NUMBER, sentence)]
    values += [WORDS[m.group(1)] for m in re.finditer(WORD, sentence)]
    return values
def angio_table(text, coreference=True):
    """Bir rapordaki her bölgenin en büyük darlığı. Cümle, büyük harften ya da rakamdan önceki noktada biter; arter adı geçmeyen bir cümle, önceki cümlenin arterine aittir."""
    best = {t: 0 for t in TERRITORIES}
    last = None
    for sentence in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", text):
        sentence = tr_lower(sentence)
        if sentence.startswith(SKIP):
            continue
        found = sorted((m.start(), v) for v, pattern in VESSELS.items() for m in re.finditer(pattern, sentence))
        if found:
            last = found[0][1]
        elif not coreference:
            continue
        if last is None:
            continue
        for value in stenoses(sentence):
            best[TERRITORY[last]] = max(best[TERRITORY[last]], value)
    return best

Üç özet şunları söyler: (1) Raporun yedi ilacıyla taburcu edilen, korunmuş ejeksiyon fraksiyonlu kalp yetersizliği (EF %55) olan ve eGFR değeri 59 olan 83 yaşında bir kadın; (2) aynı hasta, atriyal fibrilasyonla birlikte ve furosemid, spironolakton, apiksaban ve amlodipinle; (3) EF değeri %35 olan, koroner arter hastalığı nedeniyle izlenen, eGFR değeri 59 olan ve furosemid ile metoprolol kullanan bir kalp yetersizliği hastası.

**Ders sayfasında:** [Çözümlü örnek: Vakanın bir özetini denetlemek](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#worked-example-checking-a-summary-of-the-case). Aşağıdaki hücre aynı özeti ve iki özeti daha vakanın epikrizine karşı denetler.

In [ ]:
summary_1 = "83 yaşında kadın hasta. Korunmuş ejeksiyon fraksiyonlu kalp yetersizliği (EF %55), eGFR 59. Taburcu tedavisi: furosemid, spironolakton, dapagliflozin, bisoprolol, perindopril, amlodipin, rosuvastatin."
summary_2 = "83 yaşında kadın hasta; korunmuş ejeksiyon fraksiyonlu kalp yetersizliği (EF %55) ve atriyal fibrilasyon; eGFR 59; furosemid, spironolakton, apiksaban ve amlodipin ile taburcu edildi."
summary_3 = "83 yaşında kadın hasta; kalp yetersizliği, EF %35; koroner arter hastalığı nedeniyle izleniyor; eGFR 59. Taburcu tedavisi: furosemid, metoprolol."

DRUG_NAMES = sorted({tr_lower(e["text"]).split()[0] for es in archive["entities_tr"] for e in es if e["label"] == "DRUG"}, key=len, reverse=True)


def check_summary(summary, report):
    """Bir özetin söylediği ve raporun desteklemediği şeyler: Sayılar, ilaçlar ve var denen tanılar."""
    low_s, low_r = tr_lower(summary), tr_lower(report)
    problems = []
    numbers_r = set(re.findall(r"\d+(?:,\d+)?", low_r))
    problems += [f"sayı {n}" for n in re.findall(r"\d+(?:,\d+)?", low_s) if n not in numbers_r]
    for d in DRUG_NAMES:
        pattern = r"(?<!\w)" + re.escape(d) + r"(?!\w)"
        if re.search(pattern, low_s) and not re.search(pattern, low_r):
            problems.append(f"ilaç {d}")
    supported = rule_labels(report)
    claimed = assertion_labels(summary)
    problems += [f"tanı {LABELS[k]}" for k in range(len(LABELS)) if claimed[k] and not supported[k]]
    return problems

for name, summary in (("1", summary_1), ("2", summary_2), ("3", summary_3)):
    print(name, check_summary(summary, texts[case]) or "sorun bulunmadı")

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c234)*

**Python notu.** `check_summary` bulgularını `+=` ve `.append` ile `problems` listesinde toplar ve listeyi döndürür; boş bir liste yanlış sayıldığı için `... or "sorun bulunmadı"` bunun yerine metni yazdırır. Raporun sayıları, `in` işleminin hızlı çalıştığı bir *küme* içinde (`set(...)`) tutulur. Tanılar «Durak 1 · Hazır uygulama» bölümündeki iki fonksiyonu kullanır: Raporun ilaçları ve değerleriyle desteklediği tanılar için `rule_labels`, özetin var dediği tanılar için `assertion_labels`.

**Gözlemle ve düşün.** Denetim 2. özetteki yanlış atriyal fibrilasyonu ve apiksabanı buldu. Neyi denetlemez? "Korunmuş ejeksiyon fraksiyonu" ifadesini, yanlış bir tarihi ve bir tanıyı atlayan bir özeti düşünün.

**Alıştırma J.** `check_summary` 3. özette kaç sorun bulur?

In [ ]:
veri = alistirma_tablosu("J", summary_3=summary_3)   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma J", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = len(check_summary(summary_3, texts[case]))
teach.check("Alıştırma J", cevap)

**Bir görüntüleme raporunun sonucu.** Ekokardiyografiden koroner anjiyografiye kadar görüntüleme raporları bulguları özetleyen bir sonuçla biter. Aşağıdaki hücre bir anjiyografi raporunun sonucunu «Durak 3 · Hazır uygulama» bölümündeki işlem hattının tablosundan yazar; bu, her sayısı rapora kadar izlenebilen, çıkarımsal bir özettir. Hücre bu sonucu kayıt sistemindeki değerlerin verdiği sonuçla karşılaştırır.

In [ ]:
def conclusion(table):
    """Bir anjiyografi raporunun sonucu, işlem hattının tablosundan: Hastalıklı damar sayısı ve anlamlı her darlık."""
    n = n_vessels(table)
    significant = [f"{t} %{v}" if v < 100 else f"{t} tıkalı" for t, v in table.items() if v >= (50 if t == "LM" else 70)]
    if not significant:
        return "Anlamlı darlık yok."
    head = f"{n} damar hastalığı" if n else ""
    if table["LM"] >= 50:
        head = "Sol ana koroner hastalığı" + (", " + head if head else "")
    return head + ": " + ", ".join(significant) + "."

for _, r in angio[angio["split"] == "test"].head(3).iterrows():
    print(r["patient_id"], "·", conclusion(angio_table(r["text_tr"])))
for name, part in (("test", "test"), ("dış", "external")):
    d = angio[angio["split"] == part]
    same = np.mean([conclusion(angio_table(t)) == conclusion({k: int(r[f"max_{k}"]) for k in TERRITORIES})
                    for t, (_, r) in zip(d["text_tr"], d.iterrows())])
    print(name, "kayıt sistemininkiyle aynı sonuçlar:", round(float(same), 3))

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c241)*

**Klinik çalışma uygunluğu.** *Ders sayfasında: [Klinik trial uygunluk tespiti](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#clinical-trial-eligibility-detection)*

Bu alıştırma için kurgulanmış eğitim çalışması, 18-85 yaş arasında, kalp yetersizliği olan ve EF'si yüzde 40 ya da daha düşük hastaları arar. 30 mL/dk/1,73 m²'nin altındaki bir eGFR'yi ve akut koroner sendromu dışlar. Hücre beş ölçütü bu haftanın kurallarıyla her epikrizden okur ve sonucu kayıt sistemiyle, hem ölçüt ölçüt hem de çalışmanın bütünü için karşılaştırır.

**Formüllerden koda.** Bir epikrizin ölçütleri: `screen(text)`, beş evet-hayır değerinden oluşan bir sözlük. Uygun: Bütün dahil etme ölçütleri sağlanmış ve hiçbir dışlama ölçütü yok, `eligible(c)`. Akut koroner sendrom durum kuralına göre var durumunda olmalıdır: `acs_present`.

In [ ]:
from vclinic import make_cohort

registry = make_cohort(len(archive)).set_index("patient_id")
AGE = re.compile(r"(\d+)\s*(?:yaşında|y\b)")
EF = re.compile(r"(?:ejeksiyon fraksiyonu|ef)\s*%\s*(\d+)")
ACS_TEXT = re.compile(r"akut koroner sendrom|(?<!\w)aks(?!\w)")


def acs_present(text):
    """Akut koroner sendromun bir geçişi durum kuralına göre var durumundaysa, yani olumsuzlanmamışsa doğru."""
    return any(assertion(text, m.start(), m.end(), len(text)) == "present" for m in ACS_TEXT.finditer(tr_lower(text)))


def screen(text):
    """Eğitimin çalışmasının beş ölçütü, bir epikriz için: Kalp yetersizliği, yüzde 40 ya da daha düşük EF, 18-85 yaş (dahil etme); 30'un altında eGFR, akut koroner sendrom (dışlama)."""
    low = tr_lower(text)
    age, ef, egfr = AGE.search(low), EF.search(low), lab_value(text, "egfr")
    return {"HF": dict(zip(LABELS, rule_labels(text)))["HF"] == 1,
            "EF≤40": ef is not None and int(ef.group(1)) <= 40,
            "18-85": age is not None and 18 <= int(age.group(1)) <= 85,
            "eGFR<30": egfr is not None and egfr < 30,
            "ACS": acs_present(text)}


def eligible(c):
    """Uygun: Bütün dahil etme ölçütleri karşılanmış ve hiçbir dışlama ölçütü yok."""
    return c["HF"] and c["EF≤40"] and c["18-85"] and not c["eGFR<30"] and not c["ACS"]

reg = registry.loc[archive["patient_id"]]
truth_criteria = {"HF": y[:, LABELS.index("HF")] == 1, "EF≤40": reg["lvef"].to_numpy() <= 40, "18-85": reg["age"].between(18, 85).to_numpy(),
                  "eGFR<30": reg["egfr"].to_numpy() < 30, "ACS": reg["main_diagnosis"].to_numpy() == "ACS"}
truth_eligible = (truth_criteria["HF"] & truth_criteria["EF≤40"] & truth_criteria["18-85"] & ~truth_criteria["eGFR<30"] & ~truth_criteria["ACS"])
plain_acs = np.array([ACS_TEXT.search(tr_lower(t)) is not None for t in texts])
for name, mask in (("test, A hastanesi", te), ("dış, B hastanesi", ex)):
    criteria = [screen(t) for t in texts[mask]]
    agree = {c: round(float(np.mean([x[c] for x in criteria] == truth_criteria[c][mask])), 3) for c in truth_criteria}
    found = np.array([eligible(x) for x in criteria])
    se, ppv, f1 = se_ppv_f1(truth_eligible[mask], found)
    print(name, "· ölçüte göre uyum:", agree)
    print("   kayıt sisteminde uygun:", int(truth_eligible[mask].sum()), "· bulunan:", int(found.sum()),
          "· duyarlılık", round(se, 3), "· PPV", round(ppv, 3),
          "· ACS için düz arama şu kadar hastada yanlış:", int((plain_acs[mask] != truth_criteria["ACS"][mask]).sum()))

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#nbref-c243)*

**Python notu.** `screen` bir sözlük döndürdüğü için `[x[c] for x in criteria]` her epikrizden bir ölçütü alır. `reg["age"].between(18, 85)`, uçlar dahil 18 ile 85 arasındaki yaşlar için doğrudur. `~` bir dizideki True değerlerini False'a, False değerlerini True'ya çevirir; bu yüzden `~truth_criteria["ACS"]` akut koroner sendromu olmayan hastaları işaretler. `plain_acs` kelimeleri durum kuralı olmadan, tek başına arar.

**Gözlemle ve düşün.** "akut koroner sendrom" için yapılan düz bir arama birçok hastayı yanlış dışlardı. Bu epikrizlerden birini yazdırın. Hata neden bu çalışmanın hiçbir kararını değiştirmedi ve hangi çalışmada uygun hastaların kaybına yol açardı?

**«Özetler, klinik çalışma uygunluğu ve bir asistanın denetimi» bölümü tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#summaries-and-large-language-models-as-clinical-assistants).

## Haftalık proje

*Ders sayfasında: [Haftalık proje](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#weekly-project) · [6. haftanın README sayfası, Haftalık proje](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-06/README.md#haftal%C4%B1k-proje)*

**Proje.** İki hastanenin koroner anjiyografi raporları için bir işlem hattı: Dört bölgenin en büyük darlığı her rapordan okunur, dersin beş kategorisine konur ve sol ana koroner arter hastalığı, hastalıklı damar sayısı, işlem hattının yanlış okuduğu raporlar ve aynı hastaların epikrizleriyle birlikte kayıt sistemiyle karşılaştırılır.

Bu bölüm kendi başına çalışır: Başlangıç hücresi, projenin verilerini yükler ve adımların tablolarını hazırlar. Proje, her adım için bir prompt olmak üzere aşağıdaki promptlarla yapılabilir. Her adım duraklardaki promptlar gibi ilerler; ancak sonuçlar rapora girdiği için kontrol hücresi ve hazır kod yoktur. Bir promptun testi sağlanmadan sonraki prompta geçilmez ve her prompt, YZ kullanım günlüğüne yazılır.

**Başlangıç hücresi.** Aşağıdaki hücre, haftalık proje için kurulumu ve bu bölümün veri çağrılarını yineler; böylece kodlar hata vermeden çalışır. Jupyter Notebook'un önceki bölümlerini bu oturumda çalıştırdıysanız bu hücreyi çalıştırmanız gerekmeyebilir; emin değilseniz çalıştırın, yeniden çalıştırmanın bir zararı yoktur.

In [ ]:
# Başlangıç hücresi: haftalık proje
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # vclinic ders paketi: Ders klasöründeki kopya ya da yoksa GitHub'dan bir kez indirilen kopya
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=6, language="tr", packages=["transformers"])   # mesajlar, şekiller, haftanın paketleri ve beklenen cevaplar
RANDOM_STATE = 2026
import torch
from vclinic import reports

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))

from vclinic import reports
import re

if "angio" not in globals():
    archive_angio_ready = True
    def tr_lower(text):
        """Türkçe kurallarına göre küçük harf: I, ı olur; İ, i olur ve metnin uzunluğu değişmez."""
        return text.replace("I", "ı").replace("İ", "i").lower()


    def words(text):
        """Bir metnin küçük harfli kelimeleri: Harf ve rakam dizileri."""
        return re.findall(r"\w+", tr_lower(text))

    angio = reports.angiography_archive()
    TERRITORIES = ["LM", "LAD", "LCX", "RCA"]
    VESSELS = {"LM": r"sol ana koroner|lmca", "LAD": r"sol ön inen|\blad\b", "D1": r"birinci diyagonal|1\. diyagonal|\bd1\b",
               "D2": r"ikinci diyagonal|2\. diyagonal|\bd2\b", "LCX": r"sirkumfleks|\bl?cx\b", "OM1": r"birinci obtus|1\. obtus|\bom1\b",
               "OM2": r"ikinci obtus|2\. obtus|\bom2\b", "RCA": r"sağ koroner|\brca\b", "PDA": r"arka inen|posterior desendan|\bpda\b"}
    TERRITORY = {"LM": "LM", "LAD": "LAD", "D1": "LAD", "D2": "LAD", "LCX": "LCX", "OM1": "LCX", "OM2": "LCX", "RCA": "RCA", "PDA": "RCA"}
    WORDS = {"total oklüzyon": 100, "tam tıkanıklık": 100, "to": 100, "subtotal": 99, "kritik": 95,
             "ileri derecede": 80, "orta derecede": 60, "hafif": 30, "plak": 30, "plaklı": 30}
    NUMBER = r"(?:%\s?|yüzde\s)(\d{1,3})(?:\s?-\s?(\d{1,3}))?"
    WORD = r"(?<!\w)(" + "|".join(sorted(WORDS, key=len, reverse=True)) + r")(?!\w)"
    SKIP = ("endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim", "işlem")


    def stenoses(sentence):
        """Bir cümledeki her darlık derecesi, yüzde olarak: Sayılar (bir aralık üst ucuyla sayılır) ve kelimeler."""
        values = [int(m.group(2) or m.group(1)) for m in re.finditer(NUMBER, sentence)]
        values += [WORDS[m.group(1)] for m in re.finditer(WORD, sentence)]
        return values


    def angio_table(text, coreference=True):
        """Bir rapordaki her bölgenin en büyük darlığı. Cümle, büyük harften ya da rakamdan önceki noktada biter; arter adı geçmeyen bir cümle, önceki cümlenin arterine aittir."""
        best = {t: 0 for t in TERRITORIES}
        last = None
        for sentence in re.split(r"(?<=\.)\s+(?=[A-ZÇĞİÖŞÜ0-9])", text):
            sentence = tr_lower(sentence)
            if sentence.startswith(SKIP):
                continue
            found = sorted((m.start(), v) for v, pattern in VESSELS.items() for m in re.finditer(pattern, sentence))
            if found:
                last = found[0][1]
            elif not coreference:
                continue
            if last is None:
                continue
            for value in stenoses(sentence):
                best[TERRITORY[last]] = max(best[TERRITORY[last]], value)
        return best


    def category(value):
        """50'nin altı 0, 50-69 için 1, 70-89 için 2, 90-99 için 3 ve tıkanma için 4."""
        return 0 if value < 50 else 1 if value < 70 else 2 if value < 90 else 3 if value < 100 else 4


    def n_vessels(table):
        """Hastalıklı damar sayısı: Yüzde 70 ya da daha fazla darlığı olan LAD, LCX ve RCA bölgeleri."""
        return sum(table[t] >= 70 for t in ("LAD", "LCX", "RCA"))
_TER = ["LM", "LAD", "CX", "RCA"]


def _reports(mask):
    """Kayıt sistemine göre her bölgenin en büyük darlığı ve hastalıklı damar sayısıyla birlikte anjiyografi raporları."""
    d = angio[mask].reset_index(drop=True)
    t = pd.DataFrame({"rapor": d["text_tr"]})
    for name, col in zip(_TER, ["max_LM", "max_LAD", "max_LCX", "max_RCA"]):
        t[name] = d[col]
    t["hastalıklı damar"] = d["n_vessels"]
    return t
print(len(angio), "anjiyografi raporu")

if "archive" not in globals():
    archive = reports.archive()
    LABELS = list(reports.LABELS)
    texts = archive["text_tr"].to_numpy()
    y = archive[LABELS].to_numpy()
    split = archive["split"].to_numpy()
    tr, va, te, ex = (split == "training"), (split == "validation"), (split == "test"), (split == "external")


def _project_table():
    d = angio[angio["split"].isin(["test", "external"])].reset_index(drop=True)
    by_patient = archive.set_index("patient_id")["text_tr"]
    t = pd.DataFrame({"küme": ["test" if s == "test" else "dış" for s in d["split"]], "hasta": d["patient_id"],
                      "rapor": d["text_tr"]})
    for name, col in zip(["LM", "LAD", "CX", "RCA"], ["max_LM", "max_LAD", "max_LCX", "max_RCA"]):
        t[name] = d[col]
    t["hastalıklı damar"] = d["n_vessels"]
    t["epikriz"] = by_patient.loc[d["patient_id"]].to_numpy()
    return t

### Proje adımı 1: İki hastanede bölge bölge

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Bir kayıt sistemi için iki hastanenin koroner anjiyografi raporlarını okuyan bir program kurmak ve programın her koroner bölgede ne sıklıkla kayıt sistemiyle aynı darlık derecesini verdiğini öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; sentetik bir kayıt sistemi olan sanal kliniğin A hastanesine ait test kümesindeki 394 ve B hastanesindeki 373 koroner anjiyografi raporunu, yani Türkçe yazılmış toplam 767 raporu tutar; B hastanesi büyük harflerle ve kısa bir üslupla yazar. Her satır bir hastadır: "küme" sütunu kümeyi ("test" ya da "dış"), "hasta" hastayı, "rapor" raporu verir; "LM", "LAD", "CX" ve "RCA" adlı dört sütun kayıt sistemine göre dört koroner bölgenin en büyük darlığını yüzde olarak, "hastalıklı damar" ise yüzde 70 ya da daha fazla darlığı olan LAD, CX ve RCA bölgelerinin sayısını verir. "epikriz" sütunu aynı hastanın epikrizini tutar.  
> Görev: Raporu cümle cümle oku; bir cümle, ardından bir boşluk ve bir büyük harf ya da bir rakam gelen noktada biter. Her cümleyi Türkçe kurallarına göre küçük harfe çevir (önce I, ı; İ, i olur). "endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim" ya da "işlem" ile başlayan cümleleri atla. Öteki her cümlede, cümlenin andığı ilk arteri bul: Sol ana koroner için "sol ana koroner" ya da "lmca"; LAD için "sol ön inen" ya da "lad", dalları için "birinci diyagonal", "1. diyagonal" ya da "d1" ve "ikinci diyagonal", "2. diyagonal" ya da "d2"; sirkumfleks için "sirkumfleks", "cx" ya da "lcx", dalları için "birinci obtus", "1. obtus" ya da "om1" ve "ikinci obtus", "2. obtus" ya da "om2"; RCA için "sağ koroner" ya da "rca" ve "arka inen", "posterior desendan" ya da "pda". Kısa adlar yalnızca tam kelime olarak sayılır. Arter adı geçmeyen bir cümle, önceki cümlenin arterine aittir. Cümledeki her darlık derecesini oku: "%" ya da "yüzde " ifadesinden sonraki sayı ("%70-80" gibi bir aralık üst ucuyla, yani 80 olarak sayılır) ve "total oklüzyon", "tam tıkanıklık" ya da "to" (100), "subtotal" (99), "kritik" (95), "ileri derecede" (80), "orta derecede" (60) ile "hafif", "plak" ya da "plaklı" (30) kelimeleri. Her değeri arterinin bölgesine ver (dallar ana arterlerinin bölgesine aittir) ve her bölgenin en büyük değerini tut; değeri olmayan bölge 0 alır. Hastalıklı damar sayısı, değeri 70 ya da daha fazla olan LAD, CX ve RCA bölgelerini sayar. Bunu her rapora uygula. Programın ve kayıt sisteminin verdiği her en büyük darlığı beş kategoriden birine koy: 50'nin altı, 50-69, 70-89, 90-99 ve 100. Her küme ve her bölge için kategorilerin uyuştuğu raporların oranını ve dört bölgenin de uyuştuğu raporların oranını hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her küme için bir satır ve beş oran içeren bir tablo göster; "test" kümesinde dört bölgenin de uyuştuğu raporların oranını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: "test" kümesinde 394, "dış" kümesinde 373 rapor bulunmalı ve her oran 0 ile 1 arasında olmalı.  

In [ ]:
# Proje adımı 1 için veri: Promptun tarif ettiği "veri" tablosu
veri = _project_table()
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 2: Sol ana koroner arter ve hastalıklı damar sayısı

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Koroner anjiyografi raporlarını okuyan bir programın, cerrahi ile stent arasındaki kararı belirleyen iki bulguyu, yani sol ana koroner arter hastalığını ve hastalıklı damar sayısını ne kadar iyi bulduğunu öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; sentetik bir kayıt sistemi olan sanal kliniğin A hastanesine ait test kümesindeki 394 ve B hastanesindeki 373 koroner anjiyografi raporunu, yani Türkçe yazılmış toplam 767 raporu tutar; B hastanesi büyük harflerle ve kısa bir üslupla yazar. Her satır bir hastadır: "küme" sütunu kümeyi ("test" ya da "dış"), "hasta" hastayı, "rapor" raporu verir; "LM", "LAD", "CX" ve "RCA" adlı dört sütun kayıt sistemine göre dört koroner bölgenin en büyük darlığını yüzde olarak, "hastalıklı damar" ise yüzde 70 ya da daha fazla darlığı olan LAD, CX ve RCA bölgelerinin sayısını verir. "epikriz" sütunu aynı hastanın epikrizini tutar.  
> Görev: Raporu cümle cümle oku; bir cümle, ardından bir boşluk ve bir büyük harf ya da bir rakam gelen noktada biter. Her cümleyi Türkçe kurallarına göre küçük harfe çevir (önce I, ı; İ, i olur). "endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim" ya da "işlem" ile başlayan cümleleri atla. Öteki her cümlede, cümlenin andığı ilk arteri bul: Sol ana koroner için "sol ana koroner" ya da "lmca"; LAD için "sol ön inen" ya da "lad", dalları için "birinci diyagonal", "1. diyagonal" ya da "d1" ve "ikinci diyagonal", "2. diyagonal" ya da "d2"; sirkumfleks için "sirkumfleks", "cx" ya da "lcx", dalları için "birinci obtus", "1. obtus" ya da "om1" ve "ikinci obtus", "2. obtus" ya da "om2"; RCA için "sağ koroner" ya da "rca" ve "arka inen", "posterior desendan" ya da "pda". Kısa adlar yalnızca tam kelime olarak sayılır. Arter adı geçmeyen bir cümle, önceki cümlenin arterine aittir. Cümledeki her darlık derecesini oku: "%" ya da "yüzde " ifadesinden sonraki sayı ("%70-80" gibi bir aralık üst ucuyla, yani 80 olarak sayılır) ve "total oklüzyon", "tam tıkanıklık" ya da "to" (100), "subtotal" (99), "kritik" (95), "ileri derecede" (80), "orta derecede" (60) ile "hafif", "plak" ya da "plaklı" (30) kelimeleri. Her değeri arterinin bölgesine ver (dallar ana arterlerinin bölgesine aittir) ve her bölgenin en büyük değerini tut; değeri olmayan bölge 0 alır. Hastalıklı damar sayısı, değeri 70 ya da daha fazla olan LAD, CX ve RCA bölgelerini sayar. Bunu her rapora uygula. Sol ana koroner arter hastalığı, LM bölgesinde en büyük darlığın yüzde 50 ya da daha fazla olmasıdır. Her küme için programın sol ana koroner arter hastalığındaki duyarlılığını ve pozitif öngörü değerini "LM" sütununa göre hesapla; kayıt sistemindeki hastalıklı damar sayısını programınkine karşı gösteren bir tablo göster.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her küme için iki değeri ve iki tabloyu yazdır; "test" kümesinde sol ana koroner arter hastalığının pozitif öngörü değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: İki tablonun sayılarının toplamı kümedeki rapor sayısına eşit olmalı.  

In [ ]:
# Proje adımı 2 için veri: Promptun tarif ettiği "veri" tablosu
veri = _project_table()
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 3: Hatalar ve nedenleri

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Koroner anjiyografi raporlarını okuyan bir programın yanlış okuduğu her raporu görmek ve programı yanıltan cümleyi bulmak istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; sentetik bir kayıt sistemi olan sanal kliniğin A hastanesine ait test kümesindeki 394 ve B hastanesindeki 373 koroner anjiyografi raporunu, yani Türkçe yazılmış toplam 767 raporu tutar; B hastanesi büyük harflerle ve kısa bir üslupla yazar. Her satır bir hastadır: "küme" sütunu kümeyi ("test" ya da "dış"), "hasta" hastayı, "rapor" raporu verir; "LM", "LAD", "CX" ve "RCA" adlı dört sütun kayıt sistemine göre dört koroner bölgenin en büyük darlığını yüzde olarak, "hastalıklı damar" ise yüzde 70 ya da daha fazla darlığı olan LAD, CX ve RCA bölgelerinin sayısını verir. "epikriz" sütunu aynı hastanın epikrizini tutar.  
> Görev: Raporu cümle cümle oku; bir cümle, ardından bir boşluk ve bir büyük harf ya da bir rakam gelen noktada biter. Her cümleyi Türkçe kurallarına göre küçük harfe çevir (önce I, ı; İ, i olur). "endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim" ya da "işlem" ile başlayan cümleleri atla. Öteki her cümlede, cümlenin andığı ilk arteri bul: Sol ana koroner için "sol ana koroner" ya da "lmca"; LAD için "sol ön inen" ya da "lad", dalları için "birinci diyagonal", "1. diyagonal" ya da "d1" ve "ikinci diyagonal", "2. diyagonal" ya da "d2"; sirkumfleks için "sirkumfleks", "cx" ya da "lcx", dalları için "birinci obtus", "1. obtus" ya da "om1" ve "ikinci obtus", "2. obtus" ya da "om2"; RCA için "sağ koroner" ya da "rca" ve "arka inen", "posterior desendan" ya da "pda". Kısa adlar yalnızca tam kelime olarak sayılır. Arter adı geçmeyen bir cümle, önceki cümlenin arterine aittir. Cümledeki her darlık derecesini oku: "%" ya da "yüzde " ifadesinden sonraki sayı ("%70-80" gibi bir aralık üst ucuyla, yani 80 olarak sayılır) ve "total oklüzyon", "tam tıkanıklık" ya da "to" (100), "subtotal" (99), "kritik" (95), "ileri derecede" (80), "orta derecede" (60) ile "hafif", "plak" ya da "plaklı" (30) kelimeleri. Her değeri arterinin bölgesine ver (dallar ana arterlerinin bölgesine aittir) ve her bölgenin en büyük değerini tut; değeri olmayan bölge 0 alır. Hastalıklı damar sayısı, değeri 70 ya da daha fazla olan LAD, CX ve RCA bölgelerini sayar. Bunu "test" kümesindeki her rapora uygula. Programın ve kayıt sisteminin verdiği her en büyük darlığı beş kategoriden birine koy: 50'nin altı, 50-69, 70-89, 90-99 ve 100. En az bir bölgesi uyuşmayan rapor yanlıştır. Her yanlış rapor için uyuşmayan bölgeleri ve programın tanıdığı hiçbir arteri anmayan ama bir darlık derecesi içeren cümleleri göster.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Yanlış raporları bölgeleri ve cümleleriyle birlikte yazdır; yanlış raporların sayısını "cevap" adıyla sakla.  
> Test: Yanlış rapor sayısı 394'ten büyük olamaz.  

In [ ]:
# Proje adımı 3 için veri: Promptun tarif ettiği "veri" tablosu
veri = _project_table()
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 4: Anjiyografi ve epikriz birlikte

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Anjiyografide anlamlı koroner darlığı olan hastalardan kaçının epikrizinde, epikrizleri tarayan bir aramanın bulacağı biçimde koroner arter hastalığı bulunmadığını öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; sentetik bir kayıt sistemi olan sanal kliniğin A hastanesine ait test kümesindeki 394 ve B hastanesindeki 373 koroner anjiyografi raporunu, yani Türkçe yazılmış toplam 767 raporu tutar; B hastanesi büyük harflerle ve kısa bir üslupla yazar. Her satır bir hastadır: "küme" sütunu kümeyi ("test" ya da "dış"), "hasta" hastayı, "rapor" raporu verir; "LM", "LAD", "CX" ve "RCA" adlı dört sütun kayıt sistemine göre dört koroner bölgenin en büyük darlığını yüzde olarak, "hastalıklı damar" ise yüzde 70 ya da daha fazla darlığı olan LAD, CX ve RCA bölgelerinin sayısını verir. "epikriz" sütunu aynı hastanın epikrizini tutar.  
> Görev: Raporu cümle cümle oku; bir cümle, ardından bir boşluk ve bir büyük harf ya da bir rakam gelen noktada biter. Her cümleyi Türkçe kurallarına göre küçük harfe çevir (önce I, ı; İ, i olur). "endikasyon", "öykü", "sol ventrikülografi", "sv:", "girişim" ya da "işlem" ile başlayan cümleleri atla. Öteki her cümlede, cümlenin andığı ilk arteri bul: Sol ana koroner için "sol ana koroner" ya da "lmca"; LAD için "sol ön inen" ya da "lad", dalları için "birinci diyagonal", "1. diyagonal" ya da "d1" ve "ikinci diyagonal", "2. diyagonal" ya da "d2"; sirkumfleks için "sirkumfleks", "cx" ya da "lcx", dalları için "birinci obtus", "1. obtus" ya da "om1" ve "ikinci obtus", "2. obtus" ya da "om2"; RCA için "sağ koroner" ya da "rca" ve "arka inen", "posterior desendan" ya da "pda". Kısa adlar yalnızca tam kelime olarak sayılır. Arter adı geçmeyen bir cümle, önceki cümlenin arterine aittir. Cümledeki her darlık derecesini oku: "%" ya da "yüzde " ifadesinden sonraki sayı ("%70-80" gibi bir aralık üst ucuyla, yani 80 olarak sayılır) ve "total oklüzyon", "tam tıkanıklık" ya da "to" (100), "subtotal" (99), "kritik" (95), "ileri derecede" (80), "orta derecede" (60) ile "hafif", "plak" ya da "plaklı" (30) kelimeleri. Her değeri arterinin bölgesine ver (dallar ana arterlerinin bölgesine aittir) ve her bölgenin en büyük değerini tut; değeri olmayan bölge 0 alır. Hastalıklı damar sayısı, değeri 70 ya da daha fazla olan LAD, CX ve RCA bölgelerini sayar. Bunu her rapora uygula; program en az bir hastalıklı damar ya da sol ana koroner arter hastalığı bulduğunda hastada anlamlı darlık vardır. Ardından aynı hastanın epikrizini oku: Her metni Türkçe kurallarına göre küçük harfe çevir: Önce I'yı ı, İ'yi i yap, ardından öteki harfleri küçült. "koroner arter hastalığı", "kah", "iskemik kalp hastalığı", "miyokard enfarktüsü" ya da "akut koroner sendrom" tam kelime olarak geçiyor, cümlesinin sonuna ya da başka bir tanının bir sonraki geçişine kadar ardından "yok", "izlenmedi", "saptanmadı", "tanımlamıyor", "dışlandı", "(-)", "şüphe", "olası" ya da "ön tanı" gelmiyor ve cümlesinde önünde "babası", "annesi", "kardeşi", "babada", "annede" ya da "soygeçmiş" durmuyorsa ya da epikriz "klopidogrel", "tikagrelor", "pkg", "kabg" ya da "stent" kelimelerinden birini anıyorsa epikrizde koroner arter hastalığı vardır. Her küme için anlamlı darlığı olduğu hâlde epikrizinde koroner arter hastalığı bulunmayan hastaları say.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her küme için sayıları ve bu epikrizlerden üçünü yazdır; "test" kümesindeki sayıyı "cevap" adıyla sakla.  
> Test: Sayı, kümede anlamlı darlığı olan hastaların sayısından büyük olamaz.  

In [ ]:
# Proje adımı 4 için veri: Promptun tarif ettiği "veri" tablosu
veri = _project_table()
cevap = None   # önceki bir promptun cevabı silinir
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

**Haftalık projenin sonu.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/ders.html#weekly-project): «Haftalık proje» bölümü raporu ve teslimi anlatır.

## Alanlara göre uygulama görevleri

| Alan | Görev |
|---|---|
| Kalp yetersizliği | Kurala alt tipi ekleyin: Epikriz yüzde 40'ın altında bir ejeksiyon fraksiyonu veriyorsa düşük ejeksiyon fraksiyonlu kalp yetersizliği. Kural test kümesinde kaç hasta bulur ve bunları kayıt sistemiyle nasıl karşılaştırırdınız? |
| Elektrofizyoloji | Atriyal fibrilasyonu olan hastaların epikrizleri bir antikoagülan listeler. Epikrizinde hiç antikoagülan bulunmayan atriyal fibrilasyonlu hastaları sayın ve bir kalite kurulunun bu hastalar hakkında ihtiyaç duyacağı yarım sayfayı yazın. |
| Girişimsel kardiyoloji | «Durak 3 · Hazır uygulama» bölümündeki işlem hattını LAD'deki en büyük darlığın segmentini de verecek biçimde genişletin. İki damar hastalığı olan hastalarda proksimal LAD ne sıklıkla hastalıklıdır? |
| Koruyucu kardiyoloji | Her epikrizdeki LDL değerini bir sayı olarak okuyun ve koroner arter hastalığı olan hastaların hedefleriyle karşılaştırın. Koroner arter hastalığı olan hastalardan kaçı LDL değeri 70 mg/dL'nin üstündeyken ve statin almadan taburcu olur? |
| Nefroloji ve kardiyoloji | Kural kronik böbrek hastalığını tek bir eGFR değerinden belirler. Arşivdeki hangi hastaların eGFR değeri 55 ile 60 arasındadır ve üç ay sonra ölçülen ikinci bir değer bu hastaların etiketini nasıl değiştirirdi? |
| Klinik araştırma | 3.600 hastanın tamamı için kuralın verdiği altı tanıyı ve anjiyografideki hastalıklı damar sayısını içeren bir tablo kurun. Anlamlı darlığı olan hastalardan kaçının epikrizinde koroner arter hastalığı yoktur ve bu durum yalnızca epikrizlerden kurulan bir kayıt sistemi için ne anlama gelir? |
| Tıp bilişimi | B hastanesinin üslubuyla, dersin kuralının yanlış okuduğu üç yeni cümle yazın ve kuralı «Durak 1 · Hazır uygulama» bölümünün sonuçlarını bozmadan bu cümleleri doğru okuyacak biçimde değiştirin. |

## Jupyter Notebook'tan sonra

[Etkileşimli labı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-06/lab.html) açın ve *Kendini sına* sekmesindeki öz değerlendirmeyi yapın. Ardından *Düşün ve dışa aktar* sekmesinde yansıtma yazınızı yazın ve öğrenme günlüğünü dışa aktarın. Bu haftanın haftalık projesi «Durak 3 · Hazır uygulama» bölümünü genişletir: İki hastanenin anjiyografi raporlarından darlık derecesini ve damarı çıkaran ve hatalarını açıklayan bir işlem hattı.

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı, Türk Kardiyoloji Derneği. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>